# ARC-AGI-3 Coverage c8b

CPU agent; frozen public-observation policy, seed 0, 1200 total actions per game including initialization RESET. All games exposed by the competition gateway are played. Local public-development scores are not Kaggle scores.

In [ ]:
!pip install --no-index --find-links /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels arc-agi python-dotenv

In [ ]:
%%writefile /tmp/my_agent.py
"""Frozen public-observation c8b policy; CPU, seed 0, 1200 actions."""
from pathlib import Path
import sys,tempfile
_SOURCES = {'experiment_coverage_guarded.py': '"""Prefer a controller that has already completed two levels in this game."""\nfrom agent import Observation\nfrom experiment_coverage_c8 import Explorer as CoverageExplorer\n\nclass Explorer(CoverageExplorer):\n    def __init__(self,seed=0,policy=\'coverage_c8b\'):\n        if policy!=\'coverage_c8b\':raise ValueError(policy)\n        super().__init__(seed,\'coverage_c8\');self.variant=policy\n\n    def choose(self,obs):\n        if self.stats[\'level_gains_under_base\']>=2:\n            if self.level!=obs.levels:self.level=obs.levels;self.stagnation=0\n            self.intervening=False\n            action=self.base.choose(obs);self.reason=self.base.reason\n            self.stats[\'successful_base_priority_decisions\']+=1\n            return action\n        return super().choose(obs)\n', 'experiment_coverage_c8.py': '"""Revisit a stagnant visual condition with persistent action coverage.\n\nOnly public observations enter this policy. Cropped-screen equality is used as\nan exploration heuristic, never as a certified state identity or a simulator.\n"""\nfrom collections import Counter, defaultdict\nimport hashlib\nimport math\nimport random\n\nimport numpy as np\nfrom agent import Action, Observation, components\nfrom experiment_transfer_c7 import Explorer as C7Explorer\n\n\nclass Explorer:\n    def __init__(self, seed=0, policy=\'coverage_c8\'):\n        if policy != \'coverage_c8\':\n            raise ValueError(policy)\n        self.base = C7Explorer(seed, \'transfer_c7\')\n        self.rng = random.Random(seed + 1831)\n        self.variant = policy\n        self.reason = \'\'\n        self.visits = Counter()\n        self.trials = Counter()\n        self.action_trials = Counter()\n        self.effects = Counter()\n        self.failures = Counter()\n        self.successors = defaultdict(Counter)\n        self.stats = Counter()\n        self.stagnation = 0\n        self.level = None\n        self.intervening = False\n\n    @staticmethod\n    def key(obs):\n        # Border clocks are a source of false novelty. This cropping may hide\n        # real border state, so it only affects exploratory ranking.\n        pixels = obs.pixels\n        if min(pixels.shape) > 12:\n            pixels = pixels[3:-3, 3:-3]\n        return (obs.levels, obs.available, pixels.shape,\n                hashlib.blake2b(pixels.tobytes(), digest_size=16).digest())\n\n    @staticmethod\n    def action_key(action):\n        return (action.id, action.x, action.y)\n\n    def candidates(self, obs):\n        actions = [Action(a) for a in obs.available if a not in (0, 6)]\n        if 6 in obs.available:\n            objects = components(obs.pixels)\n            centers = {c.center for c in objects\n                       if 1 <= c.size <= 256 and max(c.width, c.height) <= 24}\n            # Include small single-pixel cues previously omitted by the base\n            # segmentation. Uniform coverage is a fallback for empty scenes.\n            if not centers:\n                h, w = obs.pixels.shape\n                centers = {(x, y) for y in range(4, h, 8) for x in range(4, w, 8)}\n            actions += [Action(6, x, y) for x, y in sorted(centers)]\n        return actions\n\n    def choose(self, obs):\n        if obs.levels != self.level:\n            self.level = obs.levels\n            self.stagnation = 0\n        key = self.key(obs)\n        # A time allowance plus repeated public configurations is required;\n        # unusual frames alone do not switch the controller.\n        self.intervening = (obs.state not in (\'GAME_OVER\', \'NOT_PLAYED\', \'WIN\')\n                            and self.stagnation >= 96 and self.visits[key] >= 3)\n        if not self.intervening:\n            action = self.base.choose(obs)\n            self.reason = self.base.reason\n            return action\n        actions = self.candidates(obs)\n        if not actions:\n            action = self.base.choose(obs)\n            self.reason = self.base.reason\n            return action\n        def rank(action):\n            a = self.action_key(action)\n            context = (obs.levels, a)\n            used = self.action_trials[context]\n            effect = (self.effects[context] + 1) / (used + 2)\n            failure = self.failures[context] / max(1, used)\n            successors = self.successors[key, a]\n            novelty = (1 / math.sqrt(1 + min(self.visits[s] for s in successors))) if successors else 1\n            return (4 / math.sqrt(1 + self.trials[key, a])\n                    + 2 * effect + novelty + 1 / math.sqrt(1 + used)\n                    - 3 * failure + self.rng.random() * .05)\n        action = max(actions, key=rank)\n        self.stats[\'interventions\'] += 1\n        self.reason = \'coverage: stagnant repeated public scene; test under-observed action\'\n        return action\n\n    def observe(self, before, action, after):\n        self.base.observe(before, action, after)\n        old, new = self.key(before), self.key(after)\n        if action.id:\n            a = self.action_key(action)\n            context = (before.levels, a)\n            self.trials[old, a] += 1\n            self.action_trials[context] += 1\n            self.effects[context] += old != new\n            self.failures[context] += after.state == \'GAME_OVER\'\n            self.successors[old, a][new] += 1\n            self.visits[new] += 1\n            self.stagnation += 1\n        if after.levels > before.levels:\n            self.stats[\'level_gains_after_intervention\' if self.intervening else \'level_gains_under_base\'] += after.levels - before.levels\n            self.stagnation = 0\n        # Failed attempts retain exploration counts; the base invalidates its\n        # own transition hypotheses on RESET as before.\n\n    def diagnostics(self):\n        result = self.base.diagnostics()\n        result[\'variant\'] = self.variant\n        result[\'persistent_coverage\'] = {\'stats\': dict(self.stats),\n            \'public_conditions\': len(self.visits), \'condition_actions\': len(self.trials),\n            \'scope\': \'Persistent public visual-condition/action counts used for exploration only.\'}\n        return result\n', 'experiment_transfer_c7.py': "from experiment_free_transfer import Explorer as C6Explorer\nfrom agent import Observation\n\nclass Explorer(C6Explorer):\n    def __init__(self, seed=0, policy='transfer_c7'):\n        if policy != 'transfer_c7': raise ValueError(policy)\n        super().__init__(seed, 'free_transfer')\n        self.variant = policy\n", 'agent.py': '"""Observation-only ARC-AGI-3 exploration pilot.\n\nNo game identifiers, game objects, files, network, or reference solutions enter\nthe policy. All state abstractions below are heuristics, never certified state\nquotients. The complete observation/action trace is retained by the runner.\n"""\nfrom __future__ import annotations\n\nimport hashlib\nimport random\nfrom collections import Counter, defaultdict, deque\nfrom dataclasses import dataclass\n\nimport numpy as np\n\nfrom relations import PanelHypotheses, repeated_panels\nfrom transfer import TransferMemory\nfrom partial_relations import PartialRelations\nfrom shared_dynamics import SharedDynamics\nfrom joint_dynamics import JointDynamics\nfrom association_dynamics import AssociationDynamics\nfrom reconsideration import ReconsiderationDynamics\n\n\n@dataclass(frozen=True)\nclass Observation:\n    pixels: np.ndarray\n    available: tuple[int, ...]\n    state: str\n    levels: int\n\n    @classmethod\n    def from_frame(cls, frame) -> "Observation":\n        pixels = np.asarray(frame.frame[-1], dtype=np.uint8).copy()\n        if pixels.ndim != 2 or not (0 < pixels.shape[0] <= 64 and 0 < pixels.shape[1] <= 64):\n            raise ValueError("Expected a nonempty grid at most 64 by 64")\n        if np.any(pixels > 15):\n            raise ValueError("Expected ARC palette indices 0..15")\n        pixels.setflags(write=False)\n        state = frame.state.name if hasattr(frame.state, "name") else str(frame.state)\n        return cls(pixels, tuple(sorted(set(int(a) for a in frame.available_actions))),\n                   state, int(frame.levels_completed))\n\n    def key(self) -> str:\n        metadata = repr((self.pixels.shape, self.available, self.state, self.levels)).encode()\n        return hashlib.sha256(metadata + self.pixels.tobytes()).hexdigest()\n\n\n@dataclass(frozen=True)\nclass Action:\n    id: int\n    x: int | None = None\n    y: int | None = None\n\n    def data(self) -> dict[str, int]:\n        return {} if self.id != 6 else {"x": int(self.x), "y": int(self.y)}\n\n\n@dataclass(frozen=True)\nclass Component:\n    color: int\n    x: int\n    y: int\n    width: int\n    height: int\n    size: int\n    center: tuple[int, int]\n    shape: bytes\n\n    @property\n    def signature(self) -> tuple:\n        return self.color, self.width, self.height, self.shape\n\n\ndef components(pixels: np.ndarray) -> list[Component]:\n    """Four-connected monochrome components; center always belongs to it."""\n    h, w = pixels.shape\n    seen = np.zeros((h, w), dtype=bool)\n    result = []\n    for y in range(h):\n        for x in range(w):\n            if seen[y, x]:\n                continue\n            color = int(pixels[y, x])\n            queue = deque([(x, y)])\n            seen[y, x] = True\n            cells = []\n            while queue:\n                cx, cy = queue.popleft()\n                cells.append((cx, cy))\n                for nx, ny in ((cx - 1, cy), (cx + 1, cy), (cx, cy - 1), (cx, cy + 1)):\n                    if 0 <= nx < w and 0 <= ny < h and not seen[ny, nx] and pixels[ny, nx] == color:\n                        seen[ny, nx] = True\n                        queue.append((nx, ny))\n            xs, ys = zip(*cells)\n            x0, y0, x1, y1 = min(xs), min(ys), max(xs), max(ys)\n            ww, hh = x1 - x0 + 1, y1 - y0 + 1\n            mask = np.zeros((hh, ww), dtype=np.uint8)\n            for cx, cy in cells:\n                mask[cy - y0, cx - x0] = 1\n            mx, my = sum(xs) / len(xs), sum(ys) / len(ys)\n            center = min(cells, key=lambda p: (p[0] - mx) ** 2 + (p[1] - my) ** 2)\n            result.append(Component(color, x0, y0, ww, hh, len(cells), center,\n                                    np.packbits(mask).tobytes()))\n    return result\n\n\ndef click_candidates(obs: Observation, objects: list[Component]) -> list[Action]:\n    h, w = obs.pixels.shape\n    # Object centers reduce coordinates, but do not establish which are controls.\n    centers = {c.center for c in objects if 2 <= c.size <= 256 and c.width <= 24 and c.height <= 24}\n    if not centers:\n        centers = {(x, y) for y in range(4, h, 8) for x in range(4, w, 8)}\n    if not centers:\n        centers = {(w // 2, h // 2)}\n    return [Action(6, x, y) for x, y in sorted(centers)]\n\n\nclass TransitionLedger:\n    """Observed evidence, with counterexamples to a proposed screen-only model.\n\n    A single result is never a proof of determinism. Conflicts may reflect\n    history, omitted animation, randomness, or another unobserved variable.\n    """\n    def __init__(self) -> None:\n        self.edges: dict[tuple[str, Action], Counter] = defaultdict(Counter)\n        self.first_examples: dict[tuple[str, Action, tuple], int] = {}\n        self.conflicts: list[dict] = []\n        self.transitions = 0\n\n    def add(self, before: Observation, action: Action, after: Observation) -> None:\n        key = (before.key(), action)\n        outcome = (after.key(), after.levels - before.levels, after.state)\n        outcomes = self.edges[key]\n        if outcomes and outcome not in outcomes:\n            previous = next(iter(outcomes))\n            self.conflicts.append({\n                "screen": key[0], "action": {"id": action.id, **action.data()},\n                "first_transition": self.first_examples[(key[0], action, previous)],\n                "new_transition": self.transitions,\n                "interpretation": "screen-only prediction contradicted; cause unknown",\n            })\n        if outcome not in outcomes:\n            self.first_examples[(key[0], action, outcome)] = self.transitions\n        outcomes[outcome] += 1\n        self.transitions += 1\n\n    def status(self, observation: Observation, action: Action) -> str:\n        outcomes = self.edges.get((observation.key(), action))\n        if not outcomes:\n            return "unknown"\n        if len(outcomes) > 1:\n            return "conflicting"\n        return "observed_once" if sum(outcomes.values()) == 1 else "empirically_consistent"\n\n    def summary(self) -> dict:\n        return {"transitions": self.transitions, "observed_screen_action_pairs": len(self.edges),\n                "screen_model_conflicts": len(self.conflicts), "conflict_examples": self.conflicts[:8],\n                "certified_state_merges": 0}\n\n\nclass Explorer:\n    def __init__(self, seed: int = 0, policy: str = "probe") -> None:\n        if policy not in ("uniform", "objects", "probe", "relational", "adaptive", "transfer", "forget", "partial", "shared", "shared_probe", "joint", "joint_flat", "compose", "associate", "review_pair", "review"):\n            raise ValueError(policy)\n        self.rng = random.Random(seed)\n        self.policy = policy\n        self.ledger = TransitionLedger()\n        self.steps = 0\n        self.last_action: Action | None = None\n        self.cache_key = None\n        self.cache_components: list[Component] = []\n        self.action_counts = Counter()\n        self.click_effects: dict[Action, list[int]] = defaultdict(list)\n        self.anchor_signature: tuple | None = None\n        self.motion: dict[int, tuple[int, int]] = {}\n        self.position_counts = Counter()\n        self.position_action_counts = Counter()\n        self.blocked = Counter()\n        self.reason = ""\n        self.relations = PanelHypotheses()\n        self.memory = TransferMemory(retain=policy != "forget")\n        self.partial = PartialRelations()\n        self.shared = ReconsiderationDynamics(seed, retain_color=policy != "review_pair") if policy in ("review_pair", "review") else AssociationDynamics(seed) if policy == "associate" else JointDynamics(seed, recursive=policy != "joint_flat", minimum_goals=1 if policy == "compose" else 2) if policy in ("joint", "joint_flat", "compose", "associate", "review_pair", "review") else SharedDynamics(seed, planning=policy != "shared_probe")\n\n    def objects(self, obs: Observation) -> list[Component]:\n        key = obs.key()\n        if key != self.cache_key:\n            self.cache_key, self.cache_components = key, components(obs.pixels)\n        return self.cache_components\n\n    def anchor(self, obs: Observation) -> Component | None:\n        matches = [c for c in self.objects(obs) if c.signature == self.anchor_signature]\n        return matches[0] if len(matches) == 1 else None\n\n    def choose(self, obs: Observation) -> Action:\n        if obs.state in ("NOT_PLAYED", "GAME_OVER"):\n            self.reason = "reset after terminal observation"\n            return Action(0)\n        if obs.state == "WIN":\n            raise RuntimeError("No further actions after WIN")\n        legal = [a for a in obs.available if 1 <= a <= 7]\n        if not legal:\n            raise RuntimeError("No available action in unfinished game")\n        if self.policy == "uniform":\n            action_id = self.rng.choice(legal)\n            self.reason = "uniform legal action / uniform coordinates"\n            if action_id == 6:\n                return Action(6, self.rng.randrange(obs.pixels.shape[1]),\n                              self.rng.randrange(obs.pixels.shape[0]))\n            return Action(action_id)\n        candidates = [Action(a) for a in legal if a != 6]\n        if 6 in legal:\n            candidates.extend(click_candidates(obs, self.objects(obs)))\n        if self.policy in ("partial", "shared", "shared_probe", "joint", "joint_flat", "compose", "associate", "review_pair", "review"):\n            proposal = self.partial.propose(obs, self.objects(obs), self.memory.skills)\n            if proposal:\n                (x, y), hypothesis = proposal\n                self.reason = hypothesis\n                return Action(6, x, y)\n        if self.policy in ("transfer", "forget", "partial", "shared", "shared_probe", "joint", "joint_flat", "compose", "associate", "review_pair", "review"):\n            proposal = self.memory.propose(obs, self.objects(obs))\n            if proposal:\n                (x, y), hypothesis = proposal\n                self.reason = hypothesis\n                return Action(6, x, y)\n        if self.policy in ("shared", "shared_probe", "joint", "joint_flat", "compose", "associate", "review_pair", "review") and not repeated_panels(self.objects(obs)):\n            action, self.reason = self.shared.choose(obs, self.objects(obs), candidates)\n            return action\n        if self.policy == "objects" or (self.policy in ("adaptive", "transfer", "forget", "partial") and\n                                         not repeated_panels(self.objects(obs))):\n            self.reason = "uniform choice among object centers and simple actions"\n            return self.rng.choice(candidates)\n        if self.policy in ("relational", "adaptive", "transfer", "forget", "partial", "shared", "shared_probe", "joint", "joint_flat", "compose", "associate", "review_pair", "review"):\n            proposal = self.relations.propose(obs, self.objects(obs))\n            if proposal:\n                (x, y), hypothesis = proposal\n                self.reason = "test visual relation hypothesis: " + hypothesis\n                return Action(6, x, y)\n        # Occasionally examine a coordinate omitted by object segmentation.\n        if 6 in legal and self.rng.random() < 0.05:\n            self.reason = "coordinate coverage outside current object hypothesis"\n            return Action(6, self.rng.randrange(obs.pixels.shape[1]),\n                          self.rng.randrange(obs.pixels.shape[0]))\n        anchor = self.anchor(obs)\n        position = (obs.levels, anchor.x, anchor.y) if anchor else None\n        scores = []\n        for action in candidates:\n            noise = self.rng.random() * 0.25\n            if action.id == 6:\n                effects = self.click_effects.get(action, [])\n                n = len(effects)\n                # Untried controls first; retain some probability for old no-ops.\n                effect = sum(e > 4 for e in effects) / n if n else 0.5\n                score = 2.0 / (1 + n) ** 0.5 + 1.5 * effect + noise\n                if action == self.last_action:\n                    score -= 0.4\n            elif position:\n                used = self.position_action_counts[(position, action.id)]\n                score = 3.0 / (1 + used) + noise\n                delta = self.motion.get(action.id)\n                if delta:\n                    target = (obs.levels, anchor.x + delta[0], anchor.y + delta[1])\n                    score += 2.0 / (1 + self.position_counts[target])\n                score -= min(2.0, self.blocked[(position, action.id)] * 0.7)\n            else:\n                score = 2.0 / (1 + self.action_counts[action]) ** 0.5 + noise\n            scores.append(score)\n        action = candidates[max(range(len(candidates)), key=scores.__getitem__)]\n        self.reason = "empirical effect / unvisited motion target; no state-equivalence claim"\n        return action\n\n    def observe(self, before: Observation, action: Action, after: Observation) -> None:\n        self.ledger.add(before, action, after)\n        self.steps += 1\n        self.action_counts[action] += 1\n        if self.policy not in ("probe", "relational", "adaptive", "transfer", "forget", "partial", "shared", "shared_probe", "joint", "joint_flat", "compose", "associate", "review_pair", "review"):\n            self.last_action = action\n            return\n        old_objects = self.objects(before)\n        new_objects = self.objects(after)\n        if self.policy in ("shared", "shared_probe", "joint", "joint_flat", "compose", "associate", "review_pair", "review"):\n            self.shared.observe(before, action, after, old_objects, new_objects)\n        if self.policy in ("partial", "shared", "shared_probe", "joint", "joint_flat", "compose", "associate", "review_pair", "review"):\n            self.partial.observe(before, action, after)\n        if self.policy in ("transfer", "forget", "partial", "shared", "shared_probe", "joint", "joint_flat", "compose", "associate", "review_pair", "review"):\n            self.memory.observe(before, action, after, old_objects, new_objects, self.relations)\n        if self.policy in ("relational", "adaptive", "transfer", "forget", "partial", "shared", "shared_probe", "joint", "joint_flat", "compose", "associate", "review_pair", "review"):\n            self.relations.observe(before, action, after, old_objects, new_objects)\n        if before.levels != after.levels or action.id == 0:\n            self.anchor_signature = None\n            self.motion.clear()\n            self.position_counts.clear()\n            self.position_action_counts.clear()\n            self.blocked.clear()\n            self.click_effects.clear()\n            self.action_counts.clear()\n        elif action.id == 6:\n            # Border changes are often timers, but this remains an explicit\n            # heuristic. The unfiltered pixels remain in the evidence ledger.\n            if before.pixels.shape == after.pixels.shape:\n                delta = before.pixels != after.pixels\n                interior = delta[2:-2, 2:-2] if min(delta.shape) > 4 else delta\n                self.click_effects[action].append(int(interior.sum()))\n        elif action.id in (1, 2, 3, 4, 5, 7):\n            old_groups = defaultdict(list)\n            new_groups = defaultdict(list)\n            for c in old_objects:\n                old_groups[c.signature].append(c)\n            for c in new_objects:\n                new_groups[c.signature].append(c)\n            movers = []\n            for signature, old in old_groups.items():\n                new = new_groups[signature]\n                if len(old) == len(new) == 1 and 2 <= old[0].size <= 100:\n                    a, b = old[0], new[0]\n                    dx, dy = b.x - a.x, b.y - a.y\n                    if 0 < abs(dx) + abs(dy) <= 16:\n                        movers.append((a, b, dx, dy))\n            if self.anchor_signature is None and movers:\n                displacements = {(dx, dy) for _, _, dx, dy in movers}\n                if len(displacements) == 1:\n                    self.anchor_signature = max(movers, key=lambda m: m[0].size)[0].signature\n            for a, b, dx, dy in movers:\n                if a.signature == self.anchor_signature:\n                    self.motion[action.id] = (dx, dy)\n            old_anchor = [c for c in old_objects if c.signature == self.anchor_signature]\n            new_anchor = [c for c in new_objects if c.signature == self.anchor_signature]\n            if len(old_anchor) == len(new_anchor) == 1:\n                a, b = old_anchor[0], new_anchor[0]\n                pos = (before.levels, a.x, a.y)\n                self.position_action_counts[(pos, action.id)] += 1\n                self.position_counts[(after.levels, b.x, b.y)] += 1\n                if (a.x, a.y) == (b.x, b.y):\n                    self.blocked[(pos, action.id)] += 1\n        self.last_action = action\n', 'reconsideration.py': '"""Read an existing observation again before spending another environment action.\n\nResearch pointers: primitive_operations (future distinctions), aggregation_checker\n(refinement after counterexamples), hysteresis_interpretations (versioned readings).\nThese motivate candidate construction and retained distinctions, not a proof that\nthe empirical visual model is a stable quotient of the unknown environment.\n"""\nfrom __future__ import annotations\n\nfrom collections import Counter, defaultdict, deque\nfrom dataclasses import dataclass\nimport numpy as np\n\nfrom association_dynamics import AssociationDynamics, ContactGoal, framed_patterns, pattern_pair, boundary_counts, fit_transforms, transform_pattern\nfrom shared_dynamics import entities, unique_entities, signature\n\n\ndef geometric_pairs(panels, shape):\n    h, w = shape\n    outer = [p for p in panels if min(p.x, p.y, w - p.x - p.width, h - p.y - p.height) <= 2]\n    inner = [p for p in panels if p not in outer]\n    return [(a, b) for a in outer for b in inner\n            if a.sig[0] == b.sig[0] and fit_transforms(a.pattern, b.pattern)]\n\n\n@dataclass(frozen=True)\nclass Landmark:\n    sig: tuple\n    x: int\n    y: int\n    colors: frozenset\n    parts: tuple\n\n    def goal(self, anchor, offset):\n        return ContactGoal(anchor, self.sig, self.x, self.y, offset)\n\n\ndef contact_landmarks(pixels, objects, pair, movers):\n    """Re-read neighboring small pieces as a bounded multicolor contact patch."""\n    h, w = pixels.shape\n    small = [c for c in objects if 1 <= c.size <= 36 and c.width <= 6 and c.height <= 6\n             and signature(c) not in movers and 4 <= c.x and 4 <= c.y\n             and c.x + c.width <= w - 4 and c.y + c.height <= h - 4\n             and not any(p.contains(*c.center) for p in pair)]\n    groups = []\n    remaining = set(range(len(small)))\n    while remaining:\n        start = min(remaining)\n        group = {start}\n        remaining.remove(start)\n        changed = True\n        while changed:\n            changed = False\n            for j in sorted(remaining):\n                c = small[j]\n                if not any(c.x <= small[i].x + small[i].width and small[i].x <= c.x + c.width\n                           and c.y <= small[i].y + small[i].height and small[i].y <= c.y + c.height for i in group):\n                    continue\n                joined = [small[i] for i in group] + [c]\n                if (max(o.x + o.width for o in joined) - min(o.x for o in joined) <= 6\n                        and max(o.y + o.height for o in joined) - min(o.y for o in joined) <= 6):\n                    group.add(j)\n                    remaining.remove(j)\n                    changed = True\n        cs = [small[i] for i in sorted(group)]\n        if sum(c.size for c in cs) < 2:\n            continue\n        x, y = min(c.x for c in cs), min(c.y for c in cs)\n        right, bottom = max(c.x + c.width for c in cs), max(c.y + c.height for c in cs)\n        patch = pixels[y:bottom, x:right]\n        sig = (-1, right - x, bottom - y, patch.tobytes().hex())\n        groups.append(Landmark(sig, x, y, frozenset(map(int, patch.flat)), tuple(signature(c) for c in cs)))\n    return groups\n\n\nclass ReconsiderationDynamics(AssociationDynamics):\n    def __init__(self, seed=0, retain_color=True):\n        self.retain_color = retain_color\n        self.review_active = False\n        self.review_revision = 0\n        self.review_events = []\n        self.history = deque(maxlen=48)\n        self.effect_samples = defaultdict(Counter)\n        self.reviewed = set()\n        super().__init__(seed)\n\n    def _clear_local(self):\n        super()._clear_local()\n        self.landmark_trials = Counter()\n        self.remembered_landmarks = {}\n        self.reading_pending = None\n\n    def _pair(self, panels, shape):\n        if not self.review_active:\n            return pattern_pair(panels, shape)\n        pairs = geometric_pairs(panels, shape)\n        return pairs[0] if len(pairs) == 1 else None\n\n    def review(self, obs, objects):\n        """Pure local inspection with respect to the environment: no step/API."""\n        key = obs.key()\n        if key in self.reviewed:\n            return\n        self.reviewed.add(key)\n        panels = framed_patterns(obs.pixels, objects)\n        strict = pattern_pair(panels, obs.pixels.shape)\n        candidates = geometric_pairs(panels, obs.pixels.shape)\n        self.stats[\'observations_reconsidered\'] += 1\n        if strict is None and len(candidates) == 1 and not self.review_active:\n            reference, target = candidates[0]\n            self.review_active = True\n            self.review_revision += 1\n            self.goal = self.repair = self.projected_pending = self.reading_pending = None\n            self.rejected_goals.clear()\n            self.stats[\'reading_refinements\'] += 1\n            self.review_events.append({\'revision\': self.review_revision, \'observation\': key,\n                \'level\': obs.levels, \'reason\': \'shape correspondence survives but ink condition is unmet\',\n                \'retained_current_ink\': reference.ink, \'retained_target_ink\': target.ink,\n                \'matching_transform_candidates\': sorted(fit_transforms(reference.pattern, target.pattern)),\n                \'external_actions_for_review\': 0})\n            # Read old raw observations under the new interpretation, instead\n            # of asking the environment to repeat already recorded experiments.\n            for record in self.history:\n                self._learn_contact(*record)\n            self.stats[\'history_transitions_reread\'] += len(self.history)\n\n    def _current_pair(self, obs, objects):\n        panels = framed_patterns(obs.pixels, objects)\n        pair = self._pair(panels, obs.pixels.shape)\n        if pair:\n            self.last_pair = pair\n        elif self.last_pair:\n            old, target = self.last_pair\n            current = [p for p in panels if (p.x, p.y, p.width, p.height) == (old.x, old.y, old.width, old.height)]\n            if len(current) == 1:\n                pair = current[0], target\n        return pair\n\n    def _effect(self, landmark, reading):\n        """Keep unknown/conflicting outcomes distinct from a measured identity."""\n        samples = self.effect_samples.get(landmark.sig)\n        if not samples:\n            return None\n        exact = {(after_shape, after_ink) for (before_shape, before_ink, after_shape, after_ink) in samples\n                 if (before_shape, before_ink) == reading}\n        if exact:\n            return next(iter(exact)) if len(exact) == 1 else None\n        transforms = set(range(8))\n        color_outputs = set()\n        identity_color = True\n        for before_shape, before_ink, after_shape, after_ink in samples:\n            transforms &= fit_transforms(before_shape, after_shape)\n            identity_color &= before_ink == after_ink\n            if before_ink == reading[1]:\n                color_outputs.add(after_ink)\n        shapes = {transform_pattern(reading[0], t) for t in transforms}\n        if identity_color:\n            color_outputs.add(reading[1])\n        if len(shapes) == len(color_outputs) == 1:\n            return next(iter(shapes)), next(iter(color_outputs))\n        return None\n\n    def _budget(self, obs):\n        rates = [(counts.most_common(1)[0][1], color, -counts.most_common(1)[0][0])\n                 for color, counts in self.boundary_changes.items()\n                 if counts.most_common(1)[0][0] < 0 and counts.most_common(1)[0][1] >= 3]\n        if not rates:\n            return None\n        _, color, rate = max(rates)\n        fuel = int(boundary_counts(obs.pixels)[color] // rate)\n        capacity = max(fuel, int(self.boundary_max[color] // rate))\n        return color, fuel, capacity\n\n    def _reading_route(self, scene, controls, obs, pair, cohort, landmarks, visible):\n        reference, target_panel = pair\n        anchor, offset, _ = cohort\n        budget = self._budget(obs)\n        if budget is None:\n            return None\n        resource_color, fuel, capacity = budget\n        supplies = [p for p in visible if resource_color in p.colors]\n        supply_keys = {p.sig for p in supplies}\n        operators = landmarks\n        target = target_panel.pattern, target_panel.ink\n        initial = reference.pattern, reference.ink\n        goal = ContactGoal(anchor, target_panel.sig, target_panel.x, target_panel.y, offset)\n        start = unique_entities(scene)[anchor]\n        scenes, transitions = {start: scene}, {}\n        def neighbors(xy):\n            if xy not in transitions:\n                edges = []\n                for key in controls:\n                    if key[0] not in (1, 2, 3, 4):\n                        continue\n                    after = self._predict(scenes[xy], key, obs)\n                    position = unique_entities(after).get(anchor) if after is not None else None\n                    if position is not None and position != xy:\n                        scenes.setdefault(position, after)\n                        edges.append((key, position))\n                transitions[xy] = edges\n            return transitions[xy]\n        queue = deque([(start, initial, fuel, 0, (), initial)])\n        best = {(start, initial, 0): fuel}\n        expanded = 0\n        while queue and expanded < 16000:\n            xy, reading, remaining, used, path, first_reading = queue.popleft()\n            expanded += 1\n            if reading == target and goal.distance(scenes[xy]) == 0 and path:\n                self.stats[\'reading_complete_model_plans\'] += 1\n                self.stats[\'reading_search_nodes\'] += expanded\n                return list(path), goal, first_reading\n            if remaining <= 0 or len(path) >= 100:\n                continue\n            for key, position in neighbors(xy):\n                after_reading, left, mask, known = reading, remaining - 1, used, True\n                for operator in operators:\n                    contact = operator.goal(anchor, offset)\n                    if contact.distance(scenes[position]) == 0 and contact.distance(scenes[xy]) != 0:\n                        result = self._effect(operator, after_reading)\n                        if result is None and operator.sig in supply_keys:\n                            # Explicit same-color replenishment hypothesis.\n                            result = after_reading\n                        if result is None:\n                            known = False\n                            break\n                        after_reading = result\n                if not known:\n                    continue\n                for i, supply in enumerate(supplies):\n                    if not mask & (1 << i) and supply.goal(anchor, offset).distance(scenes[position]) == 0:\n                        left, mask = capacity, mask | (1 << i)\n                state = position, after_reading, mask\n                if left <= best.get(state, -1):\n                    continue\n                best[state] = left\n                queue.append((position, after_reading, left, mask, path + (key,),\n                              first_reading if path else after_reading))\n        self.stats[\'reading_incomplete_model_plans\'] += 1\n        self.stats[\'reading_search_nodes\'] += expanded\n        return None\n\n    def _plan(self, scene, controls, obs):\n        if not self.review_active or not self.retain_color:\n            return super()._plan(scene, controls, obs)\n        objects = self.contact_objects\n        pair = self._current_pair(obs, objects)\n        cohort = self._cohort(scene, controls)\n        if pair is None or cohort is None:\n            return super()._plan(scene, controls, obs)\n        reference, target = pair\n        anchor, offset, movers = cohort\n        self.navigation_cohort = movers\n        visible = contact_landmarks(obs.pixels, objects, pair, movers)\n        for landmark in visible:\n            self.remembered_landmarks[landmark.sig, landmark.x, landmark.y] = landmark\n        # Persistent contact operators may be briefly occluded by the actor;\n        # consumed supplies must still be currently visible.\n        present = {(v.sig, v.x, v.y) for v in visible}\n        current = unique_entities(scene)\n        def occluded(p):\n            return any(x < p.x + p.sig[1] and p.x < x + sig[1]\n                       and y < p.y + p.sig[2] and p.y < y + sig[2]\n                       for sig, (x, y) in current.items() if sig in movers)\n        self.remembered_landmarks = {k: p for k, p in self.remembered_landmarks.items()\n                                    if k in present or occluded(p)}\n        landmarks = visible + [p for k, p in self.remembered_landmarks.items()\n                               if self.effect_samples.get(p.sig) and k not in present]\n        self.stats[\'compound_landmark_readings\'] += sum(len(p.parts) > 1 for p in visible)\n        complete = self._reading_route(scene, controls, obs, pair, cohort, landmarks, visible)\n        if complete:\n            path, self.goal, self.reading_pending = complete\n            self.joint_mode = False\n            return path\n        reading = reference.pattern, reference.ink\n        target_reading = target.pattern, target.ink\n        proposals = []\n        for landmark in visible:\n            effect = self._effect(landmark, reading)\n            trials = self.landmark_trials[reading, landmark.sig, landmark.x, landmark.y]\n            useful = effect is not None and ((reading[1] != target.ink and effect[1] == target.ink)\n                                            or (reading[0] != target.pattern and effect[0] == target.pattern))\n            color_hint = (reference.ink != target.ink and reference.ink in landmark.colors and target.ink in landmark.colors)\n            priority = 12 * trials - 24 * useful - 12 * color_hint\n            if effect == reading:\n                priority += 12\n            goal = landmark.goal(anchor, offset)\n            route = self._contact_route(scene, controls, obs, goal)\n            if route:\n                proposals.append((len(route) + priority, repr(goal), goal, route))\n        if reading == target_reading:\n            goal = ContactGoal(anchor, target.sig, target.x, target.y, offset)\n            route = self._contact_route(scene, controls, obs, goal)\n            if route:\n                proposals.append((len(route) - 40, repr(goal), goal, route))\n        if not proposals:\n            self.stats[\'review_no_probe_route\'] += 1\n            return []\n        _, _, goal, path = min(proposals)\n        budget = self._budget(obs)\n        if budget:\n            color, fuel, _ = budget\n            if len(path) + 3 >= fuel:\n                supplies = []\n                for landmark in visible:\n                    if color not in landmark.colors:\n                        continue\n                    contact = landmark.goal(anchor, offset)\n                    route = self._contact_route(scene, controls, obs, contact)\n                    if route and len(route) < fuel:\n                        supplies.append((len(route), repr(contact), contact, route))\n                if supplies:\n                    _, _, goal, path = min(supplies)\n        self.goal, self.joint_mode = goal, False\n        self.stats[\'review_probe_plans\'] += 1\n        return path\n\n    def choose(self, obs, objects, candidates):\n        self.review(obs, objects)\n        return super().choose(obs, objects, candidates)\n\n    def _learn_contact(self, before, after, old_objects, new_objects, cohort):\n        if (before.levels != after.levels or before.state in (\'NOT_PLAYED\', \'WIN\', \'GAME_OVER\')\n                or after.state in (\'WIN\', \'GAME_OVER\') or cohort is None):\n            return\n        old_pairs = geometric_pairs(framed_patterns(before.pixels, old_objects), before.pixels.shape)\n        new_pairs = geometric_pairs(framed_patterns(after.pixels, new_objects), after.pixels.shape)\n        if len(old_pairs) != 1 or len(new_pairs) != 1:\n            return\n        old_pair, new_pair = old_pairs[0], new_pairs[0]\n        old_reading, new_reading = (old_pair[0].pattern, old_pair[0].ink), (new_pair[0].pattern, new_pair[0].ink)\n        anchor, offset, movers = cohort\n        old_scene = entities(old_objects, before.pixels.shape)\n        new_scene = entities(new_objects, after.pixels.shape)\n        for landmark in contact_landmarks(before.pixels, old_objects, old_pair, movers):\n            goal = landmark.goal(anchor, offset)\n            if goal.distance(new_scene) == 0 and goal.distance(old_scene) != 0:\n                self.effect_samples[landmark.sig][(*old_reading, *new_reading)] += 1\n                self.landmark_trials[old_reading, landmark.sig, landmark.x, landmark.y] += 1\n                self.stats[\'contact_reading_samples\'] += 1\n                if old_reading[1] != new_reading[1]:\n                    self.stats[\'ink_change_samples\'] += 1\n\n    def observe(self, before, action, after, old_objects, new_objects):\n        controls = [(a,) for a in before.available]\n        cohort = self._cohort(entities(old_objects, before.pixels.shape), controls)\n        if action.id == 0:\n            cohort = None\n        record = before, after, old_objects, new_objects, cohort\n        if self.review_active and self.retain_color:\n            self._learn_contact(*record)\n        self.history.append(record)\n        prediction, self.reading_pending = self.reading_pending, None\n        if prediction is not None and before.levels == after.levels and after.state not in (\'WIN\', \'GAME_OVER\'):\n            pairs = geometric_pairs(framed_patterns(after.pixels, new_objects), after.pixels.shape)\n            if len(pairs) == 1:\n                actual = pairs[0][0].pattern, pairs[0][0].ink\n                self.stats[\'reading_predictions_checked\'] += 1\n                self.stats[\'reading_prediction_failures\'] += actual != prediction\n        super().observe(before, action, after, old_objects, new_objects)\n\n    def summary(self):\n        result = super().summary()\n        result[\'retains_ink_as_condition\'] = self.retain_color\n        result[\'reading_revision\'] = self.review_revision\n        result[\'review_events\'] = self.review_events\n        result[\'contact_reading_evidence\'] = {repr(k): {repr(e): n for e, n in v.items()}\n                                             for k, v in self.effect_samples.items()}\n        result[\'reconsideration_scope\'] = \'Local re-reading and checked empirical models; no environment action, no certified state equivalence\'\n        return result\n', 'shared_dynamics.py': '"""Empirical action models independent of glyphs, clicks and level completion.\n\nThe scene projection and translation/terrain models are hypotheses, not state\nquotients. Full observations stay in the runner/TransitionLedger. Ordered graph\nedges retain conflicting outcomes; predicted motion is checked after each step.\nNo game names, fixed palette values, environment internals or reference goals.\n"""\nfrom __future__ import annotations\n\nfrom collections import Counter, defaultdict, deque\nfrom dataclasses import dataclass\nimport random\n\nimport numpy as np\n\n\ndef signature(obj):\n    return obj.color, obj.width, obj.height, obj.shape.hex()\n\n\ndef entities(objects, shape):\n    h, w = shape\n    # Thin boundary strips are a display heuristic, not certified irrelevant.\n    return tuple(sorted((signature(c), c.x, c.y) for c in objects\n                        if 2 <= c.size\n                        and not ((c.height <= 2 and (c.y <= 1 or c.y + c.height >= h - 1))\n                                 or (c.width <= 2 and (c.x <= 1 or c.x + c.width >= w - 1)))))\n\n\ndef unique_entities(scene):\n    groups = defaultdict(list)\n    for sig, x, y in scene:\n        groups[sig].append((x, y))\n    return {sig: positions[0] for sig, positions in groups.items() if len(positions) == 1}\n\n\nclass OrderedEvidence:\n    """An empirical directed graph; outcome conflicts disable a reusable edge."""\n    def __init__(self):\n        self.edges = defaultdict(Counter)\n        self.conflicts = 0\n\n    def add(self, before, action, after, progress=0, terminal=False):\n        outcomes = self.edges[before, action]\n        outcome = after, int(progress), bool(terminal)\n        if outcomes and outcome not in outcomes:\n            self.conflicts += 1\n        outcomes[outcome] += 1\n\n    def predict(self, state, action):\n        outcomes = self.edges.get((state, action), {})\n        return next(iter(outcomes)) if len(outcomes) == 1 else None\n\n    def route(self, start, goals, legal, max_nodes=512):\n        queue, seen = deque([(start, [])]), {start}\n        while queue and len(seen) <= max_nodes:\n            state, path = queue.popleft()\n            if state in goals:\n                return path\n            for action in legal:\n                prediction = self.predict(state, action)\n                if prediction is None:\n                    continue\n                after, _, terminal = prediction\n                if not terminal and after not in seen:\n                    seen.add(after)\n                    queue.append((after, path + [action]))\n        return None\n\n\n@dataclass(frozen=True)\nclass GoalHypothesis:\n    mover: tuple\n    target: tuple\n    target_x: int\n    target_y: int\n    axes: tuple\n\n    def template(self):\n        # Relative role and geometry transfer; no fixed color/coordinate goal.\n        return self.mover[1:3], self.target[1:3], self.axes, self.mover[0] == self.target[0]\n\n    def distance(self, scene):\n        current = unique_entities(scene).get(self.mover)\n        if current is None or (self.target, self.target_x, self.target_y) not in scene:\n            return float(\'inf\')\n        a = current[0] + self.mover[1] / 2, current[1] + self.mover[2] / 2\n        b = self.target_x + self.target[1] / 2, self.target_y + self.target[2] / 2\n        if len(self.axes) == 1:\n            return abs(a[self.axes[0]] - b[self.axes[0]])\n        # Contact is a candidate objective; only a reported level change confirms success.\n        return sum(max(0, abs(a[i] - b[i]) - (self.mover[i + 1] + self.target[i + 1]) / 2)\n                   for i in self.axes)\n\n\nclass SharedDynamics:\n    def __init__(self, seed=0, planning=True):\n        self.rng = random.Random(seed)\n        self.planning = planning\n        self.level = None\n        self.confirmed_goals = Counter()\n        self.stats = Counter()\n        self.events = []\n        self.pending = None\n        self.repair = None\n        self.activity = defaultdict(Counter)\n        self.repair_support = Counter()\n        self._clear_local()\n\n    def _clear_local(self):\n        self.graph = OrderedEvidence()\n        self.models = defaultdict(lambda: defaultdict(Counter))\n        self.control_trials = Counter()\n        self.state_trials = Counter()\n        self.visits = Counter()\n        self.blocked = set()\n        self.passable = Counter()\n        self.obstacles = Counter()\n        self.goal = None\n        self.rejected_goals = set()\n        self.pending = None\n        self.repair = None\n        self.activity = defaultdict(Counter)\n\n    def _ensure_level(self, level):\n        if self.level != level:\n            self._clear_local()\n            self.level = level\n\n    @staticmethod\n    def control_key(action, objects):\n        if action.id != 6:\n            return (action.id,)\n        containing = [c for c in objects if c.x <= action.x < c.x + c.width\n                      and c.y <= action.y < c.y + c.height]\n        if not containing:\n            return 6, \'coordinate\', action.x, action.y\n        obj = min(containing, key=lambda c: c.size)\n        sig = signature(obj)\n        peers = sorted((c.y, c.x) for c in objects if signature(c) == sig)\n        return 6, sig, peers.index((obj.y, obj.x)), len(peers)\n\n    def _vectors(self, key, scene):\n        current = unique_entities(scene)\n        vectors = {}\n        for sig, counts in self.models.get(key, {}).items():\n            if sig in current and len(counts) == 1:\n                vectors[sig] = next(iter(counts))\n        return vectors\n\n    def _predict(self, scene, key, obs):\n        # Exact observed ordered transitions take precedence over translation.\n        state_key = (tuple(obs.available), obs.state, scene)\n        outcomes = self.graph.edges.get((state_key, key), {})\n        if len(outcomes) > 1:\n            return None\n        observed = self.graph.predict(state_key, key)\n        if observed is not None:\n            after_state, progress, terminal = observed\n            available, state, after = after_state\n            if not progress and not terminal and tuple(available) == tuple(obs.available) and state == obs.state:\n                return after if after != scene else None\n            return None\n        vectors = self._vectors(key, scene)\n        if not vectors:\n            return None\n        h, w = obs.pixels.shape\n        current = unique_entities(scene)\n        moving_colors = {sig[0] for sig in vectors}\n        for sig, (dx, dy) in vectors.items():\n            x, y = current[sig]\n            nx, ny = x + dx, y + dy\n            if not (0 <= nx <= w - sig[1] and 0 <= ny <= h - sig[2]):\n                return None\n            if (scene, key) in self.blocked:\n                return None\n            if key[0] in (1, 2, 3, 4):\n                patch = obs.pixels[ny:ny + sig[2], nx:nx + sig[1]]\n                bad = sum(int(np.count_nonzero(patch == c)) for c in self.obstacles\n                          if c not in moving_colors and not self.passable[c])\n                if bad >= max(1, patch.size // 3):\n                    return None\n        return tuple(sorted((sig, x + vectors.get(sig, (0, 0))[0],\n                             y + vectors.get(sig, (0, 0))[1]) for sig, x, y in scene))\n\n    def _goals(self, scene, controls, obs):\n        axes = defaultdict(set)\n        for key in controls:\n            for sig, vector in self._vectors(key, scene).items():\n                axes[sig].update(i for i, delta in enumerate(vector) if delta)\n        h, w = obs.pixels.shape\n        goals = []\n        for mover, motion_axes in axes.items():\n            for target, x, y in scene:\n                if target in axes or target[1] > 12 or target[2] > 12:\n                    continue\n                if x < 4 or y < 4 or x + target[1] > w - 4 or y + target[2] > h - 4:\n                    continue\n                # One-dimensional alignment requires a visible same-color reference.\n                if len(motion_axes) == 1 and mover[0] != target[0]:\n                    continue\n                goal = GoalHypothesis(mover, target, x, y, tuple(sorted(motion_axes)))\n                distance = goal.distance(scene)\n                if goal not in self.rejected_goals and 0 < distance < float(\'inf\'):\n                    priority = distance - 32 * min(2, self.confirmed_goals[goal.template()])\n                    goals.append((priority, repr(goal), goal))\n        return [g for _, _, g in sorted(goals)]\n\n    def _plan(self, scene, controls, obs):\n        if self.goal is None or not np.isfinite(self.goal.distance(scene)):\n            goals = self._goals(scene, controls, obs)\n            self.goal = goals[0] if goals else None\n            if self.goal is not None:\n                self.stats[\'goal_hypotheses\'] += 1\n                if len(self.events) < 128:\n                    self.events.append({\'status\': \'goal_hypothesis\', \'level\': self.level,\n                                        \'goal\': repr(self.goal)})\n        queue, seen = deque([(scene, [])]), {scene}\n        start_distance = self.goal.distance(scene) if self.goal else 0\n        best, best_value = [], -float(\'inf\')\n        while queue and len(seen) <= 384:\n            state, path = queue.popleft()\n            if path:\n                distance = self.goal.distance(state) if self.goal else 0\n                if self.goal and distance == 0:\n                    return path\n                gain = start_distance - distance\n                value = 3 * gain + 2 / (1 + self.visits[state]) - .3 * len(path)\n                if value > best_value:\n                    best, best_value = path, value\n            if len(path) >= 16:\n                continue\n            for key in controls:\n                after = self._predict(state, key, obs)\n                if after is not None and after != state and after not in seen:\n                    seen.add(after)\n                    queue.append((after, path + [key]))\n        if best and best_value > 0:\n            return best\n        if self.goal:\n            self.rejected_goals.add(self.goal)\n            self.goal = None\n            self.stats[\'unreachable_goal_hypotheses\'] += 1\n        return []\n\n    def choose(self, obs, objects, candidates):\n        self._ensure_level(obs.levels)\n        scene = entities(objects, obs.pixels.shape)\n        controls = {self.control_key(a, objects): a for a in candidates}\n        visible_signatures = {sig for sig, _, _ in scene}\n        filtered = {k: a for k, a in controls.items()\n                    if k[0] != 6 or k[1] == \'coordinate\' or k[1] in visible_signatures}\n        controls = filtered or controls\n        untried = [key for key in controls if self.control_trials[key] == 0]\n        repair_key = None\n        if self.repair and self.repair[\'target\'] in controls:\n            target = self.repair[\'target\']\n            if self.repair[\'phase\'] == \'retry\':\n                repair_key = target\n                self.repair[\'phase\'] = \'checking\'\n            else:\n                helpers = [k for k in controls if k != target and k not in self.repair[\'tried\']\n                           and self.activity[k][\'configuration\'] and not self.activity[k][\'motion\']]\n                if helpers:\n                    repair_key = max(helpers, key=lambda k: (self.repair_support[target, k], self.rng.random()))\n                    self.repair[\'helper\'] = repair_key\n                    self.repair[\'tried\'].add(repair_key)\n                    self.repair[\'phase\'] = \'retry\'\n                    self.stats[\'preparation_probes\'] += 1\n                else:\n                    self.repair = None\n        if repair_key is not None:\n            key, path, reason = repair_key, [repair_key], \'shared: test preparation then retry blocked goal action\'\n        elif untried:\n            # Repeated small controls are candidates, not assumed buttons.\n            def priority(key):\n                if key[0] != 6:\n                    return 10\n                if key[1] == \'coordinate\':\n                    return 0\n                sig, _, count = key[1:]\n                return min(count, 4) * 2 - sig[1] * sig[2] / 128\n            key = max(untried, key=lambda k: (priority(k), self.rng.random()))\n            path, reason = [key], \'shared: measure an untried control\'\n        else:\n            path = self._plan(scene, controls, obs) if self.planning else []\n            if path:\n                key, reason = path[0], \'shared: compose observed state transitions toward a goal/frontier hypothesis\'\n                self.stats[\'planned_actions\'] += 1\n                self.stats[\'multi_step_plans\'] += len(path) > 1\n            else:\n                key = min(controls, key=lambda k: (self.state_trials[scene, k], self.control_trials[k], self.rng.random()))\n                path, reason = [key], \'shared: probe least-observed scene/control condition\'\n        predicted = self._predict(scene, key, obs)\n        self.pending = {\'scene\': scene, \'key\': key, \'predicted\': predicted,\n                        \'goal\': self.goal, \'vectors\': self._vectors(key, scene)}\n        self.stats[\'chosen_actions\'] += 1\n        return controls[key], reason\n\n    def observe(self, before, action, after, old_objects, new_objects):\n        self._ensure_level(before.levels)\n        old = entities(old_objects, before.pixels.shape)\n        new = entities(new_objects, after.pixels.shape)\n        key = self.control_key(action, old_objects)\n        pending, self.pending = self.pending, None\n        progress = after.levels - before.levels\n        self.stats[\'observations\'] += 1\n        if action.id == 0:\n            # Reset can alter hidden state; do not retain local transition claims.\n            self._clear_local()\n            self.level = after.levels\n            self.stats[\'resets\'] += 1\n            return\n        self.control_trials[key] += 1\n        self.state_trials[old, key] += 1\n        self.visits[new] += 1\n        conflicts = self.graph.conflicts\n        self.graph.add((tuple(before.available), before.state, old), key,\n                       (tuple(after.available), after.state, new), progress, after.state in (\'WIN\', \'GAME_OVER\'))\n        self.stats[\'projection_conflicts\'] += self.graph.conflicts - conflicts\n        if progress > 0:\n            if pending and pending[\'goal\'] and pending[\'predicted\'] is not None:\n                goal = pending[\'goal\']\n                if goal.distance(pending[\'predicted\']) == 0:\n                    self.confirmed_goals[goal.template()] += 1\n                    self.stats[\'goal_confirmations\'] += 1\n                    self.events.append({\'status\': \'goal_supported_by_completion\', \'level\': after.levels,\n                                        \'template\': repr(goal.template())})\n            self.stats[\'level_completions_observed\'] += progress\n            self._ensure_level(after.levels)\n            return\n        if after.state == \'GAME_OVER\':\n            self.stats[\'terminal_failures\'] += 1\n            if pending:\n                self.blocked.add((old, key))\n            return\n        old_unique, new_unique = unique_entities(old), unique_entities(new)\n        expected_vectors = pending[\'vectors\'] if pending else self._vectors(key, old)\n        matches = False\n        if pending and expected_vectors:\n            matches = all(sig in new_unique and\n                          new_unique[sig] == (old_unique[sig][0] + dx, old_unique[sig][1] + dy)\n                          for sig, (dx, dy) in expected_vectors.items())\n            if pending[\'predicted\'] is not None:\n                self.stats[\'predictions_checked\'] += 1\n            if not matches and pending[\'predicted\'] is not None:\n                self.stats[\'prediction_failures\'] += 1\n                self.blocked.add((old, key))\n                if self.goal and not self.repair:\n                    self.repair = {\'target\': key, \'phase\': \'prepare\', \'tried\': set()}\n        if self.repair and self.repair[\'phase\'] == \'checking\':\n            if matches:\n                self.repair_support[key, self.repair[\'helper\']] += 1\n                self.stats[\'preparation_sequences_supported\'] += 1\n                if len(self.events) < 128:\n                    self.events.append({\'status\': \'preparation_restored_motion\', \'level\': self.level,\n                                        \'helper\': repr(self.repair[\'helper\']), \'target\': repr(key)})\n                self.repair = None\n            else:\n                self.repair[\'phase\'] = \'prepare\'\n        moved = False\n        for sig, (x, y) in old_unique.items():\n            if sig not in new_unique or sig[1] > 24 or sig[2] > 24 or sig[1] * sig[2] > 256:\n                continue\n            nx, ny = new_unique[sig]\n            dx, dy = nx - x, ny - y\n            if 0 < abs(dx) + abs(dy) <= 16:\n                moved = True\n                self.models[key][sig][dx, dy] += 1\n                self.stats[\'motion_effects_measured\'] += 1\n                if action.id in (1, 2, 3, 4):\n                    patch = before.pixels[ny:ny + sig[2], nx:nx + sig[1]]\n                    color, count = Counter(map(int, patch.flat)).most_common(1)[0]\n                    if color != sig[0] and count >= patch.size * .7:\n                        self.passable[color] += 1\n        if before.pixels.shape == after.pixels.shape:\n            delta = before.pixels != after.pixels\n            interior = delta[2:-2, 2:-2] if min(delta.shape) > 4 else delta\n            if moved:\n                self.activity[key][\'motion\'] += 1\n            elif interior.any():\n                self.activity[key][\'configuration\'] += 1\n            else:\n                self.activity[key][\'no_visible_effect\'] += 1\n        if action.id in (1, 2, 3, 4):\n            for sig, (dx, dy) in expected_vectors.items():\n                if old_unique.get(sig) == new_unique.get(sig):\n                    x, y = old_unique[sig]\n                    nx, ny = x + dx, y + dy\n                    if nx < 0 or ny < 0:\n                        continue\n                    patch = before.pixels[ny:ny + sig[2], nx:nx + sig[1]]\n                    if patch.size:\n                        color, count = Counter(map(int, patch.flat)).most_common(1)[0]\n                        if color != sig[0] and count >= patch.size * .7 and not self.passable[color]:\n                            self.obstacles[color] += 1\n        if self.goal and self.goal.distance(new) == 0:\n            self.rejected_goals.add(self.goal)\n            self.goal = None\n            self.stats[\'goal_reached_without_completion\'] += 1\n\n    def summary(self):\n        return {\'stats\': dict(sorted(self.stats.items())),\n                \'confirmed_goal_templates\': {repr(k): v for k, v in self.confirmed_goals.items()},\n                \'events\': self.events, \'certified_state_merges\': 0,\n                \'scope\': \'Observed ordered edges and checked translation/contact hypotheses; no game-specific goals or universal transition proof\'}\n', 'association_dynamics.py': '"""Empirical correspondence between visible patterns and contact operations.\n\nNo landmark is assigned a meaning in advance. Similar framed bitmaps suggest\na matching objective; contacts are experiments, and changed patterns identify\ncandidate preparation operations. These are visual hypotheses, not rules.\n"""\nfrom __future__ import annotations\n\nfrom collections import Counter, defaultdict, deque\nfrom dataclasses import dataclass\nimport numpy as np\n\nfrom joint_dynamics import JointDynamics\nfrom shared_dynamics import entities, unique_entities, signature\n\n\n@dataclass(frozen=True)\nclass FramedPattern:\n    sig: tuple\n    x: int\n    y: int\n    width: int\n    height: int\n    ink: int\n    pattern: tuple\n\n    def contains(self, x, y):\n        return self.x <= x < self.x + self.width and self.y <= y < self.y + self.height\n\n\ndef framed_patterns(pixels, objects):\n    """Detect square outlines and normalize exact integer bitmap enlargement."""\n    result = []\n    for c in objects:\n        if not 5 <= c.width == c.height <= 16:\n            continue\n        patch = pixels[c.y:c.y + c.height, c.x:c.x + c.width]\n        border = np.concatenate((patch[0], patch[-1], patch[:, 0], patch[:, -1]))\n        if np.mean(border == c.color) < .95:\n            continue\n        colors = set(map(int, np.unique(patch))) - {c.color}\n        if len(colors) != 1:\n            continue\n        bits = patch != c.color\n        ys, xs = np.where(bits)\n        if not len(xs):\n            continue\n        bits = bits[ys.min():ys.max() + 1, xs.min():xs.max() + 1]\n        for scale in range(min(bits.shape), 1, -1):\n            h, w = bits.shape\n            if h % scale == w % scale == 0:\n                small = bits[::scale, ::scale]\n                if np.array_equal(np.repeat(np.repeat(small, scale, 0), scale, 1), bits):\n                    bits = small\n                    break\n        pattern = bits.shape, np.packbits(bits).tobytes().hex()\n        result.append(FramedPattern(signature(c), c.x, c.y, c.width, c.height,\n                                    next(iter(colors)), pattern))\n    return result\n\n\ndef pattern_pair(panels, shape):\n    h, w = shape\n    peripheral = [p for p in panels if min(p.x, p.y, w - p.x - p.width, h - p.y - p.height) <= 2]\n    interior = [p for p in panels if p not in peripheral]\n    pairs = [(a, b) for a in peripheral for b in interior\n             if a.sig[0] == b.sig[0] and a.ink == b.ink and a.pattern[0] == b.pattern[0]]\n    return pairs[0] if len(pairs) == 1 else None\n\n\ndef boundary_counts(pixels):\n    mask = np.ones(pixels.shape, dtype=bool)\n    if min(pixels.shape) > 8:\n        mask[4:-4, 4:-4] = False\n    return np.bincount(pixels[mask], minlength=16)\n\n\ndef transform_pattern(pattern, transform):\n    shape, packed = pattern\n    bits = np.unpackbits(np.frombuffer(bytes.fromhex(packed), dtype=np.uint8))[:np.prod(shape)].reshape(shape)\n    if transform >= 4:\n        bits = np.fliplr(bits)\n    bits = np.rot90(bits, transform % 4)\n    return bits.shape, np.packbits(bits).tobytes().hex()\n\n\ndef fit_transforms(before, after):\n    return {k for k in range(8) if transform_pattern(before, k) == after}\n\n\n@dataclass(frozen=True)\nclass ContactGoal:\n    mover: tuple\n    target: tuple\n    target_x: int\n    target_y: int\n    offset: tuple\n    axes: tuple = (0, 1)\n\n    def template(self):\n        return \'contact_center\', self.mover[1:3], self.target[1:3], self.axes\n\n    def distance(self, scene):\n        xy = unique_entities(scene).get(self.mover)\n        if xy is None:\n            return float(\'inf\')\n        goal = self.target_x + self.target[1] / 2, self.target_y + self.target[2] / 2\n        return sum(max(0, abs(xy[i] + self.offset[i] - goal[i]) - .5) for i in (0, 1))\n\n\nclass AssociationDynamics(JointDynamics):\n    def _pair(self, panels, shape):\n        return pattern_pair(panels, shape)\n\n    def __init__(self, seed=0, contacts=True):\n        self.contacts = contacts\n        self.pattern_operators = Counter()\n        self.pattern_transforms = {}\n        self.motion_priors = {}\n        self.terrain_priors = None\n        super().__init__(seed, minimum_goals=1)\n\n    def _clear_local(self):\n        super()._clear_local()\n        self.contact_trials = Counter()\n        self.contact_pending = None\n        self.last_pair = None\n        self.boundary_changes = defaultdict(Counter)\n        self.boundary_max = np.zeros(16, dtype=int)\n        self.navigation_cohort = None\n        self.prior_controls = set()\n        self.known_contacts = {}\n\n    def _measured_vectors(self, key, scene):\n        vectors = super()._vectors(key, scene)\n        if key[0] in (1, 2, 3, 4):\n            current = unique_entities(scene)\n            for sig, counts in self.models.get(key, {}).items():\n                if sig in current and sig not in vectors:\n                    vector, support = counts.most_common(1)[0]\n                    if support >= 3 and support >= 3 * (sum(counts.values()) - support):\n                        vectors[sig] = vector\n        return vectors\n\n    def _vectors(self, key, scene):\n        vectors = self._measured_vectors(key, scene)\n        if self.navigation_cohort is not None:\n            return {sig: v for sig, v in vectors.items() if sig in self.navigation_cohort}\n        return vectors\n\n    def _cohort(self, scene, controls):\n        current = unique_entities(scene)\n        catalogue = {key: self._measured_vectors(key, scene) for key in controls if key[0] in (1, 2, 3, 4)}\n        axes = defaultdict(set)\n        for vectors in catalogue.values():\n            for sig, vector in vectors.items():\n                axes[sig].update(i for i, d in enumerate(vector) if d)\n        movers = [s for s, ax in axes.items() if len(ax) == 2]\n        if not movers:\n            return None\n        anchor = max(movers, key=lambda s: (s[1] * s[2], repr(s)))\n        x, y = current[anchor]\n        cohort = [anchor]\n        for sig in movers:\n            if sig == anchor:\n                continue\n            sx, sy = current[sig]\n            touching = sx <= x + anchor[1] and x <= sx + sig[1] and sy <= y + anchor[2] and y <= sy + sig[2]\n            shared = [v for v in catalogue.values() if anchor in v and sig in v]\n            if touching and shared and all(v[anchor] == v[sig] for v in shared):\n                cohort.append(sig)\n        left = min(current[s][0] for s in cohort)\n        top = min(current[s][1] for s in cohort)\n        right = max(current[s][0] + s[1] for s in cohort)\n        bottom = max(current[s][1] + s[2] for s in cohort)\n        return anchor, ((left + right) / 2 - x, (top + bottom) / 2 - y), set(cohort)\n\n    def _contact_route(self, scene, controls, obs, goal):\n        queue, seen = deque([(scene, [])]), {scene}\n        while queue and len(seen) <= 1024:\n            state, path = queue.popleft()\n            if len(path) >= 64:\n                continue\n            for key in controls:\n                if key[0] not in (1, 2, 3, 4):\n                    continue\n                after = self._predict(state, key, obs)\n                if after is None or after == state:\n                    continue\n                route = path + [key]\n                if goal.distance(after) == 0:\n                    return route\n                if after not in seen:\n                    seen.add(after)\n                    queue.append((after, route))\n        return None\n\n    def _resource_route(self, scene, controls, obs, pair, cohort, objects):\n        """Compose translations, pattern permutations and replenishment.\n\n        Every finite path spends one observed budget unit per primitive action.\n        A visually suggested supply is consumable once in the imagined path.\n        Unconfirmed supplies and fitted transforms remain checked hypotheses.\n        """\n        reference, destination = pair\n        anchor, offset, _ = cohort\n        rates = [(counts.most_common(1)[0][1], color, -counts.most_common(1)[0][0])\n                 for color, counts in self.boundary_changes.items()\n                 if counts.most_common(1)[0][0] < 0 and counts.most_common(1)[0][1] >= 3]\n        if not rates:\n            return None\n        _, color, rate = max(rates)\n        fuel = int(boundary_counts(obs.pixels)[color] // rate)\n        capacity = max(fuel, int(self.boundary_max[color] // rate))\n        if fuel <= 0 or capacity <= 0:\n            return None\n        supplies, operators = [], []\n        for c in objects:\n            if not (2 <= c.size <= 36 and c.width <= 6 and c.height <= 6):\n                continue\n            if any(p.contains(*c.center) for p in pair):\n                continue\n            goal = ContactGoal(anchor, signature(c), c.x, c.y, offset)\n            if c.color == color:\n                supplies.append(goal)\n            if self.pattern_transforms.get(signature(c)):\n                operators.append((goal, self.pattern_transforms[signature(c)]))\n        # The actor may occlude a just-used operator; its observed location is\n        # retained as a hypothesis until the next level/reset.\n        for goal, choices in self.known_contacts.values():\n            if goal.target in self.pattern_transforms and all(g.target_x != goal.target_x or g.target_y != goal.target_y for g, _ in operators):\n                operators.append((ContactGoal(anchor, goal.target, goal.target_x, goal.target_y, offset), choices))\n        if reference.pattern != destination.pattern and not operators:\n            return None\n        goal = ContactGoal(anchor, destination.sig, destination.x, destination.y, offset)\n        start = unique_entities(scene)[anchor]\n        scenes, transitions = {start: scene}, {}\n        def neighbors(xy):\n            if xy not in transitions:\n                result = []\n                for key in controls:\n                    if key[0] not in (1, 2, 3, 4):\n                        continue\n                    after = self._predict(scenes[xy], key, obs)\n                    if after is not None:\n                        target = unique_entities(after).get(anchor)\n                        if target is not None and target != xy:\n                            scenes.setdefault(target, after)\n                            result.append((key, target))\n                transitions[xy] = result\n            return transitions[xy]\n        # Dominance is restricted to this explicit product model, not a claim\n        # that pixel histories with equal coordinates are equivalent.\n        queue = deque([(start, reference.pattern, fuel, 0, ())])\n        best_fuel = {(start, reference.pattern, 0): fuel}\n        expanded = 0\n        while queue and expanded < 12000:\n            xy, pattern, left, used, path = queue.popleft()\n            expanded += 1\n            if pattern == destination.pattern and goal.distance(scenes[xy]) == 0 and path:\n                self.stats[\'resource_product_plans\'] += 1\n                self.stats[\'resource_product_nodes\'] += expanded\n                return list(path), goal\n            if left <= 0 or len(path) >= 100:\n                continue\n            for key, target in neighbors(xy):\n                new_pattern, remaining, mask = pattern, left - 1, used\n                ambiguous = False\n                for contact, choices in operators:\n                    if contact.distance(scenes[target]) == 0 and contact.distance(scenes[xy]) != 0:\n                        predictions = {transform_pattern(new_pattern, k) for k in choices}\n                        if len(predictions) == 1:\n                            new_pattern = next(iter(predictions))\n                        else:\n                            ambiguous = True\n                if ambiguous:\n                    # Unknown effects cannot silently become no-ops in a\n                    # purported complete plan. Fallback contact experiments\n                    # can distinguish the still-live hypotheses.\n                    continue\n                for i, supply in enumerate(supplies):\n                    if not mask & (1 << i) and supply.distance(scenes[target]) == 0:\n                        remaining = capacity\n                        mask |= 1 << i\n                state = target, new_pattern, mask\n                if remaining <= best_fuel.get(state, -1):\n                    continue\n                best_fuel[state] = remaining\n                queue.append((target, new_pattern, remaining, mask, path + (key,)))\n        self.stats[\'resource_product_incomplete\'] += 1\n        self.stats[\'resource_product_nodes\'] += expanded\n        return None\n\n    def _plan(self, scene, controls, obs):\n        if not self.contacts:\n            return super()._plan(scene, controls, obs)\n        # Object extraction is supplied by Explorer; reuse the current cache.\n        objects = self.contact_objects\n        panels = framed_patterns(obs.pixels, objects)\n        pair = self._pair(panels, obs.pixels.shape)\n        if pair:\n            self.last_pair = pair\n        elif self.last_pair:\n            # A mover can occlude an outline. Keep its last observed location,\n            # while requiring the peripheral reference to remain observable.\n            reference, destination = self.last_pair\n            current = [p for p in panels if (p.x, p.y, p.width, p.height) ==\n                       (reference.x, reference.y, reference.width, reference.height)]\n            if len(current) == 1:\n                pair = current[0], destination\n        cohort = self._cohort(scene, controls)\n        if pair is None or cohort is None:\n            self.navigation_cohort = None\n            return super()._plan(scene, controls, obs)\n        reference, destination = pair\n        anchor, offset, movers = cohort\n        self.navigation_cohort = movers\n        context = reference.pattern, destination.pattern\n        complete = self._resource_route(scene, controls, obs, pair, cohort, objects)\n        if complete:\n            path, goal = complete\n            self.goal = goal\n            self.joint_mode = False\n            self.contact_pending = None\n            self.stats[\'contact_plans\'] += 1\n            return path\n        targets = []\n        if reference.pattern == destination.pattern:\n            targets.append((destination.sig, destination.x, destination.y, 0))\n        else:\n            h, w = obs.pixels.shape\n            for obj in objects:\n                sig = signature(obj)\n                if not (2 <= obj.size <= 36 and obj.width <= 6 and obj.height <= 6):\n                    continue\n                if sig in movers or obj.x < 4 or obj.y < 4 or obj.x + obj.width > w - 4 or obj.y + obj.height > h - 4:\n                    continue\n                if any(p.contains(*obj.center) for p in pair):\n                    continue\n                trial = self.contact_trials[context, sig, obj.x, obj.y]\n                priority = 20 * trial - 40 * min(1, self.pattern_operators[sig])\n                targets.append((sig, obj.x, obj.y, priority))\n        proposals = []\n        for sig, x, y, priority in targets:\n            goal = ContactGoal(anchor, sig, x, y, offset)\n            path = self._contact_route(scene, controls, obs, goal)\n            if path:\n                onward = abs(x - destination.x) + abs(y - destination.y)\n                proposals.append((len(path) + priority + .05 * onward, repr(goal), goal, path))\n        if not proposals:\n            self.stats[\'contact_no_route\'] += 1\n            return super()._plan(scene, controls, obs)\n        _, _, goal, path = min(proposals)\n        # Reuse a visible numeric precondition for navigation: a boundary\n        # population that repeatedly decreases with actions is a candidate\n        # budget. Same-color landmarks are candidate replenishing operations.\n        rates = []\n        for color, counts in self.boundary_changes.items():\n            delta, support = counts.most_common(1)[0]\n            if delta < 0 and support >= 3:\n                rates.append((support, color, -delta))\n        if rates:\n            _, color, rate = max(rates)\n            remaining = int(boundary_counts(obs.pixels)[color]) / rate\n            resources = [c for c in objects if c.color == color and 2 <= c.size <= 36\n                         and c.width <= 6 and c.height <= 6\n                         and not any(p.contains(*c.center) for p in pair)]\n            terminal_target = (goal.target_x, goal.target_y) == (destination.x, destination.y)\n            step = max([sum(abs(d) for d in self._vectors(k, scene).get(anchor, (0, 0)))\n                        for k in controls] or [1]) or 1\n            reserve = 0 if terminal_target else min(\n                [(abs(c.x - goal.target_x) + abs(c.y - goal.target_y)) / step + 3 for c in resources] or [3])\n            if len(path) + reserve >= remaining:\n                preparation = []\n                for obj in resources:\n                    refill = ContactGoal(anchor, signature(obj), obj.x, obj.y, offset)\n                    route = self._contact_route(scene, controls, obs, refill)\n                    if route and len(route) < remaining:\n                        preparation.append((len(route), repr(refill), refill, route))\n                if preparation:\n                    _, _, goal, path = min(preparation)\n                    self.stats[\'boundary_budget_preparations\'] += 1\n        self.goal = goal\n        self.joint_mode = False\n        self.contact_pending = goal, context\n        self.stats[\'contact_plans\'] += 1\n        return path\n\n    def choose(self, obs, objects, candidates):\n        self._ensure_level(obs.levels)\n        self.contact_objects = objects\n        if self.contacts and any(a.id in (1, 2, 3, 4) for a in candidates):\n            self.repair = None\n        scene = entities(objects, obs.pixels.shape)\n        visible = unique_entities(scene)\n        if self.contacts and self._pair(framed_patterns(obs.pixels, objects), obs.pixels.shape):\n            for key, vectors in self.motion_priors.items():\n                for sig, vector in vectors.items():\n                    if sig in visible and not self.models[key][sig]:\n                        self.models[key][sig][vector] = 1\n                        self.prior_controls.add(key)\n            if self.terrain_priors and not self.passable:\n                self.passable.update(self.terrain_priors[0])\n                self.obstacles.update(self.terrain_priors[1])\n        temporary = [key for key in self.prior_controls if not self.control_trials[key]]\n        for key in temporary:\n            self.control_trials[key] = 1\n        try:\n            return super().choose(obs, objects, candidates)\n        finally:\n            for key in temporary:\n                self.control_trials[key] = 0\n\n    def observe(self, before, action, after, old_objects, new_objects):\n        if action.id in (1, 2, 3, 4) and before.levels == after.levels and before.pixels.shape == after.pixels.shape:\n            counts = boundary_counts(before.pixels)\n            self.boundary_max = np.maximum(self.boundary_max, counts)\n            for color, delta in enumerate(boundary_counts(after.pixels) - counts):\n                if delta:\n                    self.boundary_changes[color][int(delta)] += 1\n        pending, self.contact_pending = self.contact_pending, None\n        old_pair = self._pair(framed_patterns(before.pixels, old_objects), before.pixels.shape)\n        new_pair = self._pair(framed_patterns(after.pixels, new_objects), after.pixels.shape)\n        if old_pair and new_pair and before.levels == after.levels and old_pair[0].pattern != new_pair[0].pattern:\n            scene = entities(new_objects, after.pixels.shape)\n            controls = [(a,) for a in before.available]\n            cohort = self._cohort(entities(old_objects, before.pixels.shape), controls)\n            if cohort:\n                anchor, offset, _ = cohort\n                touched = [ContactGoal(anchor, signature(c), c.x, c.y, offset) for c in old_objects\n                           if 2 <= c.size <= 36 and c.width <= 6 and c.height <= 6\n                           and not any(p.contains(*c.center) for p in old_pair)\n                           and signature(c) not in cohort[2]]\n                for contact in touched:\n                    if contact.distance(scene) == 0:\n                        fitting = fit_transforms(old_pair[0].pattern, new_pair[0].pattern)\n                        previous = self.pattern_transforms.get(contact.target, set(range(8)))\n                        self.pattern_transforms[contact.target] = previous & fitting\n                        self.known_contacts[contact.target, contact.target_x, contact.target_y] = (contact, previous & fitting)\n                        self.pattern_operators[contact.target] += 1\n                        self.stats[\'pattern_changes_at_contacts\'] += 1\n        if pending and before.levels == after.levels and after.state not in (\'GAME_OVER\', \'WIN\'):\n            goal, context = pending\n            scene = entities(new_objects, after.pixels.shape)\n            if goal.distance(scene) == 0:\n                self.contact_trials[context, goal.target, goal.target_x, goal.target_y] += 1\n                self.stats[\'contacts_reached\'] += 1\n        super().observe(before, action, after, old_objects, new_objects)\n        if self.navigation_cohort:\n            self.motion_priors = {key: {sig: vector for sig, vector in self._vectors(key, entities(new_objects, after.pixels.shape)).items()\n                                       if sig in self.navigation_cohort}\n                                  for key in self.models if key[0] in (1, 2, 3, 4)}\n            self.terrain_priors = dict(self.passable), dict(self.obstacles)\n\n    def summary(self):\n        result = super().summary()\n        result[\'pattern_contacts_enabled\'] = self.contacts\n        result[\'pattern_operator_evidence\'] = {repr(k): v for k, v in self.pattern_operators.items()}\n        result[\'pattern_transform_candidates\'] = {repr(k): sorted(v) for k, v in self.pattern_transforms.items()}\n        return result\n', 'joint_dynamics.py': '"""Experimental composition of measured increments against joint constraints.\n\nThis is an additive hypothesis on visible object positions and color counts,\nnot a proof of dynamics or a game-specific resource model. Actual observations\ncheck predictions after every action. Old single-goal behavior stays available.\n"""\nfrom __future__ import annotations\n\nfrom collections import Counter, defaultdict\nfrom dataclasses import dataclass\nimport heapq\nimport itertools\n\nimport numpy as np\n\nfrom shared_dynamics import SharedDynamics, GoalHypothesis, entities, unique_entities\n\n\n@dataclass(frozen=True)\nclass JointGoal:\n    members: tuple\n\n    def distance(self, scene):\n        return sum(g.distance(scene) for g in self.members)\n\n    def template(self):\n        return (\'joint\', tuple(g.template() for g in self.members))\n\n\ndef projection(pixels, axis):\n    """Color population along rows/columns; a lossy measurable feature vector."""\n    interior = pixels[2:-2, 2:-2] if min(pixels.shape) > 4 else pixels\n    return np.stack([(interior == color).sum(axis=1 - axis)\n                     for color in range(16)]).astype(np.int16).ravel()\n\n\ndef prepare_increment(state, key, increments, bounds, target, active=(), depth=0):\n    """Apply the same operator search to an unmet numeric precondition.\n\n    A consumed coordinate must first be supplied. Its suppliers can have their\n    own unmet preconditions. Every intermediate state still respects bounds.\n    This is a bounded greedy proposal, never a reachability certificate.\n    """\n    if key in active or depth > 10:\n        return None\n    delta = increments[key]\n    path = []\n    current = state\n    for _ in range(20):\n        after = tuple(x + dx for x, dx in zip(current, delta))\n        violations = [(i, lo - x if x < lo else hi - x)\n                      for i, (x, (lo, hi)) in enumerate(zip(after, bounds)) if not lo <= x <= hi]\n        if not violations:\n            return path + [key], after\n        i, deficit = max(violations, key=lambda p: abs(p[1]))\n        candidates = []\n        for helper, change in increments.items():\n            if change[i] * deficit <= 0 or helper in (*active, key):\n                continue\n            proposal = prepare_increment(current, helper, increments, bounds, target,\n                                         (*active, key), depth + 1)\n            if proposal is None:\n                continue\n            prefix, supplied = proposal\n            if (supplied[i] - current[i]) * deficit <= 0:\n                continue\n            goal_distance = sum(abs(supplied[j] - x) for j, x in enumerate(target))\n            candidates.append((goal_distance + .1 * len(prefix), repr(helper), prefix, supplied))\n        if not candidates:\n            return None\n        _, _, prefix, current = min(candidates)\n        path.extend(prefix)\n        if len(path) > 80:\n            return None\n    return None\n\n\ndef recursive_composition(start, target, increments, bounds, forbidden_first=(), max_depth=80):\n    """Goal and preparation reuse the same measured action catalogue."""\n    state, path, seen = start, [], {start}\n    distance = lambda s: sum(abs(s[i] - x) for i, x in enumerate(target))\n    catalogue = dict(increments)\n    for _ in range(max_depth):\n        if distance(state) == 0:\n            return path, True\n        proposals = []\n        for key, delta in increments:\n            # Try operations whose direct effect helps at least one unmet goal.\n            if not any((target[i] - state[i]) * delta[i] > 0 for i in range(len(target))):\n                continue\n            proposal = prepare_increment(state, key, catalogue, bounds, target)\n            if proposal is None:\n                continue\n            prefix, after = proposal\n            if (not path and prefix[0] in forbidden_first) or after in seen or len(path) + len(prefix) > max_depth:\n                continue\n            gain = distance(state) - distance(after)\n            proposals.append((-gain / len(prefix), distance(after), len(prefix), repr(key), prefix, after))\n        if not proposals:\n            return path, False\n        _, _, _, _, prefix, state = min(proposals)\n        seen.add(state)\n        path.extend(prefix)\n    return path, distance(state) == 0\n\n\ndef compose_increments(start, target, increments, bounds, forbidden_first=(),\n                       max_nodes=3000, max_depth=80, use_recursive=True):\n    """Bounded search. Each item carries a measured whole-vector increment.\n\n    Only the leading target coordinates are goals. Remaining coordinates are\n    observed context; preserving their numeric bounds constrains composition.\n    The model may still be wrong. Replanning from real feedback is required.\n    """\n    recursive, complete = (recursive_composition(start, target, increments, bounds, forbidden_first, max_depth)\n                           if use_recursive else ([], False))\n    if complete:\n        return recursive, 0, True\n    n = len(target)\n    distance = lambda s: sum(abs(s[i] - target[i]) for i in range(n))\n    unit = max([sum(abs(x) for x in v[:n]) for _, v in increments] or [1]) or 1\n    start_distance = distance(start)\n    serial = itertools.count()\n    queue = [(1.5 * start_distance / unit, next(serial), 0, start, ())]\n    costs = {start: 0}\n    best = (start_distance, 0, ())\n    if recursive:\n        candidate = start\n        catalogue = dict(increments)\n        for key in recursive:\n            candidate = tuple(x + dx for x, dx in zip(candidate, catalogue[key]))\n        if distance(candidate) < start_distance:\n            best = (distance(candidate), len(recursive), tuple(recursive))\n    expanded = 0\n    while queue and expanded < max_nodes:\n        _, _, cost, state, path = heapq.heappop(queue)\n        if cost != costs.get(state):\n            continue\n        expanded += 1\n        d = distance(state)\n        if d == 0:\n            return list(path), expanded, True\n        if (d, cost) < best[:2]:\n            best = (d, cost, path)\n        if cost >= max_depth:\n            continue\n        for key, delta in increments:\n            if not path and key in forbidden_first:\n                continue\n            after = tuple(x + dx for x, dx in zip(state, delta))\n            if after == state or any(x < lo or x > hi for x, (lo, hi) in zip(after, bounds)):\n                continue\n            next_cost = cost + 1\n            if next_cost >= costs.get(after, float(\'inf\')):\n                continue\n            costs[after] = next_cost\n            value = next_cost + 1.5 * distance(after) / unit\n            heapq.heappush(queue, (value, next(serial), next_cost, after, path + (key,)))\n    return list(best[2]), expanded, False\n\n\nclass JointDynamics(SharedDynamics):\n    def __init__(self, seed=0, recursive=True, minimum_goals=2):\n        self.recursive = recursive\n        self.minimum_goals = minimum_goals\n        super().__init__(seed)\n\n    def _clear_local(self):\n        super()._clear_local()\n        self.effects = defaultdict(Counter)\n        self.joint_mode = False\n        self.projected_pending = None\n\n    def _joint_goal(self, scene, controls, obs):\n        axes = defaultdict(set)\n        for key in controls:\n            for sig, vector in self._vectors(key, scene).items():\n                axes[sig].update(i for i, delta in enumerate(vector) if delta)\n        goals = []\n        h, w = obs.pixels.shape\n        for mover, motion_axes in sorted(axes.items()):\n            if len(motion_axes) != 1:\n                continue\n            candidates = [GoalHypothesis(mover, target, x, y, tuple(motion_axes))\n                          for target, x, y in scene\n                          if target not in axes and target[0] == mover[0]\n                          and target[1] <= 12 and target[2] <= 12\n                          and 4 <= x <= w - 4 - target[1]\n                          and 4 <= y <= h - 4 - target[2]]\n            if len(candidates) == 1 and np.isfinite(candidates[0].distance(scene)):\n                goals.append(candidates[0])\n        # A single repeated relation grammar, including already satisfied members.\n        if len(goals) >= self.minimum_goals and len({g.axes for g in goals}) == 1:\n            return JointGoal(tuple(goals))\n        return None\n\n    def choose(self, obs, objects, candidates):\n        if self.joint_mode:\n            self.repair = None  # joint planning handles preparatory effects too\n        action, reason = super().choose(obs, objects, candidates)\n        if self.joint_mode and self.pending:\n            key = self.pending[\'key\']\n            axis = self.goal.members[0].axes[0] if isinstance(self.goal, JointGoal) else None\n            if axis is not None:\n                effects = self.effects.get(key)\n                if effects:\n                    effect = effects.most_common(1)[0][0]\n                    delta = np.frombuffer(effect[1 + axis], dtype=np.int16)\n                    self.projected_pending = (axis, projection(obs.pixels, axis) + delta)\n        return action, reason\n\n    def _plan(self, scene, controls, obs):\n        goal = self._joint_goal(scene, controls, obs)\n        if goal is None or goal in self.rejected_goals:\n            self.joint_mode = False\n            if isinstance(self.goal, JointGoal):\n                self.goal = None\n            return super()._plan(scene, controls, obs)\n        self.joint_mode = True\n        if self.goal != goal:\n            self.goal = goal\n            self.stats[\'joint_goal_hypotheses\'] += 1\n        axis = goal.members[0].axes[0]\n        current = unique_entities(scene)\n        start_positions = tuple(2 * current[g.mover][axis] + g.mover[axis + 1] for g in goal.members)\n        target = tuple(2 * (g.target_x if axis == 0 else g.target_y) + g.target[axis + 1] for g in goal.members)\n        counts = projection(obs.pixels, axis)\n        patterns = []\n        stable = np.ones(len(counts), dtype=bool)\n        for key in controls:\n            if not self.effects.get(key):\n                continue\n            effect = self.effects[key].most_common(1)[0][0]\n            motion = dict(effect[0])\n            delta = np.frombuffer(effect[1 + axis], dtype=np.int16)\n            samples = [np.frombuffer(e[1 + axis], dtype=np.int16)\n                       for e in self.effects[key] if e[0] == effect[0]]\n            stable &= np.all(np.stack(samples) == delta, axis=0)\n            pos_delta = tuple(2 * motion.get(g.mover, (0, 0))[axis] for g in goal.members)\n            if any(pos_delta) or np.any(delta):\n                patterns.append((key, pos_delta, delta))\n        if not patterns:\n            return []\n        matrix = np.stack([p[2] for p in patterns])\n        # Keep changing population features, deduplicating identical columns.\n        active = np.flatnonzero(np.any(matrix, axis=0) & stable)\n        representatives = {}\n        for i in active:\n            column = tuple(map(int, matrix[:, i]))\n            # Equal measured changes can be offset views of one quantity.\n            # Keep the most exposed count as an optimistic planning feature;\n            # full population predictions are still checked against the image.\n            if column not in representatives or counts[i] > counts[representatives[column]]:\n                representatives[column] = int(i)\n        indices = list(representatives.values())\n        start = start_positions + tuple(map(int, counts[indices]))\n        increments = [(key, pd + tuple(map(int, delta[indices]))) for key, pd, delta in patterns]\n        size = obs.pixels.shape[1 - axis]\n        interior_size = size - 4 if min(obs.pixels.shape) > 4 else size\n        bounds = [(0, 2 * size)] * len(target) + [(0, interior_size)] * len(indices)\n        forbidden = {key for key in controls if (scene, key) in self.blocked}\n        path, nodes, complete = compose_increments(start, target, increments, bounds, forbidden,\n                                                   use_recursive=self.recursive)\n        self.stats[\'joint_plans\'] += 1\n        self.stats[\'joint_search_nodes\'] += nodes\n        self.stats[\'joint_complete_model_plans\'] += int(complete)\n        self.stats[\'recursive_complete_model_plans\'] += int(complete and nodes == 0 and self.recursive)\n        if not path:\n            self.stats[\'joint_no_model_plan\'] += 1\n            # Unknown actions can reveal missing effects. No claimed impossibility.\n        return path\n\n    def observe(self, before, action, after, old_objects, new_objects):\n        self._ensure_level(before.levels)\n        if before.pixels.shape != after.pixels.shape:\n            # Population coordinates have changed; old increments no longer\n            # have the same domain. Keep the full transition in the ledger.\n            super().observe(before, action, after, old_objects, new_objects)\n            self._clear_local()\n            self.stats[\'population_domain_changes\'] += 1\n            return\n        pending, self.projected_pending = self.projected_pending, None\n        if action.id and before.levels == after.levels and after.state not in (\'GAME_OVER\', \'WIN\'):\n            old = unique_entities(entities(old_objects, before.pixels.shape))\n            new = unique_entities(entities(new_objects, after.pixels.shape))\n            motion = tuple(sorted((sig, (new[sig][0] - x, new[sig][1] - y))\n                                  for sig, (x, y) in old.items()\n                                  if sig in new and sig[1] <= 24 and sig[2] <= 24\n                                  and sig[1] * sig[2] <= 256\n                                  and 0 < abs(new[sig][0] - x) + abs(new[sig][1] - y) <= 16))\n            deltas = tuple(projection(after.pixels, a) - projection(before.pixels, a) for a in (0, 1))\n            if motion or any(np.any(d) for d in deltas):\n                key = self.control_key(action, old_objects)\n                self.effects[key][(motion, *(d.tobytes() for d in deltas))] += 1\n                self.stats[\'joint_effect_observations\'] += 1\n            if pending:\n                axis, predicted = pending\n                self.stats[\'population_predictions_checked\'] += 1\n                if not np.array_equal(predicted, projection(after.pixels, axis)):\n                    self.stats[\'population_prediction_failures\'] += 1\n        super().observe(before, action, after, old_objects, new_objects)\n\n    def summary(self):\n        result = super().summary()\n        result[\'recursive_preparation_enabled\'] = self.recursive\n        if self.minimum_goals != 2:\n            result[\'minimum_composed_goals\'] = self.minimum_goals\n        result[\'joint_scope\'] = \'Bounded additive composition of observed whole-vector effects and simultaneous alignment hypotheses; measured predictions are not universal rules\'\n        return result\n', 'partial_relations.py': '"""Partial glyph layouts and measured multi-cell actions.\n\nAn unknown input symbol is treated as an absent target only if every visible\noccurrence maps to a non-tile. This is a geometry-conditioned hypothesis, not\npermission to ignore arbitrary unknown symbols. Action effects are measured\non public observations before composing them as a binary linear model.\n"""\nfrom __future__ import annotations\n\nfrom collections import Counter, defaultdict\nfrom dataclasses import dataclass\n\nimport numpy as np\n\n\n@dataclass(frozen=True)\nclass GridTile:\n    x: int\n    y: int\n    width: int\n    color: int\n    decoration: tuple\n\n    @property\n    def center(self):\n        return self.x + (self.width - 1) // 2, self.y + (self.width - 1) // 2\n\n\ndef sampled_grid(patch, n):\n    """Require the entire patch to match an n by n block-constant symbol."""\n    if patch.ndim != 2 or min(patch.shape) < n:\n        return None\n    h, w = patch.shape\n    result = np.zeros((n, n), dtype=np.uint8)\n    for j in range(n):\n        for i in range(n):\n            block = patch[j * h // n:(j + 1) * h // n, i * w // n:(i + 1) * w // n]\n            if not np.all(block == block[0, 0]):\n                return None\n            result[j, i] = block[0, 0]\n    return result\n\n\ndef partial_scenes(skill, objects, pixels):\n    roles = dict(skill.symbol_roles)\n    groups = defaultdict(list)\n    for c in objects:\n        # A marked tile may have no monochrome square component. A dominant\n        # component can still propose its bounding box; the full patch is\n        # classified below before it becomes a tile or a glyph.\n        if c.width == c.height and 3 <= c.width <= 12 and c.size > c.width * c.height // 2:\n            groups[c.width].append(c)\n    scenes = []\n    h, w = pixels.shape\n    background = int(np.bincount(pixels.ravel()).argmax())\n    for width, solids in groups.items():\n        if len(solids) < 5:\n            continue\n        xsteps = Counter(b.x - a.x for a in solids for b in solids\n                         if a.y == b.y and width < b.x - a.x <= 3 * width)\n        ysteps = Counter(b.y - a.y for a in solids for b in solids\n                         if a.x == b.x and width < b.y - a.y <= 3 * width)\n        for dx, _ in xsteps.most_common(3):\n            for dy, _ in ysteps.most_common(3):\n                phases = Counter((c.x % dx, c.y % dy) for c in solids)\n                for (ox, oy), count in phases.items():\n                    if count < 5:\n                        continue\n                    candidates = [c for c in solids if (c.x % dx, c.y % dy) == (ox, oy)]\n                    tiles = {(c.x, c.y): GridTile(c.x, c.y, width, c.color, ()) for c in candidates\n                             if c.size == width * width}\n                    glyphs, patches = [], {}\n                    for y in range(oy, h - width + 1, dy):\n                        for x in range(ox, w - width + 1, dx):\n                            if (x, y) in tiles:\n                                continue\n                            glyph = sampled_grid(pixels[y:y + width, x:x + width], skill.grid_size)\n                            if glyph is None:\n                                continue\n                            patches[x, y] = glyph\n                            symbols = skill.symbols(glyph)\n                            marker = int(glyph[skill.hole])\n                            if marker not in roles and sum(s in roles for s in symbols) >= 3:\n                                glyphs.append((x, y, glyph, marker))\n                    if len(glyphs) < 2:\n                        continue\n                    glyph_at = {(x, y) for x, y, _, _ in glyphs}\n                    palette = {c.color for c in candidates if (c.x, c.y) not in glyph_at}\n                    palette.update(m for _, _, _, m in glyphs)\n                    for (x, y), grid in patches.items():\n                        if (x, y) in glyph_at:\n                            continue\n                        value = int(grid[skill.hole])\n                        colors = set(map(int, np.unique(grid)))\n                        if (value in palette and len(colors) == 2 and len(colors - palette) == 1\n                                and int(np.sum(grid == value)) > grid.size // 2):\n                            decoration = tuple(-1 if int(v) == value else int(v) for v in grid.flat)\n                            tiles[x, y] = GridTile(x, y, width, value, decoration)\n                    constraints, uses, absent = {}, Counter(), Counter()\n                    valid = True\n                    for x, y, glyph, marker in glyphs:\n                        for (j, i), symbol in zip(skill.occupied, skill.symbols(glyph)):\n                            target = x + (i - skill.hole[1]) * dx, y + (j - skill.hole[0]) * dy\n                            if symbol not in roles:\n                                # An unfamiliar symbol targeting an actual tile remains\n                                # unknown. Never weaken its constraint to a wildcard.\n                                if target in tiles:\n                                    valid = False\n                                elif target not in glyph_at:\n                                    tx, ty = target\n                                    if 0 <= tx <= w - width and 0 <= ty <= h - width:\n                                        patch = pixels[ty:ty + width, tx:tx + width]\n                                        if not np.all(patch == background):\n                                            # An unclassified visible patch is unknown,\n                                            # not evidence that the target is absent.\n                                            valid = False\n                                absent[symbol] += 1\n                            elif target not in tiles:\n                                valid = False\n                            else:\n                                eq, neq = constraints.setdefault(target, (set(), set()))\n                                (eq if roles[symbol] == "marker" else neq).add(marker)\n                                uses[target] += 1\n                    if not valid or any(n < 2 for n in absent.values()):\n                        continue\n                    decorated = sum(bool(t.decoration) for t in tiles.values())\n                    if not absent and not decorated:\n                        continue  # Leave the earlier fully observed policy unchanged.\n                    allowed = {}\n                    for point, (eq, neq) in constraints.items():\n                        if len(eq) > 1 or eq & neq:\n                            valid = False\n                            break\n                        choices = (eq or palette) - neq\n                        if not choices:\n                            valid = False\n                            break\n                        allowed[point] = frozenset(choices)\n                    if not valid:\n                        continue\n                    scenes.append({"tiles": tiles, "allowed": allowed, "palette": tuple(sorted(palette)),\n                                   "glyph_patches": {(x, y): pixels[y:y + width, x:x + width].copy()\n                                                     for x, y, _, _ in glyphs},\n                                   "geometry": (width, dx, dy, ox, oy), "skill": skill.key(),\n                                   "evidence": {"glyphs": len(glyphs), "tiles": len(tiles),\n                                                "constrained_tiles": len(allowed), "decorated_tiles": decorated,\n                                                "shared_tiles": sum(n > 1 for n in uses.values()),\n                                                "absent_symbol_occurrences": dict(absent)}})\n    scenes.sort(key=lambda s: (s["evidence"]["glyphs"], s["evidence"]["constrained_tiles"],\n                               -s["geometry"][1] * s["geometry"][2]), reverse=True)\n    return scenes\n\n\ndef solve_binary(columns, target, mask):\n    """Solve observed action effects over GF(2); return None if not in their span.\n\n    Each bit is a tile. Unconstrained tiles are removed by mask. Free action\n    choices are zero, so a returned solution is not necessarily the shortest.\n    """\n    basis = {}\n    for index, column in enumerate(columns):\n        value, witness = column & mask, 1 << index\n        while value:\n            pivot = value.bit_length() - 1\n            if pivot not in basis:\n                basis[pivot] = value, witness\n                break\n            old, used = basis[pivot]\n            value ^= old\n            witness ^= used\n    value, answer = target & mask, 0\n    while value:\n        pivot = value.bit_length() - 1\n        if pivot not in basis:\n            return None\n        old, used = basis[pivot]\n        value ^= old\n        answer ^= used\n    return [i for i in range(len(columns)) if answer >> i & 1]\n\n\nclass PartialRelations:\n    def __init__(self):\n        self.level = None\n        self.scene = None\n        self.effects = {}\n        self.disabled = set()\n        self.pending = None\n        self.stopped = False\n        self.events = []\n        self.actions = 0\n        self.probes = 0\n        self.prediction_failures = 0\n        self.completions = []\n\n    def reset(self, level):\n        self.level, self.scene = level, None\n        self.effects, self.disabled = {}, set()\n        self.pending, self.stopped = None, False\n\n    def read_values(self, obs):\n        for (x, y), original in self.scene["glyph_patches"].items():\n            h, w = original.shape\n            if not np.array_equal(obs.pixels[y:y + h, x:x + w], original):\n                return None\n        values = []\n        for point in self.scene["order"]:\n            tile = self.scene["tiles"][point]\n            patch = obs.pixels[tile.y:tile.y + tile.width, tile.x:tile.x + tile.width]\n            grid = sampled_grid(patch, self.scene["grid_size"])\n            if grid is None:\n                return None\n            value = int(grid[self.scene["hole"]])\n            if value not in self.scene["palette"]:\n                return None\n            decoration = tuple(-1 if int(v) == value else int(v) for v in grid.flat)\n            if tile.decoration:\n                if decoration != tile.decoration:\n                    return None\n            elif not np.all(grid == value):\n                return None\n            values.append(value)\n        return values\n\n    def propose(self, obs, objects, skills):\n        if self.level != obs.levels:\n            self.reset(obs.levels)\n        if self.stopped or 6 not in obs.available or not skills:\n            return None\n        if self.scene is None:\n            for skill in skills:\n                scenes = partial_scenes(skill, objects, obs.pixels)\n                scene = next((s for s in scenes if len(s["palette"]) == 2), None)\n                if scene is not None:\n                    scene.update({"order": sorted(scene["tiles"], key=lambda p: (p[1], p[0])),\n                                  "grid_size": skill.grid_size, "hole": skill.hole})\n                    self.scene = scene\n                    self.events.append({"status": "partial_layout_hypothesis", "level": obs.levels,\n                                        "skill": scene["skill"], **scene["evidence"]})\n                    break\n        if self.scene is None:\n            return None\n        values = self.read_values(obs)\n        if values is None:\n            self.stopped = True\n            self.events.append({"level": obs.levels, "status": "tile_representation_changed"})\n            return None\n        target, mask = 0, 0\n        for i, point in enumerate(self.scene["order"]):\n            allowed = self.scene["allowed"].get(point)\n            if allowed is not None and len(allowed) == 1:\n                mask |= 1 << i\n                if values[i] not in allowed:\n                    target |= 1 << i\n        if not target:\n            self.stopped = True\n            self.events.append({"level": obs.levels, "status": "partial_targets_met_without_completion"})\n            return None\n        controls = sorted(self.effects)\n        solution = solve_binary([self.effects[p] for p in controls], target, mask)\n        probe = solution is None\n        if probe:\n            unknown = [p for p in self.scene["order"] if p not in self.effects and p not in self.disabled]\n            if not unknown:\n                self.stopped = True\n                self.events.append({"level": obs.levels, "status": "target_outside_observed_action_span"})\n                return None\n            point = min(unknown, key=lambda p: (not bool(self.scene["tiles"][p].decoration), p[1], p[0]))\n        else:\n            point = controls[solution[0]]\n        self.pending = {"point": point, "values": values, "probe": probe,\n                        "predicted": self.effects.get(point)}\n        return self.scene["tiles"][point].center, ("measure multi-cell control" if probe else "compose measured binary action effects")\n\n    def observe(self, before, action, after):\n        if action.id == 0:\n            self.reset(after.levels)\n            return\n        pending, self.pending = self.pending, None\n        if pending is None:\n            return\n        self.actions += 1\n        self.probes += pending["probe"]\n        if after.levels > before.levels:\n            self.completions.append({"level_completed": after.levels, "actions": self.actions,\n                                     "measured_controls": len(self.effects), "evidence": self.scene["evidence"]})\n            self.reset(after.levels)\n            return\n        values = self.read_values(after)\n        if values is None:\n            self.stopped = True\n            self.prediction_failures += 1\n            self.events.append({"level": before.levels, "status": "unexpected_tile_change"})\n            return\n        effect = sum(1 << i for i, (a, b) in enumerate(zip(pending["values"], values)) if a != b)\n        predicted = pending["predicted"]\n        if predicted is not None and effect != predicted:\n            self.prediction_failures += 1\n            self.disabled.add(pending["point"])\n            self.effects.pop(pending["point"], None)\n            self.events.append({"level": before.levels, "status": "action_effect_counterexample",\n                                "point": list(pending["point"]), "predicted": predicted, "observed": effect})\n        else:\n            self.effects[pending["point"]] = effect\n            if pending["probe"]:\n                self.events.append({"level": before.levels, "status": "measured_action_effect",\n                                    "point": list(pending["point"]), "changed_tiles": effect.bit_count()})\n\n    def summary(self):\n        return {"actions": self.actions, "probes": self.probes,\n                "prediction_failures": self.prediction_failures,\n                "completed_levels": self.completions, "events": self.events}\n', 'transfer.py': '"""Conditional reuse of a successful visible glyph correspondence.\n\nTemplates are hypotheses with observed support. They are neither universal\ntheorems nor inferred access to the environment\'s hidden state. Input symbol\ncolors are retained; output colors are represented by their role in a glyph.\n"""\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nfrom collections import Counter, defaultdict\nfrom dataclasses import asdict, dataclass\n\nimport numpy as np\n\nfrom relations import Panel, repeated_panels\n\n\ndef panel_geometry(panel):\n    xs, ys = sorted({c.x for c in panel.cells}), sorted({c.y for c in panel.cells})\n    if len(xs) != len(ys) or not 2 <= len(xs) <= 5:\n        return None\n    occupied = tuple((ys.index(c.y), xs.index(c.x)) for c in panel.cells)\n    holes = [(j, i) for j in range(len(ys)) for i in range(len(xs)) if (j, i) not in occupied]\n    if len(holes) != 1:\n        return None\n    if len(set(np.diff(xs))) != 1 or len(set(np.diff(ys))) != 1:\n        return None\n    return len(xs), occupied, holes[0], xs, ys\n\n\ndef read_glyph(panel, pixels):\n    geometry = panel_geometry(panel)\n    if geometry is None:\n        return None\n    n, occupied, hole, xs, ys = geometry\n    width, height = panel.cells[0].width, panel.cells[0].height\n    x, y = xs[hole[1]], ys[hole[0]]\n    glyph = np.asarray([[int(pixels[y + min(height - 1, int((j + .5) * height / n)),\n                                     x + min(width - 1, int((i + .5) * width / n))])\n                         for i in range(n)] for j in range(n)], dtype=np.uint8)\n    return glyph, int(glyph[hole]), geometry\n\n\n@dataclass(frozen=True)\nclass GlyphSkill:\n    grid_size: int\n    occupied: tuple[tuple[int, int], ...]\n    hole: tuple[int, int]\n    reflect: bool\n    rotations: int\n    symbol_roles: tuple[tuple[int, str], ...]\n    reference_cells: int\n    learned_after_level: int\n\n    def key(self):\n        return hashlib.sha256(json.dumps(asdict(self), sort_keys=True).encode()).hexdigest()[:16]\n\n    def symbols(self, glyph):\n        transformed = np.rot90(np.fliplr(glyph) if self.reflect else glyph, self.rotations)\n        return tuple(int(transformed[y, x]) for y, x in self.occupied)\n\n\ndef fit_role_skills(panels, control, pixels, expected_target, completed_level):\n    """Fit color roles to references; save only after an observed completion."""\n    references = [p for p in panels if p.origin != control.origin]\n    if len(references) < 2:\n        return []\n    data = [read_glyph(p, pixels) for p in panels]\n    if any(d is None for d in data):\n        return []\n    n, occupied, hole, _, _ = data[0][2]\n    if any(d[2][:3] != (n, occupied, hole) for d in data):\n        return []\n    fitted = []\n    for reflect in (False, True):\n        for rotations in range(4):\n            roles = {}\n            valid, support = True, 0\n            for panel, (glyph, marker, _) in zip(panels, data):\n                if panel.origin == control.origin:\n                    continue\n                palette = set(panel.colors)\n                if len(palette) != 2 or marker not in palette:\n                    valid = False\n                    break\n                transformed = np.rot90(np.fliplr(glyph) if reflect else glyph, rotations)\n                for (y, x), output in zip(occupied, panel.colors):\n                    symbol = int(transformed[y, x])\n                    role = "marker" if output == marker else "other"\n                    if symbol in roles and roles[symbol] != role:\n                        valid = False\n                        break\n                    roles[symbol] = role\n                    support += 1\n            if not valid or set(roles.values()) != {"marker", "other"}:\n                continue\n            skill = GlyphSkill(n, occupied, hole, reflect, rotations,\n                               tuple(sorted(roles.items())), support, completed_level)\n            glyph, marker, _ = data[panels.index(control)]\n            other = {c for p in panels for c in p.colors} - {marker}\n            if len(other) != 1:\n                continue\n            symbols = skill.symbols(glyph)\n            if not all(s in roles for s in symbols):\n                continue\n            alternative = next(iter(other))\n            target = tuple(marker if roles[s] == "marker" else alternative for s in symbols)\n            if target == tuple(expected_target):\n                fitted.append(skill)\n    return fitted\n\n\ndef locate_panels(skill, objects, pixels):\n    """Find the learned normalized layout, permitting shared tiles and scale."""\n    groups = defaultdict(list)\n    for c in objects:\n        if c.width == c.height and 3 <= c.width <= 12 and c.size == c.width * c.height:\n            groups[c.width].append(c)\n    result, seen = [], set()\n    first_y, first_x = skill.occupied[0]\n    known_symbols = dict(skill.symbol_roles)\n    for width, tiles in groups.items():\n        at = {(c.x, c.y): c for c in tiles}\n        xsteps = {b.x - a.x for a in tiles for b in tiles\n                  if a.y == b.y and width < b.x - a.x <= 3 * width}\n        ysteps = {b.y - a.y for a in tiles for b in tiles\n                  if a.x == b.x and width < b.y - a.y <= 3 * width}\n        for tile in tiles:\n            for dx in sorted(xsteps):\n                for dy in sorted(ysteps):\n                    x0, y0 = tile.x - first_x * dx, tile.y - first_y * dy\n                    hole = (x0 + skill.hole[1] * dx, y0 + skill.hole[0] * dy)\n                    if hole in at:\n                        continue\n                    coordinates = tuple((x0 + x * dx, y0 + y * dy) for y, x in skill.occupied)\n                    if coordinates in seen or not all(p in at for p in coordinates):\n                        continue\n                    panel = Panel(tuple(at[p] for p in coordinates))\n                    parsed = read_glyph(panel, pixels)\n                    if parsed is None:\n                        continue\n                    glyph, marker, geometry = parsed\n                    if geometry[:3] != (skill.grid_size, skill.occupied, skill.hole):\n                        continue\n                    symbols = skill.symbols(glyph)\n                    if not all(s in known_symbols for s in symbols):\n                        continue\n                    seen.add(coordinates)\n                    result.append((panel, glyph, marker))\n    return result\n\n\ndef compose_targets(skill, located, palette_history=frozenset()):\n    """Compose equal/not-equal constraints, preserving an incomplete palette.\n\n    With two output colors \'other\' has one value. With more colors, it is a\n    set-valued inequality hypothesis. A missing color is not a contradiction.\n    """\n    roles = dict(skill.symbol_roles)\n    constraints, counts = {}, Counter()\n    palette = set(palette_history)\n    for panel, glyph, marker in located:\n        palette.update(set(panel.colors) | {marker})\n        for cell, symbol in zip(panel.cells, skill.symbols(glyph)):\n            if symbol not in roles:\n                return None, {"status": "unknown_symbol", "symbol": symbol}\n            key = (cell.x, cell.y, cell.width, cell.height)\n            record = constraints.setdefault(key, {"cell": cell, "equal": set(), "excluded": set()})\n            record["equal" if roles[symbol] == "marker" else "excluded"].add(marker)\n            counts[key] += 1\n    assignments = {}\n    for key, record in constraints.items():\n        equal, excluded = record["equal"], record["excluded"]\n        if len(equal) > 1 or equal & excluded:\n            return None, {"status": "inconsistent_shared_constraints", "cell": list(key),\n                          "required": sorted(equal), "excluded": sorted(excluded)}\n        allowed = (equal or palette) - excluded\n        if not allowed:\n            return None, {"status": "palette_incomplete", "probe_cell": list(key),\n                          "observed_colors": sorted(palette), "excluded": sorted(excluded)}\n        assignments[key] = (record["cell"], frozenset(allowed))\n    return assignments, {"status": "consistent_on_visible_cells", "panels": len(located),\n                         "unique_cells": len(assignments),\n                         "shared_cells": sum(n > 1 for n in counts.values())}\n\n\nclass TransferMemory:\n    def __init__(self, retain=True):\n        self.retain = retain\n        self.skills = []\n        self.level = None\n        self.pending = None\n        self.blocked = set()\n        self.palette = set()\n        self.events = []\n        self.transfer_actions = 0\n        self.prediction_failures = 0\n        self.learned_count = 0\n        self.confirmed_levels = []\n        self.last_plan = None\n        self.color_edges = defaultdict(set)\n        self.no_effect = Counter()\n        self.palette_probes = 0\n        self.level_probes = 0\n\n    def change_level(self, level):\n        self.level = level\n        self.pending = None\n        self.blocked = set()\n        self.palette = set()\n        self.last_plan = None\n        self.color_edges.clear()\n        self.no_effect.clear()\n        self.level_probes = 0\n        if not self.retain:\n            self.skills = []\n\n    def propose(self, obs, objects):\n        if self.level != obs.levels:\n            self.change_level(obs.levels)\n        if 6 not in obs.available or not self.skills:\n            return None\n        for skill in self.skills:\n            key = skill.key()\n            if key in self.blocked:\n                continue\n            located = locate_panels(skill, objects, obs.pixels)\n            if not located:\n                continue\n            observed_colors = {c.color for p, _, _ in located for c in p.cells}\n            markers = {marker for _, _, marker in located}\n            # Accumulated palette is a local hypothesis and is reset per level.\n            self.palette.update(observed_colors | markers)\n            assignments, evidence = compose_targets(skill, located, self.palette)\n            self.last_plan = evidence\n            if assignments is None:\n                event = {"level": obs.levels, "skill": key, **evidence}\n                if event not in self.events:\n                    self.events.append(event)\n                if evidence["status"] == "palette_incomplete" and self.level_probes < 6:\n                    x, y, w, h = evidence["probe_cell"]\n                    cell = next((c for c in objects if (c.x, c.y, c.width, c.height) == (x, y, w, h)), None)\n                    if cell is not None:\n                        self.level_probes += 1\n                        self.palette_probes += 1\n                        self.pending = {"skill": key, "cell": (x, y, w, h),\n                                        "before_color": cell.color, "expected_next_color": None,\n                                        "allowed_colors": [], "evidence": evidence, "probe": True}\n                        return cell.center, f"probe missing output color for glyph skill {key}"\n                continue\n            mismatches = [(cell, allowed) for cell, allowed in assignments.values() if cell.color not in allowed]\n            if not mismatches:\n                self.blocked.add(key)\n                self.events.append({"level": obs.levels, "skill": key,\n                                    "status": "target_reached_without_completion"})\n                continue\n            cell, allowed = min(mismatches, key=lambda item: (item[0].y, item[0].x))\n            successors = self.color_edges[cell.color]\n            predicted = next(iter(successors)) if len(successors) == 1 else None\n            self.pending = {"skill": key, "cell": (cell.x, cell.y, cell.width, cell.height),\n                            "before_color": cell.color, "expected_next_color": predicted,\n                            "allowed_colors": sorted(allowed), "evidence": evidence, "probe": False}\n            return cell.center, f"reuse glyph role correspondence {key}; shared cells checked"\n        return None\n\n    def observe(self, before, action, after, old_objects, new_objects, relations):\n        if self.level != before.levels:\n            self.change_level(before.levels)\n        pending = self.pending\n        if pending is not None:\n            self.transfer_actions += 1\n            if after.levels > before.levels:\n                event = {"level_completed": after.levels, "skill": pending["skill"],\n                         "status": "completion_after_transfer", "composition": pending["evidence"]}\n                self.confirmed_levels.append(event)\n            elif action.id != 0:\n                x, y, w, h = pending["cell"]\n                patch = after.pixels[y:y + h, x:x + w]\n                colors = np.unique(patch)\n                if patch.shape != (h, w) or len(colors) != 1:\n                    self.prediction_failures += 1\n                    self.blocked.add(pending["skill"])\n                    self.events.append({"level": before.levels, "status": "prediction_failed",\n                                        "skill": pending["skill"], "cell": [x, y, w, h]})\n                else:\n                    actual = int(colors[0])\n                    self.palette.update((pending["before_color"], actual))\n                    predicted = pending["expected_next_color"]\n                    if predicted is not None and predicted != actual:\n                        self.prediction_failures += 1\n                        self.events.append({"level": before.levels, "status": "color_transition_counterexample",\n                                            "predicted": predicted, "observed": actual})\n                    self.color_edges[pending["before_color"]].add(actual)\n                    if actual == pending["before_color"]:\n                        identity = (pending["cell"], actual)\n                        self.no_effect[identity] += 1\n                        if self.no_effect[identity] >= 2:\n                            self.blocked.add(pending["skill"])\n            self.pending = None\n        if after.levels > before.levels:\n            if pending is None and relations.pending and relations.target is not None:\n                panels = repeated_panels(old_objects)\n                control = next((p for p in panels if p.origin == relations.control_origin), None)\n                if control is not None:\n                    candidates = fit_role_skills(panels, control, before.pixels,\n                                                 relations.target, after.levels)\n                    existing = {s.key() for s in self.skills}\n                    for skill in candidates:\n                        if skill.key() not in existing:\n                            self.skills.append(skill)\n                            self.learned_count += 1\n                            self.events.append({"status": "learned_after_completion", "skill": asdict(skill)})\n            self.change_level(after.levels)\n        elif action.id == 0:\n            self.pending = None\n\n    def summary(self):\n        return {"retains_across_levels": self.retain, "learned_skills": self.learned_count,\n                "stored_skills": [asdict(s) for s in self.skills],\n                "transfer_actions": self.transfer_actions,\n                "prediction_failures": self.prediction_failures,\n                "palette_discovery_actions": self.palette_probes,\n                "confirmed_transfer_levels": self.confirmed_levels, "events": self.events,\n                "last_plan": self.last_plan}\n', 'relations.py': '"""Small, explicit goal-hypothesis language for repeated binary tile panels.\n\nThis is a visual inductive bias, not a general ARC solver. Layouts and colors\ncome from observations; candidate Boolean expressions are hand-written.\n"""\nfrom __future__ import annotations\n\nfrom collections import defaultdict\nfrom dataclasses import dataclass\nfrom functools import reduce\nfrom itertools import combinations\n\nimport numpy as np\n\n\n@dataclass(frozen=True)\nclass Panel:\n    cells: tuple\n\n    @property\n    def origin(self):\n        return min(c.x for c in self.cells), min(c.y for c in self.cells)\n\n    @property\n    def geometry(self):\n        x, y = self.origin\n        return tuple((c.x - x, c.y - y, c.width, c.height) for c in self.cells)\n\n    @property\n    def colors(self):\n        return tuple(c.color for c in self.cells)\n\n\ndef repeated_panels(objects) -> list[Panel]:\n    by_size = defaultdict(list)\n    for c in objects:\n        if c.width == c.height and 3 <= c.width <= 12 and c.size == c.width * c.height:\n            by_size[c.width].append(c)\n    cohorts = []\n    for width, squares in by_size.items():\n        groups = []\n        remaining = set(range(len(squares)))\n        while remaining:\n            queue = [remaining.pop()]\n            component = []\n            while queue:\n                index = queue.pop()\n                c = squares[index]\n                component.append(c)\n                adjacent = [j for j in remaining if\n                            max(abs(c.x - squares[j].x), abs(c.y - squares[j].y)) <= 2 * width]\n                remaining.difference_update(adjacent)\n                queue.extend(adjacent)\n            if 4 <= len(component) <= 25:\n                groups.append(Panel(tuple(sorted(component, key=lambda c: (c.y, c.x)))))\n        by_geometry = defaultdict(list)\n        for panel in groups:\n            by_geometry[panel.geometry].append(panel)\n        for panels in by_geometry.values():\n            palette = {color for p in panels for color in p.colors}\n            if 3 <= len(panels) <= 5 and len(palette) == 2:\n                cohorts.append(panels)\n    if not cohorts:\n        return []\n    return sorted(max(cohorts, key=lambda ps: len(ps) * len(ps[0].cells)),\n                  key=lambda p: (p.origin[1], p.origin[0]))\n\n\ndef boolean_targets(source_colors: list[tuple[int, ...]], palette: tuple[int, int]):\n    """Enumerate a compact transparent expression library, removing duplicates."""\n    vectors = [tuple(int(c == palette[1]) for c in row) for row in source_colors]\n    n = len(vectors[0])\n    proposals = []\n    all_xor = tuple(reduce(int.__xor__, column) for column in zip(*vectors))\n    proposals.append(("xor(all reference panels)", all_xor))\n    proposals.append(("not xor(all reference panels)", tuple(1 - x for x in all_xor)))\n    for i, vector in enumerate(vectors):\n        proposals.append((f"copy(reference {i})", vector))\n        proposals.append((f"not(reference {i})", tuple(1 - x for x in vector)))\n    for arity in range(2, len(vectors) + 1):\n        for indices in combinations(range(len(vectors)), arity):\n            columns = list(zip(*(vectors[i] for i in indices)))\n            for op, fn in (("xor", lambda c: reduce(int.__xor__, c)),\n                           ("and", lambda c: int(all(c))),\n                           ("or", lambda c: int(any(c)))):\n                vector = tuple(fn(c) for c in columns)\n                proposals.append((f"{op}{indices}", vector))\n                proposals.append((f"not {op}{indices}", tuple(1 - x for x in vector)))\n    proposals.extend([("constant 0", (0,) * n), ("constant 1", (1,) * n)])\n    seen = set()\n    for name, bits in proposals:\n        colors = tuple(palette[bit] for bit in bits)\n        if colors not in seen:\n            seen.add(colors)\n            yield name, colors\n\n\ndef embedded_glyph_targets(panels: list[Panel], control: Panel, pixels):\n    """Fit a small-to-large color map on other panels, then apply it.\n\n    A repeated panel may contain its own small glyph in one missing tile.\n    Candidate symmetries and color maps must agree on every reference cell.\n    This is a finite consistency check over visible examples, not a guarantee\n    about the hidden goal or about other games.\n    """\n    references = [p for p in panels if p.origin != control.origin]\n    if len(references) < 2:\n        return\n    arrays = []\n    positions = None\n    for panel in panels:\n        xs, ys = sorted({c.x for c in panel.cells}), sorted({c.y for c in panel.cells})\n        if len(xs) != len(ys) or not 2 <= len(xs) <= 5:\n            return\n        occupied = {(c.x, c.y) for c in panel.cells}\n        holes = [(x, y) for y in ys for x in xs if (x, y) not in occupied]\n        if len(holes) != 1:\n            return\n        x, y = holes[0]\n        width, height = panel.cells[0].width, panel.cells[0].height\n        n = len(xs)\n        glyph = np.asarray([[int(pixels[y + min(height - 1, int((j + 0.5) * height / n)),\n                                        x + min(width - 1, int((i + 0.5) * width / n))])\n                             for i in range(n)] for j in range(n)])\n        positions = [(ys.index(c.y), xs.index(c.x)) for c in panel.cells]\n        arrays.append((panel, glyph))\n    seen = set()\n    for reflect in (False, True):\n        for rotations in range(4):\n            mapping = {}\n            valid = True\n            control_symbols = None\n            examples = 0\n            for panel, glyph in arrays:\n                transformed = np.rot90(np.fliplr(glyph) if reflect else glyph, rotations)\n                symbols = tuple(int(transformed[y, x]) for y, x in positions)\n                if panel.origin == control.origin:\n                    control_symbols = symbols\n                    continue\n                for symbol, color in zip(symbols, panel.colors):\n                    if symbol in mapping and mapping[symbol] != color:\n                        valid = False\n                        break\n                    mapping[symbol] = color\n                    examples += 1\n            if not valid or len(mapping) < 2 or examples < 8 or control_symbols is None:\n                continue\n            if not all(s in mapping for s in control_symbols):\n                continue\n            target = tuple(mapping[s] for s in control_symbols)\n            if target not in seen:\n                seen.add(target)\n                yield (f"embedded glyph: reflect={reflect}, rotation={rotations * 90}, "\n                       f"color_map={sorted(mapping.items())}, verified_reference_cells={examples}"), target\n\n\nclass PanelHypotheses:\n    def __init__(self):\n        self.level = None\n        self.control_origin = None\n        self.layout = None\n        self.attempted = set()\n        self.target = None\n        self.name = None\n        self.pending = None\n        self.disabled = False\n        self.hypotheses_tested = 0\n        self.controls_identified = 0\n        self.prediction_failures = 0\n        self.completions = []\n\n    def clear_level(self, level):\n        self.level = level\n        self.control_origin = self.layout = self.target = self.name = self.pending = None\n        self.attempted = set()\n        self.disabled = False\n\n    def observe(self, before, action, after, old_objects, new_objects):\n        if self.level != before.levels:\n            self.clear_level(before.levels)\n        if after.levels > before.levels:\n            if self.pending:\n                self.completions.append({"completed_level": after.levels, "hypothesis": self.name})\n            self.clear_level(after.levels)\n            return\n        if action.id == 0:\n            # Do not restart rejected targets after death in the same level.\n            self.pending = None\n            return\n        old_panels, new_panels = repeated_panels(old_objects), repeated_panels(new_objects)\n        old_by_origin = {p.origin: p for p in old_panels}\n        new_by_origin = {p.origin: p for p in new_panels}\n        if self.pending:\n            origin, index, expected_color = self.pending\n            p = new_by_origin.get(origin)\n            if p is None or p.colors[index] != expected_color:\n                self.prediction_failures += 1\n                self.disabled = True\n            self.pending = None\n        if action.id != 6 or self.disabled:\n            return\n        changed = [origin for origin, p in old_by_origin.items()\n                   if origin in new_by_origin and p.geometry == new_by_origin[origin].geometry\n                   and p.colors != new_by_origin[origin].colors]\n        if len(changed) != 1:\n            return\n        origin = changed[0]\n        old, new = old_by_origin[origin], new_by_origin[origin]\n        differing = [i for i, (a, b) in enumerate(zip(old.colors, new.colors)) if a != b]\n        if len(differing) == 1:\n            c = old.cells[differing[0]]\n            if c.x <= action.x < c.x + c.width and c.y <= action.y < c.y + c.height:\n                if self.control_origin is None:\n                    self.controls_identified += 1\n                self.control_origin = origin\n                self.layout = old.geometry\n\n    def propose(self, obs, objects):\n        if self.disabled or self.control_origin is None or 6 not in obs.available:\n            return None\n        panels = repeated_panels(objects)\n        control = next((p for p in panels if p.origin == self.control_origin), None)\n        if control is None or control.geometry != self.layout:\n            return None\n        palette = tuple(sorted({color for p in panels for color in p.colors}))\n        if len(palette) != 2:\n            return None\n        if self.target == control.colors:\n            self.attempted.add(self.target)\n            self.hypotheses_tested += 1\n            self.target = None\n        if self.target is None:\n            references = [p.colors for p in panels if p.origin != self.control_origin]\n            if not references:\n                return None\n            proposals = list(embedded_glyph_targets(panels, control, obs.pixels))\n            proposals.extend(boolean_targets(references, palette))\n            for name, target in proposals:\n                if target not in self.attempted and target != control.colors:\n                    self.name, self.target = name, target\n                    break\n        if self.target is None:\n            return None\n        for index, (current, desired) in enumerate(zip(control.colors, self.target)):\n            if current != desired:\n                c = control.cells[index]\n                self.pending = (control.origin, index, desired)\n                return c.center, self.name\n        return None\n\n    def summary(self):\n        return {"controls_identified": self.controls_identified,\n                "goal_candidates_reached_without_completion": self.hypotheses_tested,\n                "toggle_prediction_failures": self.prediction_failures,\n                "completion_hypotheses": self.completions}\n', 'experiment_free_transfer.py': "from experiment_refined import Explorer as C5Explorer\nfrom agent import Observation\nfrom free_transfer_controls import FreeTransferControls\n\n\nclass Explorer(C5Explorer):\n    def __init__(self, seed=0, policy='free_transfer'):\n        if policy != 'free_transfer': raise ValueError(policy)\n        super().__init__(seed, 'refined_controls')\n        self.shared = FreeTransferControls(seed)\n        self.variant = policy\n", 'free_transfer_controls.py': 'from efficient_controls import EfficientControls\nfrom gated_controller import OrientedGatedController\nfrom transfer_space import TransferSpace\n\n\nclass FreeTransferControls(EfficientControls):\n    def choose(self, obs, objects, candidates):\n        self._role_level(obs.levels)\n        if self.gated is None:\n            self.gated = OrientedGatedController.detect(obs, objects, self.models, self.control_support, self.role_history)\n            if self.gated is None:\n                self.gated = TransferSpace.detect(obs, objects, self.models, self.control_support, self.role_history)\n        return super().choose(obs, objects, candidates)\n', 'transfer_space.py': '"""Plan observed coordinates/resources and candidate position transfers jointly.\n\nThis visual hypothesis does not require reservoirs to form an ordered chain.\nRectangular resource profiles, matching markers and visible passage segments\nare hypotheses; measured state transitions override predicted transitions.\nOnly public pixels and requested action outcomes enter this controller.\n"""\nfrom collections import Counter, defaultdict\nfrom types import SimpleNamespace\nimport heapq, itertools\nimport numpy as np\nfrom agent import Action, components\nfrom shared_dynamics import signature, entities, unique_entities\n\n\ndef scene(obs):\n    return unique_entities(entities(components(obs.pixels), obs.pixels.shape))\n\n\ndef intervals(mask):\n    start = None\n    for i, present in enumerate(list(mask) + [False]):\n        if present and start is None: start = i\n        if not present and start is not None:\n            yield start, i\n            start = None\n\n\nclass TransferSpace:\n    @classmethod\n    def detect(cls, obs, objects, models, support, history):\n        if set(obs.available) != {6} or len(history) < 2: return None\n        supported = {s for s, locations in support.items() if len(locations) >= 2}\n        controls = sorted({(c.x+c.width//2, c.y+c.height//2) for c in objects if signature(c) in supported})\n        if len(controls) < 4: return None\n        original_scene = scene(obs)\n        moved = set()\n        motion = Counter()\n        for by_signature in models.values():\n            for sig, counts in by_signature.items():\n                if sig not in original_scene: continue\n                for (dx, dy), count in counts.items():\n                    if bool(dx) != bool(dy):\n                        moved.add(sig)\n                        motion[0 if dx else 1] += count\n        if not moved: return None\n        axis = motion.most_common(1)[0][0]\n        self = cls()\n        self.transposed = axis == 1\n        self.level = obs.levels\n        control_rects = [(c.x,c.y,c.width,c.height) for c in objects if signature(c) in supported]\n        if self.transposed: control_rects = [(y,x,height,width) for x,y,width,height in control_rects]\n        self.control_rects = control_rects\n        current = self.transform(obs)\n        pixels = current.pixels\n        h, w = pixels.shape\n        objects = components(pixels)\n        current_scene = scene(current)\n        if self.transposed:\n            translated = set()\n            for sig in moved:\n                x, y = original_scene[sig]\n                translated.update(signature(c) for c in objects if (c.x, c.y, c.width, c.height, c.color) == (y, x, sig[2], sig[1], sig[0]))\n            moved = translated\n        # A third glyph may not have moved during the first two probes yet.\n        # Matching observed moving geometry supplies a tentative actor role.\n        moving_shapes = {sig[1:] for sig in moved}\n        moved.update(sig for sig in current_scene if sig[1:] in moving_shapes)\n        # Repeated thin segments bordered by the same stationary wall color.\n        forms = defaultdict(list)\n        for c in objects:\n            if not (c.x > 0 and c.y > 1 and c.x+c.width < w and c.y+c.height < h): continue\n            if c.width < 3*c.height or c.height > h//8 or c.size != c.width*c.height: continue\n            y = c.y+c.height//2\n            left, right = int(pixels[y, c.x-1]), int(pixels[y, c.x+c.width])\n            if left == right and left != c.color:\n                forms[(signature(c), left)].append(c)\n        repeated = [(key, group) for key, group in forms.items() if len(group) >= 2]\n        if not repeated: return None\n        (form, wall), segments = max(repeated, key=lambda pair: len(pair[1]))\n        thickness = form[2]\n        levels = sorted({(c.y, c.y+c.height) for c in segments})\n        if any(b > c for (_, b), (c, _) in zip(levels, levels[1:])): return None\n        bands = [(a, b) for a, b in zip([1]+[b for a,b in levels], [a for a,b in levels]+[h]) if b-a >= 2*thickness]\n        cells = []\n        for y0, y1 in bands:\n            solid = np.mean(pixels[y0:y1] == wall, axis=0) >= .6\n            for x0, x1 in intervals(~solid):\n                if x1-x0 >= 3*thickness: cells.append((x0, y0, x1, y1))\n        if len(cells) < 3 or len(cells) > 12: return None\n        self.cells = cells\n        self.controls = sorted((y,x) if self.transposed else (x,y) for x,y in controls)\n        self.windows = []\n        self.gate_rects = []\n        for c in segments:\n            x = c.x+c.width//2\n            a, b = self.cell_at(x, c.y-1), self.cell_at(x, c.y+c.height)\n            if a is not None and b is not None and a != b:\n                self.windows.append((a, b, c.x, c.x+c.width, x, c.y+c.height//2))\n                self.gate_rects.append((c.x,c.y,c.width,c.height))\n        if len(self.windows) < 2: return None\n        pairs = []\n        for mover in sorted(moved):\n            if mover not in current_scene: continue\n            for target, pos in current_scene.items():\n                if target in moved or mover[0] != target[0] or target[2] > 2*thickness: continue\n                if any(a <= pos[1] < b for a,b in levels): pairs.append((mover, target, pos))\n        if not pairs or len(pairs) > 4: return None\n        self.pairs = pairs\n        transformed_history = [(self.transform(b), self.action(a), self.transform(c)) for b,a,c in history]\n        changed_colors = Counter()\n        for before, action, after in transformed_history:\n            if before.levels != after.levels or before.pixels.shape != after.pixels.shape: continue\n            a, b = before.pixels[2:-2, 2:-2], after.pixels[2:-2, 2:-2]\n            mask = a != b\n            changed_colors.update(map(int, a[mask])); changed_colors.update(map(int, b[mask]))\n        if len(changed_colors) < 2: return None\n        likely = [color for color, count in changed_colors.most_common(2)]\n        self.fluid = max(likely, key=lambda c: np.count_nonzero(pixels == c))\n        # Static padding is not a reservoir with no fluid. Crop the inferred\n        # regions to the envelope of both empirically exchanging colors.\n        ys, xs = np.where(np.isin(pixels, likely))\n        if not len(xs): return None\n        xmin, xmax, ymin, ymax = int(xs.min()), int(xs.max()+1), int(ys.min()), int(ys.max()+1)\n        trimmed = [(max(x0,xmin),max(y0,ymin),min(x1,xmax),min(y1,ymax)) for x0,y0,x1,y1 in self.cells]\n        if any(x1 <= x0 or y1 <= y0 for x0,y0,x1,y1 in trimmed): return None\n        self.cells = trimmed\n        options = []\n        for direction in (1, -1):\n            self.direction = direction\n            differences = sum(sum(x != y for x,y in zip(self.resources(b), self.resources(c)))\n                              for b,a,c in transformed_history if b.levels == c.levels)\n            options.append((differences, direction))\n        if max(options)[0] == 0: return None\n        self.direction = max(options)[1]\n        resource = self.resources(current)\n        self.offsets = []\n        self.extents = []\n        self.goals = []\n        for mover, target, (tx, ty) in pairs:\n            x, y = current_scene[mover]\n            cell = self.cell_at(x+mover[1]/2, y+mover[2]/2)\n            if cell is None: return None\n            self.offsets.append(x-resource[cell])\n            adjacent = [c for c in objects if c.size <= 64 and c.y == y and c.height == mover[2]\n                        and (c.x+c.width == x or c.x == x+mover[1])]\n            left = min([x]+[c.x for c in adjacent]) - x\n            right = max([x+mover[1]]+[c.x+c.width for c in adjacent]) - x\n            self.extents.append((left, right))\n            destinations = {c for c in (self.cell_at(tx+target[1]/2, ty-1),\n                                       self.cell_at(tx+target[1]/2, ty+target[2])) if c is not None}\n            if not destinations: return None\n            desired = 2*tx+target[1]-mover[1]-2*self.offsets[-1]\n            self.goals.append((destinations, desired))\n        self.increments = defaultdict(Counter)\n        self.exact = {}\n        self.stats = Counter()\n        self.events = []\n        self.last_prediction = None\n        self.failed_searches = set()\n        for before, action, after in transformed_history: self.observe_inner(before, action, after)\n        if self.read(current) is None: return None\n        return self\n\n    def transform(self, obs):\n        return SimpleNamespace(pixels=obs.pixels.T if self.transposed else obs.pixels,\n                               levels=obs.levels, state=obs.state, available=obs.available)\n\n    def action(self, action):\n        return Action(action.id, action.y, action.x) if self.transposed and action.id == 6 else action\n\n    def cell_at(self, x, y):\n        return next((i for i,(x0,y0,x1,y1) in enumerate(self.cells) if x0 <= x < x1 and y0 <= y < y1), None)\n\n    def resources(self, obs):\n        result = []\n        for x0,y0,x1,y1 in self.cells:\n            a = obs.pixels[y0:y1, x0:x1] == self.fluid\n            if self.direction == 1:\n                values = np.where(a.any(axis=1), a.argmax(axis=1), x1-x0)+x0\n            else:\n                values = np.where(a.any(axis=1), (x1-x0)-a[:,::-1].argmax(axis=1), 0)+x0\n            result.append(int(np.median(values)))\n        return tuple(result)\n\n    def read(self, obs):\n        if obs.levels != self.level: return None\n        positions = scene(obs)\n        actor_cells = []\n        for mover, target, pos in self.pairs:\n            if mover not in positions: return None\n            x, y = positions[mover]\n            cell = self.cell_at(x+mover[1]/2, y+mover[2]/2)\n            if cell is None: return None\n            actor_cells.append(cell)\n        return self.resources(obs), tuple(actor_cells)\n\n    def observe_inner(self, before, action, after):\n        if before.levels != self.level or after.levels != self.level or action.id != 6 or after.state in (\'WIN\', \'GAME_OVER\'): return\n        b, a = self.read(before), self.read(after)\n        if b is None or a is None: return\n        key = (action.x, action.y)\n        for x,y,width,height in self.control_rects+self.gate_rects:\n            if x <= action.x < x+width and y <= action.y < y+height:\n                key = (x+width//2,y+height//2)\n                break\n        old_prediction = next((predicted for candidate,predicted in self.choices(b) if candidate == key), b)\n        self.exact[b, key] = a\n        if key in self.controls:\n            delta = tuple(y-x for x,y in zip(b[0], a[0]))\n            if any(delta): self.increments[key][delta] += 1\n        if old_prediction != a:\n            # A new effect changes the current hypothesis. Previously failed\n            # bounded searches may become useful again under the revised model.\n            self.failed_searches.clear()\n            self.stats[\'search_cache_invalidations\'] += 1\n        if self.last_prediction is not None:\n            self.stats[\'predictions_checked\'] += 1\n            self.stats[\'prediction_failures\'] += int(a != self.last_prediction)\n            self.last_prediction = None\n        if a[1] != b[1]:\n            self.stats[\'observed_position_transfers\'] += 1\n            if len(self.events) < 48: self.events.append({\'from\': b[1], \'to\': a[1], \'action\': key})\n\n    def observe(self, before, action, after):\n        self.observe_inner(self.transform(before), self.action(action), self.transform(after))\n\n    def valid(self, state):\n        resources, actors = state\n        if any(not x0 <= q <= x1 for q,(x0,y0,x1,y1) in zip(resources, self.cells)): return False\n        if len(actors) != len(set(actors)): return False\n        for i, cell in enumerate(actors):\n            x = resources[cell]+self.offsets[i]\n            left, right = self.extents[i]\n            if x+left < self.cells[cell][0] or x+right > self.cells[cell][2]: return False\n        return True\n\n    def finished(self, state):\n        resources, actors = state\n        return all(cell in goals and 2*resources[cell] == desired for cell,(goals,desired) in zip(actors,self.goals))\n\n    def choices(self, state):\n        resources, actors = state\n        for key, counts in self.increments.items():\n            if not counts: continue\n            delta = counts.most_common(1)[0][0]\n            predicted = tuple(x+d for x,d in zip(resources, delta)), actors\n            after = self.exact.get((state,key), predicted)\n            if after != state and self.valid(after): yield key, after\n        for a,b,lo,hi,x,y in self.windows:\n            key = (x,y)\n            required = lo if self.direction == 1 else hi\n            if resources[a] != required or resources[b] != required: continue\n            locations = tuple(b if c == a else a if c == b else c for c in actors)\n            after = self.exact.get((state,key), (resources,locations))\n            if after != state and self.valid(after): yield key, after\n\n    def search(self, start):\n        unit = max([max(map(abs,c.most_common(1)[0][0])) for c in self.increments.values() if c] or [1])\n        def heuristic(state):\n            r, actors = state\n            return sum(min(abs(2*r[c]-desired)/(2*unit) for c in goals) + (actor not in goals)\n                       for actor,(goals,desired) in zip(actors,self.goals))\n        serial = itertools.count()\n        queue = [(heuristic(start), 0, next(serial), start, ())]\n        best = {start: 0}\n        nodes = 0\n        while queue and nodes < 25000:\n            _, cost, _, state, path = heapq.heappop(queue)\n            if cost != best[state]: continue\n            nodes += 1\n            if self.finished(state):\n                self.stats[\'search_nodes\'] += nodes\n                return path\n            if cost >= 80: continue\n            for key, after in self.choices(state):\n                if cost+1 >= best.get(after, 10**9): continue\n                best[after] = cost+1\n                heapq.heappush(queue, (cost+1+heuristic(after),cost+1,next(serial),after,path+(key,)))\n        self.stats[\'search_nodes\'] += nodes\n        return None\n\n    def arrangement_goal(self, actors, resources):\n        if len(set(actors)) != len(actors): return False\n        required = {}\n        for i,(cell,(allowed,desired)) in enumerate(zip(actors,self.goals)):\n            if cell not in allowed or desired % 2: return False\n            value = desired//2\n            x = value+self.offsets[i]\n            left,right = self.extents[i]\n            x0,_,x1,_ = self.cells[cell]\n            if x+left < x0 or x+right > x1: return False\n            required[cell] = value\n        deltas = [counts.most_common(1)[0][0] for counts in self.increments.values() if counts]\n        if deltas and all(sum(d)==0 for d in deltas):\n            remaining = sum(resources)-sum(required.values())\n            free = [bounds for i,bounds in enumerate(self.cells) if i not in required]\n            if not sum(c[0] for c in free) <= remaining <= sum(c[2] for c in free): return False\n        return True\n\n    def staged_search(self, start):\n        from collections import deque\n        resources, actors = start\n        queue = deque([(actors,())]); seen = {actors}; route = None\n        while queue and len(seen) < 4096:\n            positions, path = queue.popleft()\n            if self.arrangement_goal(positions,resources): route = path; break\n            for index,(a,b,*_) in enumerate(self.windows):\n                after = tuple(b if c==a else a if c==b else c for c in positions)\n                if after != positions and len(set(after))==len(after) and after not in seen:\n                    seen.add(after);queue.append((after,path+(index,)))\n        if route is None: return None\n        if route:\n            a,b,lo,hi,x,y = self.windows[route[0]]\n            required = lo if self.direction==1 else hi\n            goal = lambda state: state[0][a]==state[0][b]==required\n            distance = lambda state: abs(state[0][a]-required)+abs(state[0][b]-required)\n            gate = (x,y)\n            if goal(start):\n                if any(key==gate for key,after in self.choices(start)): return (gate,)\n                return None\n        else:\n            goal = self.finished\n            distance = lambda state: sum(abs(2*state[0][cell]-desired)/2 for cell,(allowed,desired) in zip(state[1],self.goals))\n        unit = max([sum(map(abs,c.most_common(1)[0][0])) for c in self.increments.values() if c] or [1])\n        serial=itertools.count();heap=[(distance(start)/unit,0,next(serial),start,())];best={start:0};nodes=0\n        while heap and nodes < 25000:\n            _,cost,_,state,path=heapq.heappop(heap)\n            if cost != best[state]: continue\n            nodes+=1\n            if goal(state):\n                self.stats[\'staged_search_nodes\']+=nodes\n                return path\n            if cost >= 80: continue\n            for key,after in self.choices(state):\n                if key not in self.controls or after[1]!=actors: continue\n                if cost+1 >= best.get(after,10**9): continue\n                best[after]=cost+1\n                heapq.heappush(heap,(cost+1+distance(after)/unit,cost+1,next(serial),after,path+(key,)))\n        self.stats[\'staged_search_nodes\']+=nodes\n        return None\n\n    def choose(self, obs):\n        canonical = self.transform(obs)\n        state = self.read(canonical)\n        if state is None: return None\n        untried = [key for key in self.controls if not self.increments[key] and (state,key) not in self.exact]\n        if untried:\n            self.stats[\'control_probes\'] += 1\n            key = untried[0]\n            return self.action(Action(6,*key)), \'transfer-space: measure a control in the current resource state\'\n        if state in self.failed_searches:\n            self.stats[\'repeated_bounded_searches_avoided\'] += 1\n            return None\n        path = self.staged_search(state) if len(self.pairs)>2 else self.search(state)\n        if not path:\n            self.failed_searches.add(state)\n            self.stats[\'no_complete_plan\'] += 1\n            return None\n        key = path[0]\n        self.last_prediction = next(after for k,after in self.choices(state) if k == key)\n        self.stats[\'planned_actions\'] += 1\n        return self.action(Action(6,*key)), \'transfer-space: jointly plan resources and a candidate position transfer\'\n\n    def summary(self):\n        return {\'kind\': \'transfer_space\', \'stats\': dict(self.stats), \'events\': self.events,\n                \'cells\': self.cells, \'windows\': self.windows, \'actors\': len(self.pairs),\n                \'direction\': self.direction, \'transposed\': self.transposed,\n                \'scope\': \'Measured controls and candidate coordinate transfers; branched visual regions allowed; no saved solution actions.\'}\n', 'gated_controller.py': '"""A staged visual hypothesis for aligned reservoirs and clickable passages.\n\nAll coordinates, colors, goals and action increments come from public images\nand actual transitions. The rectangular-reservoir interpretation is a visual\nhypothesis, checked again after every requested action; not a dynamics proof.\n"""\nfrom collections import Counter, defaultdict, deque\nfrom types import SimpleNamespace\nimport heapq,itertools\nimport numpy as np\nfrom agent import Action,components\nfrom shared_dynamics import signature,unique_entities,entities\n\n\nclass GatedController:\n    @classmethod\n    def detect(cls,obs,objects,models,support,history):\n        if set(obs.available)!={6}:\n            return None\n        forms={s for s,places in support.items() if len(places)>=2}\n        controls=sorted({(c.x+c.width//2,c.y+c.height//2) for c in objects if signature(c) in forms})\n        if len(controls)<4:\n            return None\n        h,w=obs.pixels.shape\n        narrow=[c for c in objects if c.width<=w//8 and c.height>=max(h//8,3*c.width)]\n        widths=Counter(c.width for c in narrow)\n        if not widths:return None\n        bw=widths.most_common(1)[0][0]\n        narrow=[c for c in narrow if c.width==bw]\n        bars=sorted({c.x for c in narrow})\n        if len(bars)<3 or any(b-a<=2*bw for a,b in zip(bars,bars[1:])):return None\n        bar_color=Counter()\n        for c in narrow:bar_color[c.color]+=c.size\n        dominant=bar_color.most_common(1)[0][0]\n        windows=[(bars.index(c.x),c.y,c.y+c.height,c.x+bw//2,c.y+c.height//2) for c in narrow if c.color!=dominant]\n        if not windows:return None\n        scene=unique_entities(entities(objects,obs.pixels.shape))\n        movers=set()\n        for by_signature in models.values():\n            for sig,counts in by_signature.items():\n                if sig in scene and any(dy and not dx for dx,dy in counts):movers.add(sig)\n        # Transfer the empirically moving glyph\'s role to matching geometry.\n        moving_shapes={sig[1:] for sig in movers}\n        movers.update(sig for sig in scene if sig[1:] in moving_shapes)\n        pairs=[]\n        for mover in movers:\n            for target,(x,y) in scene.items():\n                if target!=mover and target[0]==mover[0] and target not in movers and x in bars and target[1]==bw and target[2]<=2*bw:\n                    pairs.append((mover,target,x,y))\n        if not pairs or len(pairs)>4:return None\n        pairs.sort()\n        mover,target,tx,ty=pairs[0]\n        ax,ay=scene[mover]\n        if ay+mover[2]>=h:return None\n        fluid=int(obs.pixels[ay+mover[2],ax+mover[1]//2])\n        if fluid in (mover[0],dominant):return None\n        self=cls()\n        self.level=obs.levels;self.pairs=pairs\n        self.fluid=fluid;self.bars=bars;self.bw=bw;self.h=h\n        starts=[0]+[x+bw for x in bars];ends=bars+[w]\n        self.columns=[(a+b-1)//2 for a,b in zip(starts,ends)]\n        self.ranges=list(zip(starts,ends));self.windows=windows;self.controls=controls\n        bar_tops=[min(c.y for c in objects if c.x==x and c.width==bw and c.color==dominant) for x in bars]\n        self.minimum_levels=[bar_tops[0]]+[max(a,b) for a,b in zip(bar_tops,bar_tops[1:])]+[bar_tops[-1]]\n        self.destinations=[]\n        for mover,target,tx,ty in pairs:\n            cx=scene[mover][0]+mover[1]/2\n            chamber=next((i for i,(a,b) in enumerate(self.ranges) if a<=cx<b),None)\n            if chamber is None:return None\n            marker=bars.index(tx)\n            ys=np.flatnonzero(obs.pixels[2:,self.columns[chamber]]==fluid)\n            level=int(ys[0]+2) if len(ys) else h\n            desired=ty+(target[2]-mover[2])/2+level-scene[mover][1]\n            feasible=[i for i in (marker,marker+1) if self.minimum_levels[i]<=desired<=h]\n            if not feasible:return None\n            self.destinations.append(min(feasible,key=lambda i:abs(i-chamber)))\n        self.destinations=tuple(self.destinations)\n        self.increments=defaultdict(Counter);self.exact={};self.trials=Counter();self.stats=Counter();self.events=[]\n        self.last_prediction=None\n        self.failed_passages=set()\n        for before,action,after in history:self.observe(before,action,after)\n        if self.read(obs) is None:return None\n        return self\n\n    def read(self,obs):\n        if obs.levels!=self.level or obs.pixels.shape[0]!=self.h:return None\n        scene=unique_entities(entities(components(obs.pixels),obs.pixels.shape))\n        positions=[scene.get(pair[0]) for pair in self.pairs]\n        if any(pos is None for pos in positions):return None\n        levels=[]\n        for x in self.columns:\n            ys=np.flatnonzero(obs.pixels[2:,x]==self.fluid)\n            levels.append(int(ys[0]+2) if len(ys) else self.h)\n        chambers=[]\n        for pos,(mover,*_) in zip(positions,self.pairs):\n            center=pos[0]+mover[1]/2\n            chamber=next((i for i,(a,b) in enumerate(self.ranges) if a<=center<b),None)\n            if chamber is None:return None\n            chambers.append(chamber)\n        return tuple(levels),tuple(chambers),positions\n\n    def observe(self,before,action,after):\n        if before.levels!=self.level or after.levels!=self.level or action.id!=6 or after.state in (\'GAME_OVER\',\'WIN\'):return\n        b=self.read(before);a=self.read(after)\n        if b is None or a is None:return\n        key=(action.x,action.y)\n        if key in self.controls:\n            delta=tuple(y-x for x,y in zip(b[0],a[0]))\n            self.trials[key]+=1\n            self.exact[b[0],key]=a[0]\n            if any(delta):self.increments[key][delta]+=1\n        if self.last_prediction is not None:\n            self.stats[\'predictions_checked\']+=1\n            self.stats[\'prediction_failures\']+=int(self.last_prediction!=a[0])\n            self.last_prediction=None\n        if a[1]!=b[1]:\n            self.stats[\'observed_passages\']+=1\n            if len(self.events)<48:self.events.append({\'kind\':\'passage\',\'from\':b[1],\'to\':a[1],\'action\':key})\n        else:\n            for edge,top,bottom,x,y in self.windows:\n                if key==(x,y):\n                    self.failed_passages.add((b[0],b[1],edge))\n                    self.stats[\'failed_passage_probes\']+=1\n\n    def search(self,start,goal,heuristic):\n        catalogue=[(k,c.most_common(1)[0][0]) for k,c in self.increments.items() if c]\n        serial=itertools.count();q=[(heuristic(start),0,next(serial),start,())];costs={start:0};nodes=0\n        while q and nodes<18000:\n            _,cost,_,state,path=heapq.heappop(q)\n            if cost!=costs[state]:continue\n            nodes+=1\n            if goal(state):\n                self.stats[\'search_nodes\']+=nodes\n                return path\n            if cost>=60:continue\n            for key,delta in catalogue:\n                after=self.exact.get((state,key),tuple(x+d for x,d in zip(state,delta)))\n                if after==state or any(x<lo or x>self.h for x,lo in zip(after,self.minimum_levels)):continue\n                if cost+1>=costs.get(after,1e9):continue\n                costs[after]=cost+1\n                heapq.heappush(q,(cost+1+heuristic(after),cost+1,next(serial),after,path+(key,)))\n        self.stats[\'search_nodes\']+=nodes\n        return None\n\n    def choose(self,obs):\n        reading=self.read(obs)\n        if reading is None:return None\n        state,chambers,positions=reading\n        unit=max([max(abs(v) for v in c.most_common(1)[0][0]) for c in self.increments.values() if c] or [1])\n        if chambers!=self.destinations:\n            # Plan all actor transfers together; a passage can exchange both.\n            queue=deque([(chambers,())]);seen={chambers};route=None\n            while queue and len(seen)<4096:\n                arrangement,path=queue.popleft()\n                if arrangement==self.destinations:route=path;break\n                for edge,*_ in self.windows:\n                    after=tuple(edge+1 if c==edge else edge if c==edge+1 else c for c in arrangement)\n                    if after!=arrangement and len(set(after))==len(after) and after not in seen:\n                        seen.add(after);queue.append((after,path+(edge,)))\n            if not route:return None\n            edge=route[0]\n            choices=[g for g in self.windows if g[0]==edge]\n            if not choices:return None\n            _,top,bottom,x,y=choices[0]\n            goal=lambda s:s[edge]==s[edge+1]==bottom and (s,chambers,edge) not in self.failed_passages\n            def distance(s):\n                return (abs(s[edge]-bottom)+abs(s[edge+1]-bottom))/(2*unit)\n            if goal(state):\n                self.stats[\'passage_probes\']+=1\n                return Action(6,x,y),\'gated: equalized adjacent levels; test visible passage\'\n            reason=\'gated: prepare equal adjacent levels at the next visible passage\'\n        else:\n            desired=[(chamber,2*ty+target[2]-mover[2]+2*(state[chamber]-pos[1])) for chamber,pos,(mover,target,tx,ty) in zip(chambers,positions,self.pairs)]\n            goal=lambda s:all(2*s[c]==value for c,value in desired)\n            distance=lambda s:sum(abs(2*s[c]-value) for c,value in desired)/(2*unit)\n            reason=\'gated: align the transported actor with the visible reference\'\n        # A zero effect at a full/empty boundary is not a permanent no-op.\n        # Retry an unmodelled control only in a new observed resource state.\n        untried=[k for k in self.controls if not self.increments[k] and (state,k) not in self.exact]\n        if untried:\n            key=untried[0];self.stats[\'control_probes\']+=1\n            return Action(6,*key),\'gated: measure a remaining repeated control\'\n        path=self.search(state,goal,distance)\n        if path:\n            key=path[0];delta=self.increments[key].most_common(1)[0][0]\n            self.last_prediction=self.exact.get((state,key),tuple(x+d for x,d in zip(state,delta)))\n            self.stats[\'planned_actions\']+=1\n            return Action(6,*key),reason\n        self.stats[\'no_complete_plan\']+=1\n        return None\n\n    def summary(self):\n        return {\'stats\':dict(self.stats),\'events\':self.events,\'actors\':len(self.pairs),\'learned_controls\':len(self.increments),\'scope\':\'Public-image reservoir/window hypothesis, measured click increments, staged alignment; no game source or fixed action tape.\'}\n\n\ndef transpose_obs(obs):\n    return SimpleNamespace(pixels=obs.pixels.T,levels=obs.levels,state=obs.state,available=obs.available)\n\n\nclass OrientedGatedController:\n    @classmethod\n    def detect(cls,obs,objects,models,support,history):\n        if set(obs.available)!={6} or not any(len(v)>=2 for v in support.values()):return None\n        vertical=GatedController.detect(obs,objects,models,support,history)\n        if vertical:return vertical\n        transposed=transpose_obs(obs);other_objects=components(transposed.pixels)\n        # Reject unsupported geometry before reconstructing historical models.\n        # These are the same necessary layout checks used by detect().\n        h,w=transposed.pixels.shape\n        narrow=[c for c in other_objects if c.width<=w//8 and c.height>=max(h//8,3*c.width)]\n        widths=Counter(c.width for c in narrow)\n        if not widths:return None\n        bw=widths.most_common(1)[0][0]\n        bars=sorted({c.x for c in narrow if c.width==bw})\n        if len(bars)<3 or any(b-a<=2*bw for a,b in zip(bars,bars[1:])):return None\n        other_support=defaultdict(set)\n        for c in objects:\n            if (c.x,c.y) in support.get(signature(c),set()):\n                other=next((t for t in other_objects if t.x==c.y and t.y==c.x and t.width==c.height and t.height==c.width and t.color==c.color),None)\n                if other is not None:other_support[signature(other)].add((other.x,other.y))\n        other_history=[];other_models=defaultdict(lambda:defaultdict(Counter))\n        for before,action,after in history:\n            b,a=transpose_obs(before),transpose_obs(after)\n            translated=Action(action.id,action.y,action.x) if action.id==6 else action\n            other_history.append((b,translated,a))\n            old=unique_entities(entities(components(b.pixels),b.pixels.shape));new=unique_entities(entities(components(a.pixels),a.pixels.shape))\n            for sig,pos in old.items():\n                if sig in new:\n                    delta=(new[sig][0]-pos[0],new[sig][1]-pos[1])\n                    if any(delta):other_models[(translated.x,translated.y)][sig][delta]+=1\n        inner=GatedController.detect(transposed,other_objects,other_models,other_support,other_history)\n        if inner is None:return None\n        self=cls();self.inner=inner;return self\n\n    def choose(self,obs):\n        result=self.inner.choose(transpose_obs(obs))\n        if result is None:return None\n        action,reason=result\n        return Action(action.id,action.y,action.x),reason\n\n    def observe(self,before,action,after):\n        translated=Action(action.id,action.y,action.x) if action.id==6 else action\n        self.inner.observe(transpose_obs(before),translated,transpose_obs(after))\n\n    def summary(self):\n        return dict(self.inner.summary(),orientation=\'transposed\')\n', 'efficient_controls.py': '"""Use repeated, stationary click controls only after observed effects support them.\n\nThis is an empirical control-role hypothesis, not a state-equivalence claim.\nNo palette, coordinate, level, game name or saved action sequence is supplied.\n"""\nfrom collections import Counter, defaultdict\nimport numpy as np\nfrom boundary_motion import BoundaryMotionDynamics\nfrom shared_dynamics import signature\nfrom gated_controller import OrientedGatedController\n\n\nclass EfficientControls(BoundaryMotionDynamics):\n    def __init__(self, seed=0):\n        super().__init__(seed)\n        self.control_role_level = None\n        self.control_support = defaultdict(set)\n        self.control_role_stats = Counter()\n        self.gated=None\n        self.role_history=[]\n        self.gated_reports=[]\n\n    def _role_level(self, level):\n        if level != self.control_role_level:\n            if self.gated is not None:self.gated_reports.append(self.gated.summary())\n            self.gated=None\n            self.role_history=[]\n            self.control_role_level = level\n            self.control_support.clear()\n\n    @staticmethod\n    def _clicked(action, objects):\n        if action.id != 6:\n            return None\n        matches = [c for c in objects if c.x <= action.x < c.x+c.width and c.y <= action.y < c.y+c.height]\n        return min(matches, key=lambda c:c.size) if matches else None\n\n    def choose(self, obs, objects, candidates):\n        self._role_level(obs.levels)\n        if self.gated is None:\n            self.gated=OrientedGatedController.detect(obs,objects,self.models,self.control_support,self.role_history)\n        if self.gated is not None:\n            proposal=self.gated.choose(obs)\n            if proposal is not None:\n                self.pending=None\n                self.projected_pending=None\n                return proposal\n        supported = {form for form, locations in self.control_support.items() if len(locations)>=2}\n        if supported and candidates and all(a.id==6 for a in candidates):\n            selected=[]\n            for action in candidates:\n                obj=self._clicked(action,objects)\n                if obj is not None and signature(obj) in supported:\n                    selected.append(action)\n            if len(selected)>=2:\n                self.control_role_stats[\'filtered_decisions\']+=1\n                self.control_role_stats[\'candidates_removed\']+=len(candidates)-len(selected)\n                candidates=selected\n        return super().choose(obs,objects,candidates)\n\n    def observe(self,before,action,after,old_objects,new_objects):\n        self._role_level(before.levels)\n        if self.gated is not None:self.gated.observe(before,action,after)\n        if before.levels==after.levels and action.id:\n            self.role_history.append((before,action,after))\n            self.role_history=self.role_history[-32:]\n        obj=self._clicked(action,old_objects)\n        if obj is not None and obj.size<=64 and max(obj.width,obj.height)<=12 and before.levels==after.levels and after.state not in (\'GAME_OVER\',\'WIN\') and before.pixels.shape==after.pixels.shape:\n            stable=any(signature(c)==signature(obj) and (c.x,c.y)==(obj.x,obj.y) for c in new_objects)\n            changed=np.count_nonzero(before.pixels[2:-2,2:-2]!=after.pixels[2:-2,2:-2])\n            if stable and changed:\n                self.control_support[signature(obj)].add((obj.x,obj.y))\n                self.control_role_stats[\'effective_stationary_clicks\']+=1\n        super().observe(before,action,after,old_objects,new_objects)\n\n    def summary(self):\n        result=super().summary()\n        result[\'gated_reservoirs\']={\'completed\':self.gated_reports,\'current\':self.gated.summary() if self.gated else None}\n        result[\'control_roles\']={\'stats\':dict(self.control_role_stats),\'supported_forms\':len([s for s in self.control_support.values() if len(s)>=2]),\'scope\':\'Same-level repeated stationary click-form hypothesis from public changes; retained across RESET, invalidated on level change.\'}\n        return result\n', 'boundary_motion.py': '"""Fit a boundary-triggered long-motion hypothesis from public transitions.\n\nNo palette indices, coordinates, exit sequence, or game ID are constants.\nRules are accepted only when a measured nonlocal transition fits the visible\nstrip and maximal-clear-run hypothesis exactly. Transfer remains falsifiable.\n"""\nfrom collections import Counter, defaultdict\nimport numpy as np\nfrom goal_navigation import GoalNavigationDynamics\nfrom context_contact import body_box\nfrom shared_dynamics import entities\n\n\ndef strip(pixels,box,axis,sign):\n    x,y,w,h=box\n    if axis==0:\n        xx=x+w if sign<0 else x-1\n        return pixels[y:y+h,xx] if 0<=xx<pixels.shape[1] else np.array([])\n    yy=y+h if sign<0 else y-1\n    return pixels[yy,x:x+w] if 0<=yy<pixels.shape[0] else np.array([])\n\n\ndef slide_destination(pixels,nominal_box,rule,ignore_boxes=()):\n    boundary,floor,axis,sign,stride=rule\n    edge=strip(pixels,nominal_box,axis,sign)\n    if edge.size==0 or not np.all(edge==boundary):return None\n    grid=pixels.copy()\n    for x,y,w,h in ignore_boxes:\n        if x>=0 and y>=0:grid[y:y+h,x:x+w]=floor\n    current=list(nominal_box)\n    for _ in range(64):\n        candidate=list(current);candidate[axis]+=sign*stride\n        x,y,w,h=candidate\n        if not (0<=x and 0<=y and x+w<=grid.shape[1] and y+h<=grid.shape[0]):break\n        # A visible majority of the empirically fitted floor is the proposal\'s\n        # travel criterion. Objects smaller than the body need not block it.\n        if np.count_nonzero(grid[y:y+h,x:x+w]==floor)*2<=w*h:break\n        current=candidate\n    return tuple(current) if tuple(current)!=tuple(nominal_box) else None\n\n\ndef fit_slide(pixels,old_box,nominal_box,actual_box,stride):\n    residual=actual_box[0]-nominal_box[0],actual_box[1]-nominal_box[1]\n    axes=[i for i,d in enumerate(residual) if d]\n    if len(axes)!=1 or stride<=0:return None\n    axis=axes[0];sign=1 if residual[axis]>0 else -1\n    if abs(residual[axis])<stride*2:return None\n    edge=strip(pixels,nominal_box,axis,sign)\n    if edge.size==0 or not np.all(edge==edge[0]):return None\n    x,y,w,h=nominal_box\n    patch=pixels[y:y+h,x:x+w]\n    if patch.shape!=(h,w):return None\n    floor=int(np.bincount(patch.ravel(),minlength=16).argmax())\n    boundary=int(edge[0])\n    if floor==boundary:return None\n    rule=boundary,floor,axis,sign,stride\n    return rule if slide_destination(pixels,nominal_box,rule,(old_box,))==actual_box else None\n\n\nclass BoundaryMotionDynamics(GoalNavigationDynamics):\n    def __init__(self,seed=0):\n        self.slide_stats=Counter();self.slide_events=[]\n        super().__init__(seed)\n\n    def _clear_local(self):\n        super()._clear_local()\n        self.slide_rules=Counter();self.slide_sources=defaultdict(list)\n        self.slide_pending=None\n\n    def _slide(self,scene,key,obs,predicted):\n        if not self.nav_active or not self.navigation_cohort or predicted is None:return None\n        old=body_box(scene,self.navigation_cohort);nominal=body_box(predicted,self.navigation_cohort)\n        if old is None or nominal is None:return None\n        delta=nominal[0]-old[0],nominal[1]-old[1]\n        stride=max(abs(x) for x in delta)\n        if not stride or sum(bool(x) for x in delta)!=1:return None\n        observed=body_box(self.observed_scene,self.navigation_cohort) if self.observed_scene else None\n        proposals=[]\n        for rule,support in self.slide_rules.items():\n            if support<2 or rule[-1]!=stride:continue\n            destination=slide_destination(obs.pixels,nominal,rule,tuple(b for b in (old,observed) if b))\n            if destination is not None:proposals.append((destination,rule))\n        if len({p[0] for p in proposals})!=1:return None\n        destination,rule=proposals[0]\n        return destination,rule,old\n\n    def _predict(self,scene,key,obs):\n        predicted=super()._predict(scene,key,obs)\n        proposal=self._slide(scene,key,obs,predicted)\n        if proposal is None:return predicted\n        destination,_,old=proposal\n        from context_navigation import translate\n        return translate(scene,self.navigation_cohort,(destination[0]-old[0],destination[1]-old[1]))\n\n    def choose(self,obs,objects,candidates):\n        action,reason=super().choose(obs,objects,candidates)\n        if self.nav_active and self.pending:\n            # Query the parent prediction to avoid applying the proposed slide twice.\n            scene=self.pending[\'scene\'];key=self.pending[\'key\']\n            nominal=super()._predict(scene,key,obs)\n            proposal=self._slide(scene,key,obs,nominal)\n            if proposal:\n                destination,rule,old=proposal\n                self.slide_pending={\'action_index\':self.stats[\'observations\']+2,\n                    \'before\':obs.key(),\'action\':action.id,\'rule\':rule,\n                    \'predicted_body\':destination,\'source_action_indices\':list(self.slide_sources[rule]),\n                    \'movers\':self.navigation_cohort}\n                reason+=\'; test boundary-motion model fitted from earlier public transitions\'\n        return action,reason\n\n    def observe(self,before,action,after,old_objects,new_objects):\n        old_scene,new_scene=entities(old_objects,before.pixels.shape),entities(new_objects,after.pixels.shape)\n        usable=(before.levels==self.intervention_level and after.levels==before.levels\n                and action.id in (1,2,3,4) and after.state not in (\'GAME_OVER\',\'WIN\'))\n        budget=self._budget(before) if usable else None\n        usable=usable and budget is not None and budget[1]>0\n        cohort=self._cohort(old_scene,[(a,) for a in before.available]) if usable else None\n        fitted=None\n        if cohort:\n            old,new=body_box(old_scene,cohort[2]),body_box(new_scene,cohort[2])\n            vectors=self._measured_vectors((action.id,),old_scene)\n            nominal={vectors[s] for s in cohort[2] if s in vectors}\n            if old and new and len(nominal)==1:\n                dx,dy=next(iter(nominal));forecast=(old[0]+dx,old[1]+dy,*old[2:])\n                fitted=fit_slide(before.pixels,old,forecast,new,max(abs(dx),abs(dy)))\n        pending,self.slide_pending=self.slide_pending,None\n        if pending:\n            actual=body_box(new_scene,pending.pop(\'movers\'))\n            pending.update(after=after.key(),actual_body=actual,matched=actual==pending[\'predicted_body\'],\n                           levels_gained=after.levels-before.levels)\n            self.slide_events.append(dict(kind=\'boundary_prediction\',**pending))\n        super().observe(before,action,after,old_objects,new_objects)\n        if fitted:\n            self.slide_rules[fitted]+=1\n            self.slide_sources[fitted].append(self.stats[\'observations\']+1)\n            self.slide_stats[\'exact_public_transition_fits\']+=1\n            self.slide_events.append({\'kind\':\'boundary_fit\',\'action_index\':self.stats[\'observations\']+1,\n                \'before\':before.key(),\'after\':after.key(),\'rule\':fitted,\'support\':self.slide_rules[fitted]})\n\n    def summary(self):\n        r=super().summary();r[\'boundary_motion\']={\'stats\':dict(self.slide_stats),\'events\':self.slide_events,\n            \'rules\':[{\'rule\':k,\'support\':n,\'sources\':self.slide_sources[k]} for k,n in self.slide_rules.items()],\n            \'scope\':\'Empirically fitted visible-strip and clear-floor-run hypothesis, applied only after goal shape+ink match; no fixed tail\'}\n        return r\n', 'context_navigation.py': '"""Test local exceptions to an otherwise successful movement prediction.\n\nThe navigation key deliberately omits distant pixels, target readings and the\nresource display. It retains position, actor geometry and a two-pixel terrain\nfringe. This is a falsifiable projection, not a certified state equivalence or\nerasure of the agent\'s other memories. Conflicting outcomes remain unknown.\nOnly public before/after observations supply transitions; no game identifiers,\nlevel numbers, action budgets or game implementation enter the policy.\n"""\nfrom collections import Counter, defaultdict\nimport hashlib\n\nfrom scope_extension import ScopeExtensionDynamics\nfrom shared_dynamics import entities, unique_entities\n\n\ndef navigation_view(scene, pixels, movers, observed_scene=None):\n    """A local pose and raw terrain fringe, or None when it is unobservable.\n\n    When simulating a pose, the real actor can occlude its terrain fringe. Such\n    a query stays unknown instead of treating the occlusion as free terrain.\n    """\n    current = unique_entities(scene)\n    movers = frozenset(movers or ())\n    if not movers or any(sig not in current for sig in movers):\n        return None\n    anchor = max(movers, key=lambda sig: (sig[1]*sig[2], repr(sig)))\n    ax, ay = current[anchor]\n    geometry = tuple(sorted((sig, current[sig][0]-ax, current[sig][1]-ay)\n                            for sig in movers))\n    left = min(current[sig][0] for sig in movers)\n    top = min(current[sig][1] for sig in movers)\n    right = max(current[sig][0]+sig[1] for sig in movers)\n    bottom = max(current[sig][1]+sig[2] for sig in movers)\n    observed = unique_entities(observed_scene if observed_scene is not None else scene)\n    h, w = pixels.shape\n    fringe = []\n    for y in range(top-2, bottom+2):\n        for x in range(left-2, right+2):\n            if left <= x < right and top <= y < bottom:\n                continue\n            if not (0 <= x < w and 0 <= y < h):\n                fringe.append(-1)\n                continue\n            if any(sig in observed and observed[sig][0] <= x < observed[sig][0]+sig[1]\n                   and observed[sig][1] <= y < observed[sig][1]+sig[2]\n                   for sig in movers):\n                return None\n            fringe.append(int(pixels[y, x]))\n    return (h, w), anchor, (ax, ay), geometry, tuple(fringe)\n\n\ndef rigid_delta(before_scene, after_scene, movers):\n    before, after = unique_entities(before_scene), unique_entities(after_scene)\n    if not movers or any(sig not in before or sig not in after for sig in movers):\n        return None\n    deltas = {(after[sig][0]-before[sig][0], after[sig][1]-before[sig][1])\n              for sig in movers}\n    return next(iter(deltas)) if len(deltas) == 1 else None\n\n\ndef translate(scene, movers, delta):\n    dx, dy = delta\n    return tuple(sorted((sig, x+(dx if sig in movers else 0),\n                         y+(dy if sig in movers else 0)) for sig, x, y in scene))\n\n\nclass ContextNavigationDynamics(ScopeExtensionDynamics):\n    def __init__(self, seed=0, mode=\'edges\'):\n        if mode not in (\'edges\', \'veto\'):\n            raise ValueError(mode)\n        self.navigation_mode = mode\n        self.navigation_stats = Counter()\n        self.navigation_events = []\n        super().__init__(seed, adaptive=False)\n\n    def _clear_local(self):\n        super()._clear_local()\n        self.navigation_evidence = defaultdict(Counter)\n        self.navigation_sources = defaultdict(list)\n        self.navigation_exceptions = set()\n        self.navigation_pending = None\n        self.current_observed_scene = None\n        self.navigation_view_cache = None\n\n    def _view(self, scene, obs, movers):\n        if self.navigation_view_cache is None:\n            return navigation_view(scene, obs.pixels, movers, self.current_observed_scene)\n        key = scene, frozenset(movers or ())\n        if key not in self.navigation_view_cache:\n            self.navigation_view_cache[key] = navigation_view(scene, obs.pixels, movers, self.current_observed_scene)\n        return self.navigation_view_cache[key]\n\n    def _lookup(self, scene, key, obs, movers):\n        if not self.scope_active or key[0] not in (1, 2, 3, 4):\n            return None, None\n        view = self._view(scene, obs, movers)\n        if view is None:\n            return None, None\n        condition = view, key\n        if condition not in self.navigation_exceptions:\n            return None, None\n        outcomes = self.navigation_evidence[condition]\n        if len(outcomes) != 1:\n            return condition, None\n        return condition, next(iter(outcomes))\n\n    def _predict(self, scene, key, obs):\n        movers = self.navigation_cohort\n        condition, delta = self._lookup(scene, key, obs, movers)\n        if condition is None:\n            return super()._predict(scene, key, obs)\n        if delta is None or self.navigation_mode == \'veto\' or delta == (0, 0):\n            return None\n        # Transport only the measured rigid actor. Contact/reading models still\n        # decide what a newly touched operator might do to shape and ink.\n        return translate(scene, movers, delta)\n\n    def choose(self, obs, objects, candidates):\n        self._ensure_level(obs.levels)\n        self.current_observed_scene = entities(objects, obs.pixels.shape)\n        self.navigation_view_cache = {}\n        try:\n            return self._choose_with_context(obs, objects, candidates)\n        finally:\n            self.navigation_view_cache = None\n\n    def _choose_with_context(self, obs, objects, candidates):\n        action, reason = super().choose(obs, objects, candidates)\n        if self.pending and self.scope_active:\n            condition, delta = self._lookup(self.pending[\'scene\'], self.pending[\'key\'],\n                                             obs, self.navigation_cohort)\n            if condition is not None:\n                self.navigation_pending = {\n                    \'action_index\': self.stats[\'observations\']+2,\n                    \'observation\': obs.key(), \'action\': {\'id\': action.id, **action.data()},\n                    \'context_digest\': hashlib.sha256(repr(condition).encode()).hexdigest(),\n                    \'source_action_indices\': list(self.navigation_sources[condition]),\n                    \'predicted_delta\': delta,\n                    \'prediction_used_for_motion\': self.navigation_mode == \'edges\' and delta is not None,\n                    \'movers\': tuple(sorted(self.navigation_cohort)),\n                }\n                reason += \'; local motion exception / retained position and terrain fringe\'\n        return action, reason\n\n    def observe(self, before, action, after, old_objects, new_objects):\n        pending, self.navigation_pending = self.navigation_pending, None\n        old_scene = entities(old_objects, before.pixels.shape)\n        new_scene = entities(new_objects, after.pixels.shape)\n        usable = (self.scope_active and action.id in (1, 2, 3, 4)\n                  and before.levels == after.levels\n                  and before.state not in (\'NOT_PLAYED\', \'WIN\', \'GAME_OVER\')\n                  and after.state not in (\'WIN\', \'GAME_OVER\'))\n        cohort = self._cohort(old_scene, [(a,) for a in before.available]) if usable else None\n        record = None\n        if cohort is not None:\n            movers = cohort[2]\n            view = navigation_view(old_scene, before.pixels, movers)\n            actual = rigid_delta(old_scene, new_scene, movers)\n            vectors = self._measured_vectors((action.id,), old_scene)\n            expected = {vectors[sig] for sig in movers if sig in vectors}\n            nominal = next(iter(expected)) if len(expected) == 1 and all(sig in vectors for sig in movers) else None\n            if view is not None and actual is not None and nominal is not None:\n                record = (view, (action.id,)), actual, nominal\n        # Evaluate the frozen prospective prediction before new evidence enters.\n        if pending is not None:\n            actual = rigid_delta(old_scene, new_scene, pending.pop(\'movers\')) if usable else None\n            checked = pending[\'predicted_delta\'] is not None and actual is not None\n            pending.update(actual_delta=actual, observation_after=after.key(),\n                           checked=checked,\n                           matched=(pending[\'predicted_delta\'] == actual) if checked else None)\n            self.navigation_events.append(dict(kind=\'local_prediction\', **pending))\n            self.navigation_stats[\'selected_local_predictions\'] += 1\n            self.navigation_stats[\'local_predictions_checked\'] += checked\n            self.navigation_stats[\'local_prediction_hits\'] += checked and pending[\'predicted_delta\'] == actual\n            self.navigation_stats[\'local_prediction_failures\'] += checked and pending[\'predicted_delta\'] != actual\n        super().observe(before, action, after, old_objects, new_objects)\n        if record is not None:\n            condition, actual, nominal = record\n            outcomes = self.navigation_evidence[condition]\n            conflict = bool(outcomes) and actual not in outcomes\n            new_exception = condition not in self.navigation_exceptions and actual != nominal\n            outcomes[actual] += 1\n            step = self.stats[\'observations\']+1\n            self.navigation_sources[condition].append(step)\n            self.navigation_stats[\'physical_transition_samples\'] += 1\n            if actual != nominal:\n                self.navigation_exceptions.add(condition)\n                self.navigation_stats[\'departures_from_global_motion\'] += 1\n            if new_exception or conflict:\n                self.navigation_events.append({\n                    \'kind\': \'local_exception\' if new_exception else \'local_conflict\',\n                    \'action_index\': step, \'observation\': before.key(),\n                    \'observation_after\': after.key(), \'action\': {\'id\': action.id, **action.data()},\n                    \'context_digest\': hashlib.sha256(repr(condition).encode()).hexdigest(),\n                    \'nominal_delta\': nominal, \'actual_delta\': actual,\n                    \'distinct_outcomes\': len(outcomes), \'source_action_indices\': list(self.navigation_sources[condition]),\n                })\n            self.navigation_stats[\'conflicting_contexts_observed\'] += conflict\n\n    def summary(self):\n        result = super().summary()\n        result[\'context_navigation\'] = {\n            \'mode\': self.navigation_mode,\n            \'projection\': \'Position, rigid actor geometry, two-pixel terrain fringe; distant pixels and resource/target displays omitted\',\n            \'scope\': \'Empirical local-motion hypothesis; conflicts veto reuse; no calibrated posterior or full memory erasure\',\n            \'stats\': dict(sorted(self.navigation_stats.items())),\n            \'events\': self.navigation_events,\n            \'current_contexts\': len(self.navigation_evidence),\n            \'current_exceptions\': len(self.navigation_exceptions),\n            \'current_conflicts\': sum(len(v) > 1 for v in self.navigation_evidence.values()),\n        }\n        return result\n', 'scope_extension.py': '"""Extend a successful predictor\'s domain, then check its new predictions.\n\nThe gate is empirical, not a calibrated Bayesian confidence. A single pairing\nrestriction is relaxed: membership of the same rotation/reflection orbit.\nPixels, ink, frame geometry and observed failures remain available.\n"""\nfrom collections import Counter, deque\nimport numpy as np\n\nfrom association_dynamics import framed_patterns, fit_transforms, transform_pattern\nfrom reconsideration import ReconsiderationDynamics, geometric_pairs, contact_landmarks\nfrom shared_dynamics import entities, unique_entities\n\n\ndef structural_pairs(panels, shape):\n    h, w = shape\n    outer = [p for p in panels if min(p.x, p.y, w-p.x-p.width, h-p.y-p.height) <= 2]\n    inner = [p for p in panels if p not in outer]\n    return [(a,b) for a in outer for b in inner\n            if a.sig[0] == b.sig[0] and a.pattern[0] == b.pattern[0]]\n\n\ndef bitmap(pattern):\n    shape, packed = pattern\n    return np.unpackbits(np.frombuffer(bytes.fromhex(packed), dtype=np.uint8))[:int(np.prod(shape))].reshape(shape)\n\n\ndef encoded(bits):\n    return bits.shape, np.packbits(bits).tobytes().hex()\n\n\ndef affine_shape_rules(samples):\n    """Fit eight coordinate symmetries plus a fixed, inferred XOR mask.\n\n    Masks are fitted only from public before/after samples. Every candidate has\n    to fit all samples. Ambiguous predictions on a new input stay unknown.\n    """\n    pairs = [(s[0],s[2]) for s in samples]\n    if not pairs:\n        return []\n    result = []\n    for transform in range(8):\n        first = transform_pattern(pairs[0][0], transform)\n        if first[0] != pairs[0][1][0]:\n            continue\n        mask = bitmap(first) ^ bitmap(pairs[0][1])\n        if all(transform_pattern(a, transform)[0] == b[0] == mask.shape\n               and np.array_equal(bitmap(transform_pattern(a, transform)) ^ mask, bitmap(b))\n               for a,b in pairs):\n            result.append((transform, encoded(mask)))\n    return result\n\n\ndef context_core(sig, passable):\n    """A candidate retrieval key, never a replacement for the raw patch.\n\n    Remove one uniform boundary only if that color has measured traversals,\n    and retain an interior at least 3 by 3. Rotation, reflection and ink are\n    unchanged. A supply\'s colored ring is therefore not collapsed to a dot.\n    """\n    if sig[0] != -1 or min(sig[1:3]) < 5:\n        return sig\n    patch = np.frombuffer(bytes.fromhex(sig[3]),dtype=np.uint8).reshape(sig[2],sig[1])\n    border = np.concatenate((patch[0],patch[-1],patch[:,0],patch[:,-1]))\n    color = int(border[0])\n    if not passable.get(color,0) or not np.all(border == color):\n        return sig\n    core = patch[1:-1,1:-1]\n    return -1,core.shape[1],core.shape[0],core.tobytes().hex()\n\n\nclass ScopeExtensionDynamics(ReconsiderationDynamics):\n    def __init__(self, seed=0, adaptive=True, minimum_hits=12):\n        self.adaptive_scope = adaptive\n        self.minimum_hits = minimum_hits\n        self.scope_active = False\n        self.hit_streak = 0\n        self.scope_stats = Counter()\n        self.scope_events = []\n        self.recent_successes = deque(maxlen=12)\n        self.effect_cache = {}\n        self.sample_cache = {}\n        self.planning_cache = None\n        self.scope_pending = None\n        super().__init__(seed)\n\n    def _clear_local(self):\n        super()._clear_local()\n        self.scope_pending = None\n        self.frontier_trials = Counter()\n        self.frontier_goal = None\n        self.frontier_started = 0\n\n    def _pair(self, panels, shape):\n        if not self.scope_active:\n            return super()._pair(panels, shape)\n        pairs = structural_pairs(panels, shape)\n        return pairs[0] if len(pairs) == 1 else None\n\n    def _current_pair(self, obs, objects):\n        if self.scope_active:\n            pairs = structural_pairs(framed_patterns(obs.pixels, objects), obs.pixels.shape)\n            if len(pairs) > 1:\n                return None\n        return super()._current_pair(obs, objects)\n\n    def review(self, obs, objects):\n        super().review(obs, objects)\n        if self.scope_active:\n            return\n        panels = framed_patterns(obs.pixels, objects)\n        old_pairs = geometric_pairs(panels, obs.pixels.shape)\n        broad_pairs = structural_pairs(panels, obs.pixels.shape)\n        if not old_pairs and len(broad_pairs) == 1:\n            self.scope_stats[\'uncovered_structural_pair_observations\'] += 1\n            if self.hit_streak >= self.minimum_hits:\n                a,b = broad_pairs[0]\n                self.scope_active = self.review_active = True\n                self.goal = self.repair = self.reading_pending = self.projected_pending = None\n                self.rejected_goals.clear()\n                self.effect_cache.clear()\n                self.sample_cache.clear()\n                self.scope_events.append({\'kind\':\'scope_extension\', \'observation\':obs.key(),\n                    \'level\':obs.levels, \'consecutive_checked_hits\':self.hit_streak,\n                    \'supporting_predictions\':list(self.recent_successes),\n                    \'relaxed_condition\':\'same rotation/reflection orbit\',\n                    \'retained_conditions\':[\'unique peripheral/interior frame pairing\',\'frame color\',\'bitmap dimensions\',\'ink and full pixels\'],\n                    \'reference\':(a.pattern,a.ink),\'target\':(b.pattern,b.ink),\n                    \'external_actions_for_reinterpretation\':0})\n                self.scope_stats[\'extensions\'] += 1\n\n    def _sample_sources(self, sig):\n        if not self.scope_active or not self.adaptive_scope:\n            return [sig] if self.effect_samples.get(sig) else []\n        core = context_core(sig,self.passable)\n        return [s for s,v in self.effect_samples.items() if v and context_core(s,self.passable) == core]\n\n    def _samples(self, sig):\n        if sig not in self.sample_cache:\n            combined = Counter()\n            for source in self._sample_sources(sig):\n                combined.update(self.effect_samples[source])\n            self.sample_cache[sig] = combined\n        return self.sample_cache[sig]\n\n    def _effect(self, landmark, reading):\n        if not self.scope_active:\n            return super()._effect(landmark, reading)\n        key = landmark.sig, reading\n        if key in self.effect_cache:\n            return self.effect_cache[key]\n        result = super()._effect(landmark, reading)\n        if result is None and self.adaptive_scope:\n            samples = self._samples(landmark.sig)\n            exact = {(s[2],s[3]) for s in samples if (s[0],s[1]) == reading}\n            if exact:\n                result = next(iter(exact)) if len(exact) == 1 else None\n                self.effect_cache[key] = result\n                return result\n            transforms = set(range(8))\n            for sample in samples:\n                transforms &= fit_transforms(sample[0],sample[2])\n            # Reuse an observed shape transition across ink only when the\n            # recorded shape outcomes agree. This is a checked factorization\n            # hypothesis; conflicting observations keep the result unknown.\n            predictions = {s[2] for s in samples if s[0] == reading[0]}\n            if not predictions:\n                if len(samples) >= 2 and all(s[0] == s[2] for s in samples):\n                    # Minimal-change extrapolation: an operator repeatedly\n                    # observed to leave shape unchanged may do so on a new\n                    # shape. This is falsifiable, not identification of every\n                    # symmetry compatible with the old examples.\n                    predictions = {reading[0]}\n                elif transforms and samples:\n                    predictions = {transform_pattern(reading[0],g) for g in transforms}\n                else:\n                    rules = affine_shape_rules(samples)\n                    predictions = {encoded(bitmap(transform_pattern(reading[0],g)) ^ bitmap(mask))\n                                   for g,mask in rules if transform_pattern(reading[0],g)[0] == mask[0]}\n            colors = {s[3] for s in samples if s[1] == reading[1]}\n            if samples and all(s[1] == s[3] for s in samples):\n                colors.add(reading[1])\n            if len(predictions) == len(colors) == 1:\n                result = next(iter(predictions)), next(iter(colors))\n        self.effect_cache[key] = result\n        return result\n\n    def _reading_route(self, scene, controls, obs, pair, cohort, landmarks, visible):\n        if self.scope_active and self.adaptive_scope:\n            # A currently visible patch and its remembered padded rendering\n            # describe one contact hypothesis at the same center, not two\n            # sequential effects. Keep the visible raw context in this plan.\n            unique = {}\n            for landmark in landmarks:\n                key=(2*landmark.x+landmark.sig[1],2*landmark.y+landmark.sig[2],\n                     context_core(landmark.sig,self.passable))\n                unique.setdefault(key,landmark)\n            landmarks=list(unique.values())\n        if self.scope_active and bitmap(pair[0].pattern).sum() != bitmap(pair[1].pattern).sum():\n            # This only rules out a path in the current empirical catalogue.\n            # Unknown contacts are still probed; the environment is not declared\n            # unsolvable. A supported non-symmetry sample disables the shortcut.\n            all_symmetric = True\n            for landmark in landmarks:\n                samples = self._samples(landmark.sig)\n                possible = set(range(8))\n                for sample in samples:\n                    possible &= fit_transforms(sample[0],sample[2])\n                if not possible:\n                    all_symmetric = False\n                    break\n            if all_symmetric:\n                self.scope_stats[\'current_catalogue_count_invariant_blocks_plan\'] += 1\n                return None\n        if self.scope_active:\n            # Over-approximate spatial paths by freely composing known contact\n            # effects. Exhaustion proves only that this catalogue cannot yet\n            # produce the target reading, so spend effort on a new experiment.\n            # Hitting the cap is inconclusive and falls through to path search.\n            initial = pair[0].pattern,pair[0].ink\n            target = pair[1].pattern,pair[1].ink\n            seen,queue = {initial},deque([initial])\n            while queue and target not in seen and len(seen) < 256:\n                reading = queue.popleft()\n                for landmark in landmarks:\n                    result = self._effect(landmark,reading)\n                    if result is not None and result not in seen:\n                        seen.add(result)\n                        queue.append(result)\n            if not queue and target not in seen:\n                self.scope_stats[\'current_catalogue_reading_unreachable\'] += 1\n                return None\n        return super()._reading_route(scene,controls,obs,pair,cohort,landmarks,visible)\n\n    def _plan(self, scene, controls, obs):\n        committed = self.frontier_goal\n        path = super()._plan(scene, controls, obs)\n        if not self.scope_active or not self.adaptive_scope or self.reading_pending is not None:\n            return path\n        pair = self._current_pair(obs, self.contact_objects)\n        cohort = self._cohort(scene, controls)\n        budget = self._budget(obs)\n        if pair is None or cohort is None or budget is None:\n            return path\n        reference, target = pair\n        reading = reference.pattern, reference.ink\n        anchor, offset, movers = cohort\n        if committed is not None and committed.distance(scene) != 0:\n            route = self._contact_route(scene,controls,obs,committed)\n            if (route and len(route)+3 < budget[1]\n                    and self.stats[\'observations\']-self.frontier_started < 64):\n                self.goal,self.joint_mode = committed,False\n                self.scope_stats[\'committed_frontier_steps\'] += 1\n                return route\n        self.frontier_goal = None\n        options = []\n        for landmark in contact_landmarks(obs.pixels, self.contact_objects, pair, movers):\n            condition = reading, landmark.sig, landmark.x, landmark.y\n            if self.frontier_trials[condition]:\n                continue\n            samples = self._samples(landmark.sig)\n            if any((s[0],s[1]) == reading for s in samples):\n                continue\n            goal = landmark.goal(anchor, offset)\n            route = self._contact_route(scene, controls, obs, goal)\n            if not route or len(route)+3 >= budget[1]:\n                continue\n            # Test an acquired effect on an unseen input first. Unknown\n            # operators remain a separate class of exploratory experiment.\n            effect = self._effect(landmark, reading)\n            if effect == reading:\n                # Do not consume a resource merely to confirm an unchanged\n                # reading. The parent fuel-aware planner can still choose it.\n                continue\n            priority = 0 if effect is not None else 1 if samples else 2\n            if reading[0] == target.pattern and reading[1] != target.ink:\n                # Preserve an attained shape while investigating the remaining\n                # color condition. A color hint is a hypothesis, never success.\n                if reference.ink in landmark.colors and target.ink in landmark.colors:\n                    priority -= 4\n                elif effect is None or effect[0] != reading[0]:\n                    priority += 4\n            if effect is not None and effect[0] == target.pattern and effect[1] == target.ink:\n                priority -= 8\n            options.append((priority,len(route),repr(goal),goal,route))\n        if options:\n            _,_,_,self.goal,path = min(options)\n            self.frontier_goal = self.goal\n            self.frontier_started = self.stats[\'observations\']\n            self.joint_mode = False\n            self.scope_stats[\'frontier_routes\'] += 1\n        return path\n\n    def _predict(self, scene, key, obs):\n        if self.planning_cache is None or not self.scope_active:\n            return super()._predict(scene,key,obs)\n        cohort = None if self.navigation_cohort is None else frozenset(self.navigation_cohort)\n        cache_key = scene,key,cohort\n        if cache_key not in self.planning_cache:\n            self.planning_cache[cache_key] = super()._predict(scene,key,obs)\n        return self.planning_cache[cache_key]\n\n    def choose(self, obs, objects, candidates):\n        # Predictions are pure while one observation is being planned. Never\n        # reuse this cache after a new action, observation or model update.\n        self.planning_cache = {}\n        self.effect_cache.clear()\n        self.sample_cache.clear()\n        try:\n            action, reason = super().choose(obs, objects, candidates)\n        finally:\n            self.planning_cache = None\n        if not self.scope_active:\n            return action, reason\n        self.scope_stats[\'expanded_scope_decisions\'] += 1\n        scene = entities(objects, obs.pixels.shape)\n        pair = self._current_pair(obs, objects)\n        cohort = self._cohort(scene, [(a,) for a in obs.available])\n        predicted_scene = self.pending.get(\'predicted\') if self.pending else None\n        prediction, contacts = None, []\n        reading = (pair[0].pattern,pair[0].ink) if pair else None\n        if pair and cohort and predicted_scene is not None:\n            anchor,offset,movers = cohort\n            contacts = [l for l in contact_landmarks(obs.pixels,objects,pair,movers)\n                        if l.goal(anchor,offset).distance(scene) != 0\n                        and l.goal(anchor,offset).distance(predicted_scene) == 0]\n            if not contacts:\n                prediction = reading\n            elif len(contacts) == 1:\n                prediction = self._effect(contacts[0],reading)\n        self.scope_pending = {\'observation\':obs.key(),\'level\':obs.levels,\n            \'action_index\':self.stats[\'observations\']+2,\'action\':{\'id\':action.id,**action.data()},\n            \'input_reading\':reading,\'prediction\':prediction,\n            \'predicted_contacts\':[l.sig for l in contacts],\n            \'effect_sample_sources\':[self._sample_sources(l.sig) for l in contacts],\n            \'prediction_kind\':\'contact\' if contacts else \'transport\',\n            \'status\':\'pending\'}\n        return action, reason\n\n    def _learn_contact(self, before, after, old_objects, new_objects, cohort):\n        if not self.scope_active:\n            return super()._learn_contact(before,after,old_objects,new_objects,cohort)\n        if (before.levels != after.levels or before.state in (\'NOT_PLAYED\',\'WIN\',\'GAME_OVER\')\n                or after.state in (\'WIN\',\'GAME_OVER\') or cohort is None):\n            return\n        old = structural_pairs(framed_patterns(before.pixels,old_objects),before.pixels.shape)\n        new = structural_pairs(framed_patterns(after.pixels,new_objects),after.pixels.shape)\n        if len(old) != 1 or len(new) != 1:\n            return\n        old_pair, new_pair = old[0],new[0]\n        reading = old_pair[0].pattern,old_pair[0].ink\n        outcome = new_pair[0].pattern,new_pair[0].ink\n        anchor,offset,movers = cohort\n        old_scene = entities(old_objects,before.pixels.shape)\n        new_scene = entities(new_objects,after.pixels.shape)\n        touched = [l for l in contact_landmarks(before.pixels,old_objects,old_pair,movers)\n                   if l.goal(anchor,offset).distance(old_scene) != 0\n                   and l.goal(anchor,offset).distance(new_scene) == 0]\n        if len(touched) != 1:\n            self.scope_stats[\'ambiguous_contacts_not_attributed\'] += len(touched) > 1\n            return\n        landmark = touched[0]\n        predicted = self._effect(landmark,reading)\n        sources = self._sample_sources(landmark.sig)\n        sample = (*reading,*outcome)\n        self.effect_samples[landmark.sig][sample] += 1\n        self.effect_cache.clear()\n        self.sample_cache.clear()\n        self.landmark_trials[reading,landmark.sig,landmark.x,landmark.y] += 1\n        self.frontier_trials[reading,landmark.sig,landmark.x,landmark.y] += 1\n        symmetric = bool(fit_transforms(reading[0],outcome[0]))\n        rules = affine_shape_rules(self.effect_samples[landmark.sig]) if self.adaptive_scope else []\n        self.scope_events.append({\'kind\':\'contact_trial\',\'level\':before.levels,\n            \'observation\':before.key(),\'observation_after\':after.key(),\'landmark\':landmark.sig,\n            \'effect_sample_sources\':sources,\n            \'input_reading\':reading,\'prediction_before_learning\':predicted,\'actual_reading\':outcome,\n            \'prediction_matches\':None if predicted is None else predicted == outcome,\n            \'same_symmetry_orbit\':symmetric,\'remaining_affine_candidates\':len(rules),\n            \'has_inferred_nonzero_mask\':any(np.any(bitmap(mask)) for _,mask in rules)})\n        self.scope_stats[\'contact_trials\'] += 1\n        self.scope_stats[\'contact_predictions_available\'] += predicted is not None\n        self.scope_stats[\'contact_prediction_failures\'] += predicted is not None and predicted != outcome\n        self.scope_stats[\'observed_non_symmetry_effects\'] += not symmetric\n\n    def observe(self, before, action, after, old_objects, new_objects):\n        pending, self.scope_pending = self.scope_pending, None\n        old_prediction = self.reading_pending\n        was_active = self.scope_active\n        # The parent evaluator has a narrower pair detector. Expanded predictions\n        # are checked below against the preserved structural correspondence.\n        if was_active:\n            self.reading_pending = None\n        super().observe(before,action,after,old_objects,new_objects)\n        usable = before.levels == after.levels and action.id != 0 and after.state not in (\'WIN\',\'GAME_OVER\')\n        pairs = ((structural_pairs if was_active else geometric_pairs)(\n            framed_patterns(after.pixels,new_objects),after.pixels.shape)) if usable else []\n        actual = (pairs[0][0].pattern,pairs[0][0].ink) if len(pairs) == 1 else None\n        if not was_active and old_prediction is not None and actual is not None:\n            matched = old_prediction == actual\n            self.hit_streak = self.hit_streak+1 if matched else 0\n            self.scope_stats[\'prior_predictions_checked\'] += 1\n            self.scope_stats[\'prior_prediction_hits\'] += matched\n            if matched:\n                self.recent_successes.append({\'before\':before.key(),\'after\':after.key(),\'prediction\':old_prediction})\n            else:\n                self.recent_successes.clear()\n        if pending is not None:\n            checkable = pending[\'prediction\'] is not None and actual is not None\n            pending.update(status=\'observed\',observation_after=after.key(),actual_reading=actual,\n                           checked=checkable,matched=pending[\'prediction\']==actual if checkable else None,\n                           levels_gained=after.levels-before.levels,state_after=after.state)\n            self.scope_events.append(dict(kind=\'expanded_prediction\',**pending))\n            self.scope_stats[\'expanded_predictions_checked\'] += checkable\n            self.scope_stats[\'expanded_prediction_hits\'] += checkable and pending[\'prediction\']==actual\n            self.scope_stats[\'expanded_prediction_failures\'] += checkable and pending[\'prediction\']!=actual\n            self.scope_stats[\'expanded_predictions_unscored\'] += not checkable\n        if action.id == 0:\n            self.hit_streak = 0\n            self.recent_successes.clear()\n\n    def summary(self):\n        result = super().summary()\n        result[\'scope_extension\'] = {\'adaptive\':self.adaptive_scope,\'minimum_checked_hits\':self.minimum_hits,\n            \'active\':self.scope_active,\'prior_checked_hit_streak\':self.hit_streak,\n            \'stats\':dict(sorted(self.scope_stats.items())),\'events\':self.scope_events,\n            \'scope\':\'Empirical prediction-domain experiments; no calibrated Bayesian confidence or certified state quotient\'}\n        return result\n', 'context_contact.py': '"""Public contact experiments and composition after the first non-D4 effect.\n\nThe initial policy is unchanged until a non-symmetry reading change is actually\nobserved. Exact acquired transitions take precedence; unknown inputs stay unknown.\n"""\nfrom collections import Counter, defaultdict, deque\nfrom dataclasses import replace\nimport numpy as np\n\nfrom unknown_contact import UnknownContactDynamics\nfrom association_dynamics import ContactGoal, framed_patterns, fit_transforms, transform_pattern\nfrom reconsideration import contact_landmarks, Landmark\nfrom shared_dynamics import entities, unique_entities\nfrom public_observer import MotionObserver, patches, unique_by_key, center\n\n\ndef canonical_sig(sig):\n    if sig[0] != -1:\n        return sig\n    a = np.frombuffer(bytes.fromhex(sig[3]), dtype=np.uint8).reshape(sig[2], sig[1])\n    if min(a.shape) >= 5:\n        edge = np.concatenate((a[0], a[-1], a[:, 0], a[:, -1]))\n        if np.all(edge == edge[0]):\n            a = a[1:-1, 1:-1]\n    return a.shape[1], a.shape[0], a.tobytes().hex()\n\n\ndef body_box(scene, movers):\n    current = unique_entities(scene)\n    if not movers or any(s not in current for s in movers):\n        return None\n    x, y = min(current[s][0] for s in movers), min(current[s][1] for s in movers)\n    right = max(current[s][0] + s[1] for s in movers)\n    bottom = max(current[s][1] + s[2] for s in movers)\n    return x, y, right - x, bottom - y\n\n\ndef aligned(a, b):\n    return all(abs(x-y) <= .5 for x, y in zip(center(a), center(b)))\n\n\nclass ContextContactDynamics(UnknownContactDynamics):\n    def __init__(self, seed=0, mode=\'coupled\', intervention_level=4):\n        self.context_active = False\n        self.contact_table = defaultdict(Counter)\n        self.non_symmetric = set()\n        self.symmetric = set()\n        self.resource_keys = set()\n        self.context_events = []\n        self.context_stats = Counter()\n        super().__init__(seed, mode, intervention_level)\n\n    def _clear_local(self):\n        super()._clear_local()\n        self.motion = MotionObserver()\n        self.reference_box = None\n        self.last_causal_landmark = None\n        self.context_choice = None\n        self.learning_action = None\n\n    def _reading_at_box(self, obs, objects, box):\n        if box is None:\n            return None\n        ps = [p for p in framed_patterns(obs.pixels, objects)\n              if (p.x, p.y, p.width, p.height) == box]\n        return (ps[0].pattern, ps[0].ink) if len(ps) == 1 else None\n\n    def _current_pair(self, obs, objects):\n        pair = super()._current_pair(obs, objects)\n        if obs.levels == self.intervention_level and pair is not None:\n            p = pair[0]\n            self.reference_box = p.x, p.y, p.width, p.height\n        return pair\n\n    def _predict_effect(self, key, reading):\n        samples = self.contact_table.get(key, {})\n        exact = {(c, d) for (a, b, c, d), n in samples.items() if n and (a, b) == reading}\n        if exact:\n            return exact, \'measured\' if len(exact) == 1 else \'conflicting\'\n        # No non-D4 extrapolation: the observed upper-operator cycle does not\n        # fit a fixed D4+XOR model. A new input explicitly remains unknown.\n        if not samples or key in self.non_symmetric:\n            return set(), \'unknown\'\n        transforms = set(range(8))\n        for (a, _, c, _), n in samples.items():\n            if n:\n                transforms &= fit_transforms(a, c)\n        colors = {d for (a, b, c, d), n in samples.items() if n and b == reading[1]}\n        if all(b == d for (_, b, _, d) in samples):\n            colors.add(reading[1])\n        possibilities = {(transform_pattern(reading[0], t), color)\n                         for t in transforms for color in colors}\n        return possibilities, \'extrapolated\' if possibilities else \'unknown\'\n\n    def _effect(self, landmark, reading):\n        if self.context_active and self.level == self.intervention_level:\n            key = canonical_sig(landmark.sig)\n            if key in self.contact_table:\n                predictions, status = self._predict_effect(key, reading)\n                return next(iter(predictions)) if len(predictions) == 1 else None\n        return super()._effect(landmark, reading)\n\n    def _learn_contact(self, before, after, old_objects, new_objects, cohort):\n        active_before = self.context_active\n        if not active_before or before.levels != self.intervention_level:\n            super()._learn_contact(before, after, old_objects, new_objects, cohort)\n        if (before.levels != self.intervention_level or before.levels != after.levels\n                or after.state in (\'WIN\', \'GAME_OVER\') or cohort is None\n                or self.learning_action is None or self.learning_action.id == 0):\n            return\n        old_pair = self._current_pair(before, old_objects)\n        reading = self._reading_at_box(before, old_objects, self.reference_box)\n        outcome = self._reading_at_box(after, new_objects, self.reference_box)\n        if reading is None or outcome is None:\n            self.context_stats[\'unreadable_reference\'] += 1\n            return\n        old_scene, new_scene = entities(old_objects, before.pixels.shape), entities(new_objects, after.pixels.shape)\n        before_box, after_box = body_box(old_scene, cohort[2]), body_box(new_scene, cohort[2])\n        if before_box is None or after_box is None:\n            return\n        before_patches, after_patches = unique_by_key(patches(before)), unique_by_key(patches(after))\n        delta = after_box[0]-before_box[0], after_box[1]-before_box[1]\n        possible = self.motion.possible_contacts(before, self.learning_action.id, before_box,\n                        delta, self.stats[\'observations\'] + 2)\n        dynamic = {tuple(p[\'patch_key\']) for p in possible}\n        touched = {}\n        for key, p in before_patches.items():\n            if p.box == before_box or aligned(before_box, p.box):\n                continue\n            q = after_patches.get(key)\n            actual_alignment = q is not None and aligned(after_box, q.box)\n            old_alignment = aligned(after_box, p.box)\n            if actual_alignment or old_alignment or key in dynamic:\n                touched[key] = p\n        if len(touched) != 1:\n            if reading != outcome:\n                self.context_events.append({\'kind\': \'unattributed_reading_change\',\n                    \'action_index\': self.stats[\'observations\'] + 2, \'before\': before.key(),\n                    \'after\': after.key(), \'input\': reading, \'output\': outcome,\n                    \'possible_patch_count\': len(touched)})\n                self.context_stats[\'unattributed_changes\'] += 1\n            return\n        key, patch = next(iter(touched.items()))\n        sample = (*reading, *outcome)\n        self.contact_table[key][sample] += 1\n        symmetric = bool(fit_transforms(reading[0], outcome[0]))\n        if reading[0] != outcome[0]:\n            (self.symmetric if symmetric else self.non_symmetric).add(key)\n        if not symmetric:\n            self.context_active = True\n        self.last_causal_landmark = Landmark(patch.raw_key, patch.raw_box[0], patch.raw_box[1],\n                        frozenset(int(c) for c in np.frombuffer(bytes.fromhex(patch.raw_key[3]), dtype=np.uint8)), ())\n        resource_before, resource_after = self._budget(before), self._budget(after)\n        if (resource_before and resource_after and resource_before[0] == resource_after[0]\n                and resource_after[1] > resource_before[1]):\n            self.resource_keys.add(key)\n        if active_before:\n            self.effect_samples[patch.raw_key][sample] += 1\n            self.effect_cache.clear()\n            self.sample_cache.clear()\n        event = {\'kind\': \'context_contact\', \'action_index\': self.stats[\'observations\'] + 2,\n            \'before\': before.key(), \'after\': after.key(), \'patch_key\': key,\n            \'raw_patch\': patch.raw_key, \'patch_box_before\': patch.box,\n            \'action\': self.learning_action.id, \'input\': reading, \'output\': outcome,\n            \'status\': \'observed_no_change\' if reading == outcome else \'observed_change\',\n            \'same_symmetry_orbit\': symmetric,\n            \'dynamic_motion_hypothesis\': key in dynamic,\n            \'target_visible_before\': old_pair is not None,\n            \'attribution_scope\': \'unique public contact candidate; not an isolated causal proof\'}\n        self.context_events.append(event)\n        self.context_stats[\'contact_samples\'] += 1\n        self.context_stats[\'dynamic_samples\'] += key in dynamic\n\n    def _catalogue(self, start, target, landmarks):\n        queue, seen = deque([(start, [])]), {start}\n        while queue and len(seen) < 128:\n            reading, route = queue.popleft()\n            if reading == target:\n                return route, \'reachable_in_current_exact_catalogue\'\n            for landmark in landmarks:\n                key = canonical_sig(landmark.sig)\n                outcomes, status = self._predict_effect(key, reading)\n                if status != \'measured\' or len(outcomes) != 1:\n                    continue\n                after = next(iter(outcomes))\n                if after not in seen:\n                    seen.add(after)\n                    queue.append((after, route+[key]))\n        return [], \'unknown_at_bound\' if queue else \'unreachable_in_current_exact_catalogue\'\n\n    def _plan(self, scene, controls, obs):\n        original = super()._plan(scene, controls, obs)\n        if not self.context_active or obs.levels != self.intervention_level:\n            return original\n        pair = self._current_pair(obs, self.contact_objects)\n        cohort, resource = self._cohort(scene, controls), self._budget(obs)\n        if pair is None or cohort is None or resource is None:\n            return original\n        reference, target = pair\n        reading, destination = (reference.pattern, reference.ink), (target.pattern, target.ink)\n        anchor, offset, movers = cohort\n        self.navigation_cohort = movers\n        visible = contact_landmarks(obs.pixels, self.contact_objects, pair, movers)\n        landmarks = list(visible)\n        box = body_box(scene, movers)\n        for remembered in [*self.remembered_landmarks.values(), self.last_causal_landmark]:\n            if (remembered is not None and canonical_sig(remembered.sig) in self.contact_table\n                    and box is not None\n                    and aligned(box, (remembered.x, remembered.y, remembered.sig[1], remembered.sig[2]))\n                    and all(canonical_sig(p.sig) != canonical_sig(remembered.sig) for p in landmarks)):\n                landmarks.append(remembered)\n        semantic, catalogue_status = self._catalogue(reading, destination, landmarks)\n        self.context_stats[catalogue_status] += 1\n        options, supplies = [], []\n        goal_shape_orbit = bool(fit_transforms(reading[0], destination[0]))\n        for landmark in landmarks:\n            key = canonical_sig(landmark.sig)\n            predictions, evidence_status = self._predict_effect(key, reading)\n            prediction = next(iter(predictions)) if len(predictions) == 1 else self._effect(landmark, reading)\n            goal = landmark.goal(anchor, offset)\n            path = self._contact_route(scene, controls, obs, goal)\n            if not path:\n                continue\n            if key in self.resource_keys or resource[0] in landmark.colors:\n                if len(path) < resource[1]:\n                    supplies.append((0 if key in self.resource_keys else 1, len(path), repr(goal), goal, path))\n            if reading == destination:\n                continue\n            if semantic:\n                rank = -10 if key == semantic[0] else 20\n                stage = \'compose_exact_acquired_effects\'\n            elif reading[0] == destination[0]:\n                # A new ink is an unmeasured input to a measured color-changing\n                # operator. Keep testing that operator instead of ranking it\n                # alongside known shape-changing contacts.\n                color_operator = any(a == c and b != d for (a,b,c,d) in self.contact_table.get(key, {}))\n                color_operator = color_operator or any(\n                    canonical_sig(sig) == key and any(a == c and b != d for a,b,c,d in samples)\n                    for sig,samples in self.effect_samples.items())\n                rank = (0 if prediction == destination else 1\n                        if prediction and prediction[0] == reading[0] and prediction[1] != reading[1]\n                        else 2 if color_operator else 20)\n                stage = \'preserve_matched_shape_and_test_color\'\n            elif goal_shape_orbit:\n                moving = any(k[0] == key and any(s[\'delta\'] != (0, 0) for s in rows)\n                             for k, rows in self.motion.samples.items())\n                rank = (0 if prediction and prediction[0] == destination[0] else\n                        1 if key in self.symmetric else 2 if moving else\n                        15 if key in self.non_symmetric else 8)\n                stage = \'test_symmetry_or_moving_contact_within_target_orbit\'\n            else:\n                rank = (0 if prediction and fit_transforms(prediction[0], destination[0]) else\n                        1 if key in self.non_symmetric and evidence_status == \'unknown\' else\n                        2 if key in self.non_symmetric else 5 if evidence_status == \'unknown\' else 12)\n                stage = \'leave_unreachable_orbit_by_recontacting_non_D4_effect\'\n            condition = reading, key\n            options.append((rank, self.unknown_attempts[condition], len(path), repr(goal),\n                            goal, path, key, stage, evidence_status, len(predictions)))\n        if reading == destination:\n            goal = ContactGoal(anchor, target.sig, target.x, target.y, offset)\n            path = self._contact_route(scene, controls, obs, goal)\n            if path:\n                options.append((-20, 0, len(path), repr(goal), goal, path, None,\n                                \'test_public_goal_after_shape_and_color_match\', \'observed\', 1))\n        if not options:\n            return original\n        rank, _, _, _, goal, path, key, stage, evidence_status, nmodels = min(options)\n        if len(path)+1 >= resource[1]:\n            if not supplies:\n                return original\n            _, _, _, goal, path = min(supplies)\n            stage = \'prepare_resource_for_selected_contact\'\n        self.unknown_attempts[reading, key] += 1\n        if self._probe_original is not None:\n            self.state_trials = self._probe_original\n            self._probe_original = self._probe_evidence = None\n        self.goal, self.joint_mode = goal, False\n        self.reading_pending = None\n        self.branch = None\n        self.context_choice = {\'stage\': stage, \'catalogue_status\': catalogue_status,\n            \'input\': reading, \'target\': destination, \'selected_patch\': key,\n            \'evidence_status\': evidence_status, \'model_output_count\': nmodels,\n            \'rank\': rank, \'planned_length\': len(path), \'fuel_estimate\': resource[1]}\n        return path\n\n    def choose(self, obs, objects, candidates):\n        self.context_choice = None\n        action, reason = super().choose(obs, objects, candidates)\n        if self.context_choice:\n            self.context_events.append({\'kind\': \'context_plan\', \'action_index\': self.stats[\'observations\']+2,\n                \'observation\': obs.key(), \'action\': action.id, **self.context_choice})\n            reason += \'; \' + self.context_choice[\'stage\']\n        return action, reason\n\n    def observe(self, before, action, after, old_objects, new_objects):\n        self.learning_action = action\n        step = self.stats[\'observations\'] + 2\n        super().observe(before, action, after, old_objects, new_objects)\n        if before.levels == self.intervention_level and before.levels == after.levels:\n            self.motion.observe(before, action.id, after, step)\n        self.learning_action = None\n\n    def summary(self):\n        result = super().summary()\n        result[\'context_contact\'] = {\'active\': self.context_active,\n            \'stats\': dict(self.context_stats), \'events\': self.context_events,\n            \'non_symmetric_patch_keys\': sorted(self.non_symmetric),\n            \'symmetric_patch_keys\': sorted(self.symmetric),\n            \'exact_table\': [{\'patch\': key, \'samples\': [{\'transition\': s, \'count\': n} for s,n in rows.items()]}\n                            for key,rows in self.contact_table.items()]}\n        return result\n', 'public_observer.py': '"""Minimal public-frame reader and action-conditioned moving-patch hypotheses.\n\nNo game source, game identifier, hidden state, or future frame enters prediction.\nPredictions are possible contacts supported by a finite prefix, not game rules.\n"""\nfrom __future__ import annotations\nfrom collections import Counter, defaultdict\nfrom dataclasses import dataclass\nimport hashlib\nimport json\nfrom pathlib import Path\nimport sys\nimport numpy as np\n\nsys.path.insert(0, str(Path(__file__).resolve().parent/\'reference\'))\nfrom agent import components\nfrom association_dynamics import framed_patterns\nfrom reconsideration import contact_landmarks\n\n\ndef integer(value, name):\n    if isinstance(value, (bool, np.bool_)) or not isinstance(value, (int, np.integer)):\n        raise ValueError(f\'{name} must be an integer\')\n    return int(value)\n\n\n@dataclass(frozen=True)\nclass Observation:\n    pixels: np.ndarray\n    available: tuple[int,...]\n    state: str\n    levels: int\n    frames: tuple[np.ndarray,...]\n    win_levels: int\n\n    @classmethod\n    def from_record(cls, record):\n        frames=[]\n        for frame in record[\'frames\']:\n            a=np.asarray(frame)\n            if (a.ndim!=2 or not all(0<s<=64 for s in a.shape)\n                or a.dtype.kind not in \'iu\' or np.any(a<0) or np.any(a>15)):\n                raise ValueError(\'Every frame must be an integer palette grid, at most 64 by 64\')\n            a=a.astype(np.uint8,copy=True)\n            a.setflags(write=False)\n            frames.append(a)\n        if not frames:\n            raise ValueError(\'At least one frame is required\')\n        available=tuple(sorted(set(integer(a,\'action\') for a in record[\'available_actions\'])))\n        if any(a<0 or a>7 for a in available):\n            raise ValueError(\'Unknown action identifier\')\n        levels=integer(record[\'levels_completed\'],\'levels_completed\')\n        win=integer(record[\'win_levels\'],\'win_levels\')\n        if not 0<=levels<=win:\n            raise ValueError(\'Invalid level counts\')\n        state=record[\'state\']\n        if state not in (\'NOT_PLAYED\',\'NOT_FINISHED\',\'GAME_OVER\',\'WIN\'):\n            raise ValueError(\'Unknown public game state\')\n        return cls(frames[-1],available,state,levels,tuple(frames),win)\n\n    @classmethod\n    def from_frame(cls, frame):\n        return cls.from_record({\'frames\':frame.frame,\'available_actions\':frame.available_actions,\n            \'state\':frame.state.name if hasattr(frame.state,\'name\') else str(frame.state),\n            \'levels_completed\':frame.levels_completed,\'win_levels\':frame.win_levels})\n\n    def record(self):\n        return {\'frames\':[a.tolist() for a in self.frames], \'available_actions\':list(self.available),\n                \'state\':self.state,\'levels_completed\':self.levels,\'win_levels\':self.win_levels}\n\n    def key(self):\n        # Same final-frame key as baseline; full returned packet remains separate.\n        metadata=repr((self.pixels.shape,self.available,self.state,self.levels)).encode()\n        return hashlib.sha256(metadata+self.pixels.tobytes()).hexdigest()\n\n    def packet_key(self):\n        return hashlib.sha256(json.dumps(self.record(),sort_keys=True,separators=(\',\',\':\')).encode()).hexdigest()\n\n\n@dataclass(frozen=True)\nclass Patch:\n    key: tuple\n    box: tuple[int,int,int,int]\n    raw_box: tuple[int,int,int,int]\n    raw_key: tuple\n\n\ndef patches(obs):\n    cs=components(obs.pixels)\n    panels=framed_patterns(obs.pixels,cs)\n    result=[]\n    for p in contact_landmarks(obs.pixels,cs,panels,set()):\n        a=np.frombuffer(bytes.fromhex(p.sig[3]),dtype=np.uint8).reshape(p.sig[2],p.sig[1])\n        if len(np.unique(a))<2:\n            continue  # Uniform terrain is retained in pixels, not a glyph track.\n        x,y=p.x,p.y\n        # Retrieval proposal only: retain both raw and trimmed representations.\n        if min(a.shape)>=5:\n            edge=np.concatenate((a[0],a[-1],a[:,0],a[:,-1]))\n            if np.all(edge==edge[0]):\n                a=a[1:-1,1:-1]\n                x,y=x+1,y+1\n        key=(a.shape[1],a.shape[0],a.tobytes().hex())\n        result.append(Patch(key,(x,y,a.shape[1],a.shape[0]),(p.x,p.y,p.sig[1],p.sig[2]),p.sig))\n    return result\n\n\ndef unique_by_key(items):\n    groups=defaultdict(list)\n    for item in items:\n        groups[item.key].append(item)\n    return {k:v[0] for k,v in groups.items() if len(v)==1}\n\n\ndef center(box):\n    x,y,w,h=box\n    return x+w/2,y+h/2\n\n\nclass MotionObserver:\n    def __init__(self):\n        self.samples=defaultdict(list)\n        self.last_observed_index=0\n\n    def observe(self,before,action,after,action_index):\n        action_index=integer(action_index,\'action_index\')\n        if action_index<=self.last_observed_index:\n            raise ValueError(\'Observation indices must increase\')\n        self.last_observed_index=action_index\n        if (action==0 or before.levels!=after.levels or before.state==\'GAME_OVER\'\n            or after.state in (\'WIN\',\'GAME_OVER\')):\n            self.samples.clear()\n            return\n        a,b=unique_by_key(patches(before)),unique_by_key(patches(after))\n        for key in a.keys() & b.keys():\n            dx,dy=b[key].box[0]-a[key].box[0],b[key].box[1]-a[key].box[1]\n            self.samples[key,action].append({\'delta\':(dx,dy),\'source_action_index\':action_index})\n\n    def possible_contacts(self,obs,action,mover_before,mover_delta,decision_action_index):\n        if decision_action_index<=self.last_observed_index:\n            raise ValueError(\'Prediction must precede the action it predicts\')\n        if action not in obs.available:\n            raise ValueError(\'Action is unavailable\')\n        mx,my=center(mover_before)\n        after_center=(mx+mover_delta[0],my+mover_delta[1])\n        result=[]\n        for key,p in unique_by_key(patches(obs)).items():\n            if p.box==tuple(mover_before):\n                continue\n            previous=self.samples.get((key,action),[])\n            candidates=defaultdict(list)\n            for s in previous:\n                candidates[tuple(s[\'delta\'])].append(s[\'source_action_index\'])\n            for (dx,dy),indices in candidates.items():\n                cx,cy=center(p.box)\n                dynamic_distance=sum(max(0,abs(after_center[i]-(cx+dx,cy+dy)[i])-.5) for i in (0,1))\n                static_distance=sum(max(0,abs(after_center[i]-(cx,cy)[i])-.5) for i in (0,1))\n                if dynamic_distance==0:\n                    result.append({\'patch_key\':key,\'before_box\':p.box,\'delta\':(dx,dy),\n                        \'predicted_box\':(p.box[0]+dx,p.box[1]+dy,p.box[2],p.box[3]),\n                        \'source_action_indices\':indices,\'support\':len(indices),\n                        \'static_contact_distance\':static_distance,\'dynamic_contact_distance\':dynamic_distance,\n                        \'status\':\'possible_contact_from_observed_motion; not a certified effect\'})\n        return result\n\n\nclass JsonlRecorder:\n    """Append only whitelisted public data; refuse to overwrite an existing log."""\n    def __init__(self,path):\n        self.file=Path(path).open(\'x\',encoding=\'utf-8\')\n        self.index=0\n\n    def append(self,action,observation,reason=\'\'):\n        aid=integer(action[\'id\'],\'action id\')\n        if not 0<=aid<=7:\n            raise ValueError(\'Unknown action id\')\n        safe_action={\'id\':aid}\n        if aid==6:\n            for k,bound in ((\'x\',observation.pixels.shape[1]),(\'y\',observation.pixels.shape[0])):\n                safe_action[k]=integer(action[k],k)\n                if not 0<=safe_action[k]<bound:\n                    raise ValueError(\'Click outside frame\')\n        self.index+=1\n        row={\'action_index\':self.index,\'action\':safe_action,\'reason\':str(reason),**observation.record()}\n        self.file.write(json.dumps(row,separators=(\',\',\':\'))+\'\\n\')\n        self.file.flush()\n\n    def close(self):\n        self.file.close()\n', 'unknown_contact.py': '"""One intervention: prioritize unmeasured contact effects at a public level.\n\nReuses the frozen fiber_probe_coupled reader, movement model, resource estimate\nand effect learner. No game ID, hidden game state, fixed shape or fixed color is\nused. intervention_level is a declared experimental gate in public progress.\n"""\nfrom collections import Counter\n\nfrom fiber_probe import FiberProbeDynamics\nfrom reconsideration import contact_landmarks\n\n\ndef measured_effect_status(samples, reading):\n    """Extrapolated identity is not a measured no-change on this input."""\n    outcomes = Counter()\n    for (before_shape, before_ink, after_shape, after_ink), count in samples.items():\n        if (before_shape, before_ink) == reading and count:\n            outcomes[after_shape, after_ink] += count\n    if not outcomes:\n        return \'unknown\', outcomes\n    if len(outcomes) > 1:\n        return \'conflicting\', outcomes\n    return (\'observed_no_change\' if reading in outcomes else \'observed_change\'), outcomes\n\n\nclass UnknownContactDynamics(FiberProbeDynamics):\n    def __init__(self, seed=0, mode=\'coupled\', intervention_level=4):\n        self.intervention_level = intervention_level\n        self.unknown_stats = Counter()\n        self.unknown_events = []\n        super().__init__(seed=seed, mode=mode)\n\n    def _clear_local(self):\n        super()._clear_local()\n        self.unknown_commitment = None\n        self.unknown_attempts = Counter()\n        self.unknown_choice = None\n\n    def _accept_route(self, obs, goal, path, detail):\n        # Parent may have installed a read-only fallback score view. This is a\n        # route, so do not report it as an action selected by fallback scoring.\n        if self._probe_original is not None:\n            self.state_trials = self._probe_original\n            self._probe_original = self._probe_evidence = None\n        self.goal, self.joint_mode = goal, False\n        self.unknown_choice = detail\n        return path\n\n    def _plan(self, scene, controls, obs):\n        original_path = super()._plan(scene, controls, obs)\n        if (obs.levels != self.intervention_level or not self.scope_active\n                or self.reading_pending is not None or self.branch is not None):\n            return original_path\n        pair = self._current_pair(obs, self.contact_objects)\n        cohort = self._cohort(scene, controls)\n        budget = self._budget(obs)\n        if pair is None or cohort is None or budget is None or budget[1] <= 0:\n            self.unknown_stats[\'missing_public_precondition\'] += 1\n            return original_path\n        reference, target = pair\n        reading = reference.pattern, reference.ink\n        if reading == (target.pattern, target.ink):\n            return original_path  # Keep the already matched goal route.\n        anchor, offset, movers = cohort\n        self.navigation_cohort = movers\n        committed = self.unknown_commitment\n        if committed:\n            goal = committed[\'goal\']\n            status, _ = measured_effect_status(self.effect_samples.get(committed[\'sig\'], {}), reading)\n            if (reading == committed[\'reading\'] and status in (\'unknown\', \'conflicting\')\n                    and goal.distance(scene) != 0\n                    and self.stats[\'observations\'] - committed[\'started\'] < 64):\n                path = self._contact_route(scene, controls, obs, goal)\n                if path and len(path) + 3 < budget[1]:\n                    self.unknown_stats[\'committed_route_steps\'] += 1\n                    detail = dict(committed[\'detail\'], continued=True,\n                                  planned_length=len(path), measured_status=status)\n                    return self._accept_route(obs, goal, path, detail)\n            self.unknown_commitment = None\n        options = []\n        for landmark in contact_landmarks(obs.pixels, self.contact_objects, pair, movers):\n            samples = self.effect_samples.get(landmark.sig, {})\n            status, outcomes = measured_effect_status(samples, reading)\n            self.unknown_stats[\'candidate_\' + status] += 1\n            if status not in (\'unknown\', \'conflicting\'):\n                continue\n            goal = landmark.goal(anchor, offset)\n            path = self._contact_route(scene, controls, obs, goal)\n            if not path or len(path) + 3 >= budget[1]:\n                continue\n            condition = reading, landmark.sig, landmark.x, landmark.y\n            # The only rank change: unknown relation before conflicting, then\n            # fewer attempted experiments and a shorter measured-motion route.\n            # No predicted no-change, guessed usefulness, or color hint prunes it.\n            rank = (0 if status == \'unknown\' else 1,\n                    self.unknown_attempts[condition], len(path), repr(goal))\n            detail = {\n                \'landmark\': landmark.sig, \'xy\': [landmark.x, landmark.y],\n                \'input_reading\': reading, \'measured_status\': status,\n                \'exact_sample_count\': sum(outcomes.values()),\n                \'exact_outcome_count\': len(outcomes),\n                \'previous_route_attempts\': self.unknown_attempts[condition],\n                \'model_prediction_before_learning\': self._effect(landmark, reading),\n                \'planned_length\': len(path), \'resource_estimate\': budget[1],\n                \'continued\': False,\n            }\n            options.append((rank, condition, goal, path, detail))\n        if not options:\n            self.unknown_stats[\'no_reachable_unknown_contact\'] += 1\n            return original_path\n        _, condition, goal, path, detail = min(options, key=lambda item: item[0])\n        self.unknown_attempts[condition] += 1\n        self.unknown_commitment = {\'goal\': goal, \'reading\': reading,\n            \'sig\': detail[\'landmark\'], \'started\': self.stats[\'observations\'], \'detail\': detail}\n        self.unknown_stats[\'new_routes\'] += 1\n        return self._accept_route(obs, goal, path, detail)\n\n    def choose(self, obs, objects, candidates):\n        self.unknown_choice = None\n        action, reason = super().choose(obs, objects, candidates)\n        if self.unknown_choice is not None:\n            if not self.pending or self.pending[\'key\'][0] != action.id:\n                raise AssertionError(\'Unknown-contact route does not match actual pending action\')\n            event = {\'kind\': \'unknown_contact_route\',\n                \'action_index\': self.stats[\'observations\'] + 2,\n                \'level\': obs.levels, \'observation\': obs.key(),\n                \'action\': {\'id\': action.id, **action.data()}, **self.unknown_choice}\n            self.unknown_events.append(event)\n            self.unknown_stats[\'intervention_decisions\'] += 1\n            reason += \'; prioritize an unmeasured contact effect, using existing motion model\'\n        return action, reason\n\n    def summary(self):\n        result = super().summary()\n        result[\'unknown_contact\'] = {\n            \'intervention_completed_levels\': self.intervention_level,\n            \'stats\': dict(self.unknown_stats), \'events\': self.unknown_events,\n            \'scope\': \'One contact-priority intervention; finite public reading evidence; raw patches retained; no hidden-state or goal guarantee\',\n        }\n        return result\n', 'fiber_probe.py': '"""Apply observed return costs to fallback probes as well as planned routes.\n\nThe original chooser constructs all pending predictions for its actual choice.\nA read-only score view changes only the current scene\'s fallback ranking, and\nis restored before observation. Plans, initial controls and repair experiments\nretain their original choice rules. Costs are finite heuristic trial counts.\n"""\nfrom collections import Counter\nfrom collections.abc import Mapping\n\nfrom fiber_hysteresis import FiberDynamics\nfrom tesla_dynamics import pose\n\n\nclass ProbeTrialView(Mapping):\n    def __init__(self, original, scene, penalties):\n        self.original, self.scene, self.penalties = original, scene, dict(penalties)\n\n    def __getitem__(self, item):\n        scene, key = item\n        return self.original[item]+(self.penalties.get(key, 0) if scene == self.scene else 0)\n\n    def __iter__(self):\n        return iter(self.original)\n\n    def __len__(self):\n        return len(self.original)\n\n\nclass FiberProbeDynamics(FiberDynamics):\n    def __init__(self, seed=0, mode=\'coupled\'):\n        self.probe_stats = Counter()\n        self.probe_events = []\n        self._probe_original = None\n        self._probe_evidence = None\n        super().__init__(seed, mode)\n\n    def _plan(self, scene, controls, obs):\n        path = super()._plan(scene, controls, obs)\n        if path or not self.scope_active or self.planning_context is None or not self.navigation_cohort:\n            return path\n        budget = self._budget(obs)\n        position = pose(scene, self.navigation_cohort)\n        if not budget or budget[1] <= 0 or position is None:\n            return path\n        node = self.planning_context, position\n        costs = {k: self.flow.resistance[node, k] for k in controls if k[0] in (1, 2, 3, 4)}\n        if not any(costs.values()):\n            return path\n        if self._probe_original is not None:\n            raise RuntimeError(\'Nested probe scoring is not supported\')\n        self._probe_original = self.state_trials\n        self._probe_evidence = {\'scene\': scene, \'costs\': costs,\n            \'base_trials\': {k: self.state_trials[scene, k] for k in costs},\n            \'sources\': {k: list(self.flow.sources.get((node, k), [])) for k in costs}}\n        self.state_trials = ProbeTrialView(self.state_trials, scene, costs)\n        return path\n\n    def choose(self, obs, objects, candidates):\n        self._probe_original = self._probe_evidence = None\n        try:\n            action, reason = super().choose(obs, objects, candidates)\n            if self._probe_evidence is not None:\n                evidence = self._probe_evidence\n                key = self.pending[\'key\']\n                event = {\'kind\': \'probe_cost\', \'action_index\': self.stats[\'observations\']+2,\n                    \'observation\': obs.key(), \'level\': obs.levels,\n                    \'action\': {\'id\': action.id, **action.data()}, \'selected_control\': list(key),\n                    \'choices\': [{\'control\': list(k), \'base_trials\': evidence[\'base_trials\'][k],\n                        \'resistance\': cost, \'adjusted_trials\': evidence[\'base_trials\'][k]+cost,\n                        \'source_action_indices\': evidence[\'sources\'][k]}\n                        for k, cost in sorted(evidence[\'costs\'].items())]}\n                self.probe_events.append(event)\n                self.probe_stats[\'scored_fallback_choices\'] += 1\n                self.probe_stats[\'selected_positive_cost\'] += evidence[\'costs\'].get(key, 0) > 0\n                reason += \'; observed return cost included in fallback probe ranking\'\n            return action, reason\n        finally:\n            if self._probe_original is not None:\n                self.state_trials = self._probe_original\n            self._probe_original = self._probe_evidence = None\n\n    def summary(self):\n        result = super().summary()\n        result[\'fiber_probe\'] = {\'stats\': dict(self.probe_stats), \'events\': self.probe_events,\n            \'scope\': \'Same finite return costs added to current-scene fallback trial ranking; no action veto or new motion prediction\'}\n        return result\n', 'tesla_dynamics.py': '"""Directed route resistance inspired by, not simulating, a Tesla conduit.\n\nOnly repeated, already-observed, resource-consuming returns raise resistance.\nThe coarse return key is an exploration cue, never a transition-equivalence\nclaim. The original full-scene predictor and the public resource gauge remain.\nNew outcomes reopen a route; reverse directions are not automatically penalized.\n"""\nfrom collections import Counter, deque\nfrom dataclasses import dataclass\nimport hashlib\nimport heapq\nimport itertools\n\nfrom association_dynamics import framed_patterns\nfrom scope_extension import ScopeExtensionDynamics, structural_pairs\nfrom shared_dynamics import entities, unique_entities\n\n\ndef digest(value):\n    return hashlib.sha256(repr(value).encode()).hexdigest()\n\n\ndef reading_context(obs, objects):\n    pairs = structural_pairs(framed_patterns(obs.pixels, objects), obs.pixels.shape)\n    if len(pairs) != 1:\n        return None\n    a, b = pairs[0]\n    return (tuple(obs.pixels.shape), (a.pattern, a.ink),\n            (b.pattern, b.ink), (b.sig, b.x, b.y))\n\n\ndef pose(scene, movers):\n    current = unique_entities(scene)\n    if not movers or any(s not in current for s in movers):\n        return None\n    return tuple(sorted((s, *current[s]) for s in movers))\n\n\n@dataclass(frozen=True)\nclass FlowSample:\n    step: int\n    before: tuple\n    action: tuple\n    after: tuple\n    fuel_before: int\n    fuel_after: int\n    novel: bool\n\n\nclass ReturnResistance:\n    def __init__(self, window=24, maximum=24):\n        self.window, self.maximum = window, maximum\n        self.recent = deque(maxlen=window)\n        self.outcomes = set()\n        self.resistance = Counter()\n        self.sources = {}\n        self.last_step = 0\n\n    def push(self, step, before, action, after, fuel_before, fuel_after):\n        if step <= self.last_step:\n            raise ValueError(\'One chronological physical transition per step\')\n        self.last_step = step\n        # Resource depletion/replenishment and changed reading conditions are\n        # explicit boundaries. They cannot become ordinary movement examples.\n        if (before is None or after is None or fuel_before is None or fuel_after is None\n                or fuel_before <= 0 or fuel_after <= 0 or fuel_after > fuel_before\n                or before[0] != after[0]):\n            self.recent.clear()\n            return [], False\n        if self.recent and self.recent[-1].after != before:\n            self.recent.clear()\n        sample_key = before, action, after\n        novel = sample_key not in self.outcomes\n        self.outcomes.add(sample_key)\n        gate = before, action\n        events = []\n        if novel and self.resistance[gate]:\n            events.append({\'kind\': \'gate_reopened\', \'action_index\': step,\n                \'gate\': digest(gate), \'old_resistance\': self.resistance[gate],\n                \'reason\': \'new observed outcome for a previously resisted direction\'})\n            del self.resistance[gate]\n            self.sources.pop(gate, None)\n        item = FlowSample(step, before, action, after, fuel_before, fuel_after, novel)\n        self.recent.append(item)\n        trail = list(self.recent)\n        for start in range(len(trail)-1, -1, -1):\n            if trail[start].before != after:\n                continue\n            loop = trail[start:]\n            spent = loop[0].fuel_before-loop[-1].fuel_after\n            if spent > 0 and not any(t.novel for t in loop):\n                entry = loop[0].before, loop[0].action\n                old = self.resistance[entry]\n                self.resistance[entry] = min(self.maximum, old+len(loop))\n                self.sources[entry] = [t.step for t in loop]\n                events.append({\'kind\': \'gate_resistance\', \'action_index\': step,\n                    \'gate\': digest(entry), \'entry_action\': list(loop[0].action),\n                    \'source_action_indices\': [t.step for t in loop],\n                    \'fuel_before\': loop[0].fuel_before, \'fuel_after\': loop[-1].fuel_after,\n                    \'fuel_spent\': spent, \'old_resistance\': old,\n                    \'new_resistance\': self.resistance[entry],\n                    \'unchanged_reading\': True, \'all_outcomes_seen_before\': True})\n            break\n        return events, novel\n\n\ndef weighted_route(start, controls, transition, at_goal, penalty, max_steps=64, max_nodes=1024):\n    """Positive-cost routing with cost/length Pareto labels and stable ties."""\n    serial = itertools.count()\n    queue = [(0, 0, next(serial), start, (), False)]\n    best = {start: [(0, 0)]}\n    expanded = 0\n    while queue and expanded < max_nodes:\n        cost, length, _, state, path, terminal = heapq.heappop(queue)\n        if terminal:\n            return list(path)\n        if (cost, length) not in best.get(state, ()) or length >= max_steps:\n            continue\n        expanded += 1\n        for key in controls:\n            after = transition(state, key)\n            if after is None or after == state:\n                continue\n            new_cost, new_length = cost+1+max(0, penalty(state, key)), length+1\n            new_path = path+(key,)\n            # A contact can require leaving and returning to the start pose.\n            terminal = at_goal(after)\n            if not terminal:\n                labels = best.get(after, [])\n                if any(c <= new_cost and n <= new_length for c, n in labels):\n                    continue\n                best[after] = [(c, n) for c, n in labels if not (new_cost <= c and new_length <= n)]\n                best[after].append((new_cost, new_length))\n            heapq.heappush(queue, (new_cost, new_length, next(serial), after, new_path, terminal))\n    return None\n\n\nclass TeslaDynamics(ScopeExtensionDynamics):\n    def __init__(self, seed=0, directional=True, branch_horizon=8):\n        self.directional = directional\n        self.branch_horizon = branch_horizon\n        self.tesla_stats = Counter()\n        self.tesla_events = []\n        self.branch_serial = 0\n        super().__init__(seed, adaptive=False)\n\n    def _clear_local(self):\n        super()._clear_local()\n        self.flow = ReturnResistance()\n        self.branch = None\n        self.branch_requested = False\n        self.flow_pending = None\n        self.planning_context = None\n\n    def _contact_route(self, scene, controls, obs, goal):\n        if not (self.scope_active and self.directional and self.flow.resistance\n                and self.navigation_cohort and self.planning_context is not None):\n            return super()._contact_route(scene, controls, obs, goal)\n        movers = self.navigation_cohort\n        context = self.planning_context\n        def penalty(state, key):\n            p = pose(state, movers)\n            return self.flow.resistance[((context, p), key)] if p else 0\n        self.tesla_stats[\'weighted_route_queries\'] += 1\n        return weighted_route(scene, [k for k in controls if k[0] in (1, 2, 3, 4)],\n            lambda s, k: self._predict(s, k, obs), lambda s: goal.distance(s) == 0, penalty)\n\n    def _finish_branch(self, reason, action_index, observation, levels_gained=0):\n        if self.branch is None:\n            return\n        b = self.branch\n        self.tesla_events.append({\'kind\': \'branch_end\', \'branch_id\': b[\'id\'],\n            \'action_index\': action_index, \'observation\': observation, \'reason\': reason,\n            \'executed_actions\': b[\'executed\'], \'new_transition_outcomes\': b[\'new_outcomes\'],\n            \'levels_gained\': levels_gained})\n        self.tesla_stats[\'branches_finished\'] += 1\n        self.tesla_stats[\'branch_\'+reason] += 1\n        self.branch = None\n\n    def _plan(self, scene, controls, obs):\n        context = reading_context(obs, self.contact_objects) if self.scope_active else None\n        self.planning_context = context\n        budget = self._budget(obs) if context is not None else None\n        if self.branch:\n            b = self.branch\n            if (context == b[\'context\'] and budget and len(b[\'remaining\'])+2 < budget[1]\n                    and b[\'remaining\'][0] in controls\n                    and self._predict(scene, b[\'remaining\'][0], obs) is not None):\n                self.goal, self.joint_mode = b[\'goal\'], False\n                self.tesla_stats[\'committed_branch_decisions\'] += 1\n                return list(b[\'remaining\'])\n            self._finish_branch(\'precondition_changed\', self.stats[\'observations\']+1, obs.key())\n        path = super()._plan(scene, controls, obs)\n        if (self.branch_requested and context is not None and budget and path and len(path) > 1\n                and budget[1] > 4 and self.reading_pending is None):\n            length = min(self.branch_horizon, len(path), budget[1]-3)\n            self.branch_serial += 1\n            self.branch = {\'id\': self.branch_serial, \'context\': context, \'goal\': self.goal,\n                           \'remaining\': list(path[:length]), \'executed\': 0, \'new_outcomes\': 0}\n            self.branch_requested = False\n            self.tesla_stats[\'branches_started\'] += 1\n            self.tesla_events.append({\'kind\': \'branch_start\', \'branch_id\': self.branch_serial,\n                \'action_index\': self.stats[\'observations\']+2, \'observation\': obs.key(),\n                \'planned_actions\': list(path[:length]), \'fuel_before\': budget[1],\n                \'directional_resistance_enabled\': self.directional,\n                \'previous_gate_source_steps\': sorted({j for v in self.flow.sources.values() for j in v})})\n            return list(path[:length])\n        return path\n\n    def choose(self, obs, objects, candidates):\n        self._ensure_level(obs.levels)\n        self.planning_context = reading_context(obs, objects) if self.scope_active else None\n        action, reason = super().choose(obs, objects, candidates)\n        if not self.scope_active:\n            return action, reason\n        scene = entities(objects, obs.pixels.shape)\n        cohort = self._cohort(scene, [(a,) for a in obs.available])\n        context = reading_context(obs, objects)\n        budget = self._budget(obs)\n        if cohort and context is not None and budget and self.pending:\n            movers = cohort[2]\n            p = pose(scene, movers)\n            predicted = self.pending[\'predicted\']\n            node = (context, p) if p is not None else None\n            gate = node, self.pending[\'key\']\n            self.flow_pending = {\'before\': node, \'movers\': tuple(sorted(movers)),\n                \'fuel\': budget, \'key\': self.pending[\'key\'], \'observation\': obs.key(),\n                \'predicted_pose\': pose(predicted, movers) if predicted is not None else None,\n                \'resistance\': self.flow.resistance[gate],\n                \'resistance_sources\': list(self.flow.sources.get(gate, [])),\n                \'branch_id\': self.branch[\'id\'] if self.branch else None}\n            if self.branch:\n                reason += \'; bounded branch with checked motion and resource reserve\'\n        return action, reason\n\n    def observe(self, before, action, after, old_objects, new_objects):\n        pending, self.flow_pending = self.flow_pending, None\n        step = self.stats[\'observations\']+2\n        usable = (pending is not None and action.id in (1, 2, 3, 4)\n                  and before.levels == after.levels and after.state not in (\'WIN\', \'GAME_OVER\'))\n        # The last action still costs one action when it completes a level or\n        # terminates a game. Count it before the parent clears local memory.\n        if self.branch and pending and pending[\'branch_id\'] == self.branch[\'id\']:\n            b = self.branch\n            if b[\'remaining\'][0] == pending[\'key\']:\n                b[\'remaining\'].pop(0)\n                b[\'executed\'] += 1\n                self.tesla_stats[\'branch_external_actions\'] += 1\n            else:\n                self._finish_branch(\'another_control_selected\', step, after.key())\n        if not usable:\n            self._finish_branch(\'level_or_terminal_boundary\', step, after.key(), after.levels-before.levels)\n        super().observe(before, action, after, old_objects, new_objects)\n        if not usable:\n            return\n        context = reading_context(after, new_objects)\n        p = pose(entities(new_objects, after.pixels.shape), pending[\'movers\'])\n        node = (context, p) if context is not None and p is not None else None\n        budget = self._budget(after)\n        before_fuel = pending[\'fuel\'][1]\n        after_fuel = budget[1] if budget and budget[0] == pending[\'fuel\'][0] else None\n        events, novel = self.flow.push(step, pending[\'before\'], pending[\'key\'], node, before_fuel, after_fuel)\n        for event in events:\n            event.update(observation=before.key(), observation_after=after.key())\n            self.tesla_events.append(event)\n            self.tesla_stats[event[\'kind\']] += 1\n        self.tesla_stats[\'public_transitions_processed\'] += 1\n        self.tesla_stats[\'new_transition_outcomes\'] += novel\n        if pending[\'resistance\']:\n            self.tesla_events.append({\'kind\': \'resisted_direction_taken\', \'action_index\': step,\n                \'observation\': before.key(), \'observation_after\': after.key(),\n                \'action\': {\'id\': action.id, **action.data()}, \'resistance\': pending[\'resistance\'],\n                \'source_action_indices\': pending[\'resistance_sources\'],\n                \'applied_to_route_cost\': self.directional})\n            self.tesla_stats[\'resisted_directions_taken\'] += 1\n        if self.branch and pending[\'branch_id\'] == self.branch[\'id\']:\n            b = self.branch\n            b[\'new_outcomes\'] += novel\n            if context != b[\'context\']:\n                self._finish_branch(\'reading_changed\', step, after.key())\n            elif after_fuel is None or after_fuel <= 0:\n                self._finish_branch(\'resource_boundary\', step, after.key())\n            elif after_fuel > before_fuel:\n                self._finish_branch(\'resource_increased\', step, after.key())\n            elif pending[\'predicted_pose\'] is None or pending[\'predicted_pose\'] != p:\n                self._finish_branch(\'motion_prediction_failed\', step, after.key())\n            elif not b[\'remaining\']:\n                self._finish_branch(\'horizon_reached\', step, after.key())\n        if any(e[\'kind\'] == \'gate_resistance\' for e in events):\n            self._finish_branch(\'unproductive_return\', step, after.key())\n            self.branch_requested = True\n\n    def summary(self):\n        result = super().summary()\n        result[\'tesla\'] = {\'directional_resistance\': self.directional,\n            \'branch_horizon\': self.branch_horizon,\n            \'stats\': dict(sorted(self.tesla_stats.items())), \'events\': self.tesla_events,\n            \'current_resistances\': {repr(k): v for k, v in self.flow.resistance.items() if v},\n            \'scope\': \'Directed soft route costs and bounded checked branches; return cues are not state equivalences; no fluid simulation or calibrated posterior\'}\n        return result\n', 'fiber_hysteresis.py': '"""Finite, fluid-inspired coupled relays; not a calibrated fiber simulation.\n\nThe input c_i is an observed return penalty, in heuristic action-cost units.\nEach available direction is a parallel channel; a unit-conductance bypass\nremains.  Q=n+1, R_i=1+c_i+2*s_i, P=Q/(1+sum(1/R_i)), q_i=P/R_i.\nLoading L_i=c_i+2*(1-q_i) switches an open relay above 3, and releases a\nclosed relay below 1.5.  Between these thresholds the occupied branch stays.\nAn obstruction redistributes load to the other channels through shared P.\nThe numerical mapping and constants are a policy proposal, not paper values.\n"""\nfrom collections import Counter\nfrom dataclasses import dataclass\nimport math\n\nfrom tesla_dynamics import ReturnResistance, TeslaDynamics, digest\n\n\n@dataclass(frozen=True)\nclass FiberParameters:\n    shape_cost: float = 2.0\n    redistribution_gain: float = 2.0\n    snap_up: float = 3.0\n    snap_down: float = 1.5\n    bypass_conductance: float = 1.0\n\n\nPARAMETERS = FiberParameters()\nMODES = (\'coupled\', \'independent\', \'instant\')\n\n\nclass FiberNetwork:\n    """Small quasistatic relay network with an explicit settling convention.\n\n    Controls have fixed sorted priority when several relays are unstable.\n    A repeated configuration aborts relaxation to the zero-shape cost\n    baseline, and is reported; it is never asserted to be an equilibrium.\n    No waiting, random sampling, or time-dependent decay creates the memory.\n    """\n    def __init__(self, controls, mode=\'coupled\'):\n        if mode not in MODES:\n            raise ValueError(mode)\n        self.controls = tuple(sorted(set(controls)))\n        if not self.controls or any(len(k) != 1 or k[0] not in (1, 2, 3, 4)\n                                    for k in self.controls):\n            raise ValueError(\'One to four available directional controls required\')\n        self.mode = mode\n        self.shapes = [0] * len(self.controls)\n        self.evidence_steps = set()\n\n    def hydraulic_state(self, loads, shapes=None):\n        p = PARAMETERS\n        shapes = self.shapes if shapes is None else shapes\n        resistance = [1+c+p.shape_cost*s for c, s in zip(loads, shapes)]\n        total = len(resistance)+p.bypass_conductance\n        pressure = total/(p.bypass_conductance+sum(1/r for r in resistance))\n        flows = [pressure/r for r in resistance]\n        # In the independent ablation, each relay instead sees its own\n        # network with every other channel clamped at unit conductance.\n        sensed = flows if self.mode != \'independent\' else [\n            total/(len(resistance)+1/r)/r for r in resistance]\n        drive = [c+p.redistribution_gain*(1-q) for c, q in zip(loads, sensed)]\n        return {\'pressure\': pressure, \'flows\': flows,\n                \'bypass_flow\': pressure*p.bypass_conductance, \'total_flow\': total,\n                \'sensed_flows\': sensed, \'drive\': drive,\n                \'route_penalties\': [r-1 for r in resistance]}\n\n    def relax(self, loads):\n        if (len(loads) != len(self.controls)\n                or any(not math.isfinite(c) or c < 0 for c in loads)):\n            raise ValueError(\'Finite nonnegative load per control required\')\n        previous = self.shapes[:]\n        if self.mode == \'instant\':\n            self.shapes = [0] * len(loads)\n        initial = self.shapes[:]\n        seen, switches = set(), []\n        aborted = False\n        while True:\n            state = tuple(self.shapes)\n            if state in seen:\n                self.shapes = [0] * len(loads)\n                aborted = True\n                break\n            seen.add(state)\n            hydraulic = self.hydraulic_state(loads)\n            change = None\n            for i, (shape, drive) in enumerate(zip(self.shapes, hydraulic[\'drive\'])):\n                if not shape and drive > PARAMETERS.snap_up+1e-12:\n                    change = i, 1, PARAMETERS.snap_up\n                    break\n                if shape and drive < PARAMETERS.snap_down-1e-12:\n                    change = i, 0, PARAMETERS.snap_down\n                    break\n            if change is None:\n                break\n            i, target, threshold = change\n            switches.append({\'control\': list(self.controls[i]), \'from\': self.shapes[i],\n                             \'to\': target, \'drive\': hydraulic[\'drive\'][i],\n                             \'threshold\': threshold})\n            self.shapes[i] = target\n        return {\'controls\': [list(k) for k in self.controls], \'base_penalties\': list(loads),\n                \'shapes_before\': previous, \'relaxation_start_shapes\': initial,\n                \'shapes_after\': self.shapes[:], \'switches\': switches,\n                \'settling_aborted\': aborted, **self.hydraulic_state(loads)}\n\n\nclass FiberResistance:\n    """Keep Tesla\'s evidence detector, change only its route-cost response.\n\n    All load changes follow observed transitions. A network is local to a\n    reading context and mover pose; unfamiliar gates get no shape surcharge.\n    Its state is never merged with the transition predictor\'s state.\n    """\n    def __init__(self, mode=\'coupled\'):\n        self.mode = mode\n        self.raw = ReturnResistance()\n        self.resistance = Counter()\n        self.sources = {}\n        self.networks = {}\n        self.control_keys = ((1,), (2,), (3,), (4,))\n\n    def push(self, step, before, action, after, fuel_before, fuel_after):\n        old = {k: v for k, v in self.raw.resistance.items() if v}\n        events, novel = self.raw.push(step, before, action, after, fuel_before, fuel_after)\n        new = {k: v for k, v in self.raw.resistance.items() if v}\n        dirty = {gate[0] for gate in old.keys() | new.keys() if old.get(gate, 0) != new.get(gate, 0)}\n        for node in sorted(dirty, key=repr):\n            controls = tuple(sorted(self.control_keys))\n            if not controls:\n                continue\n            network = self.networks.get(node)\n            if network is None or network.controls != controls:\n                network = self.networks[node] = FiberNetwork(controls, self.mode)\n            network.evidence_steps.add(step)\n            for key in controls:\n                network.evidence_steps.update(self.raw.sources.get((node, key), []))\n            loads = [new.get((node, key), 0) for key in controls]\n            outcome = network.relax(loads)\n            for key, penalty in zip(controls, outcome[\'route_penalties\']):\n                gate = node, key\n                self.resistance[gate] = penalty\n                if penalty:\n                    self.sources[gate] = sorted(network.evidence_steps)\n                else:\n                    self.sources.pop(gate, None)\n            events.append({\'kind\': \'fiber_update\', \'action_index\': step, \'node\': digest(node),\n                           \'mode\': self.mode, \'source_action_indices\': sorted(network.evidence_steps),\n                           **outcome})\n        return events, novel\n\n\nclass FiberDynamics(TeslaDynamics):\n    def __init__(self, seed=0, mode=\'coupled\'):\n        if mode not in MODES:\n            raise ValueError(mode)\n        self.fiber_mode = mode\n        super().__init__(seed)\n\n    def _clear_local(self):\n        super()._clear_local()\n        self.flow = FiberResistance(self.fiber_mode)\n\n    def observe(self, before, action, after, old_objects, new_objects):\n        self.flow.control_keys = tuple((i,) for i in before.available if i in (1, 2, 3, 4))\n        super().observe(before, action, after, old_objects, new_objects)\n\n    def summary(self):\n        result = super().summary()\n        result[\'fibers\'] = {\'mode\': self.fiber_mode, \'parameters\': vars(PARAMETERS),\n            \'networks_retained\': len(self.flow.networks),\n            \'scope\': \'Finite coupled relay policy with fixed shared flow and retained branch state; no physical fiber simulation, calibrated probability, or state-equivalence claim\'}\n        return result\n', 'goal_navigation.py': '"""Reuse measured local motion exceptions only after public goal reading matches.\n\nPosition + actor geometry + raw terrain fringe is a falsifiable retrieval key,\nnot a complete state. Conflicting deltas disable reuse; no game ID is used.\n"""\nfrom collections import Counter, defaultdict\nimport hashlib\nfrom context_contact import ContextContactDynamics\nfrom context_navigation import navigation_view, rigid_delta, translate\nfrom shared_dynamics import entities\n\n\nclass GoalNavigationDynamics(ContextContactDynamics):\n    def __init__(self, seed=0):\n        self.nav_stats=Counter()\n        self.nav_events=[]\n        super().__init__(seed)\n\n    def _clear_local(self):\n        super()._clear_local()\n        self.nav_evidence=defaultdict(Counter)\n        self.nav_sources=defaultdict(list)\n        self.nav_exceptions=set()\n        self.nav_active=False\n        self.observed_scene=None\n        self.nav_cache=None\n        self.nav_pending=None\n\n    def _view(self,scene,obs,movers):\n        key=scene,frozenset(movers or ())\n        if self.nav_cache is not None:\n            if key not in self.nav_cache:\n                self.nav_cache[key]=navigation_view(scene,obs.pixels,movers,self.observed_scene)\n            return self.nav_cache[key]\n        return navigation_view(scene,obs.pixels,movers,self.observed_scene)\n\n    def _predict(self,scene,key,obs):\n        if self.nav_active and self.navigation_cohort:\n            view=self._view(scene,obs,self.navigation_cohort)\n            condition=view,key\n            if view is not None and condition in self.nav_exceptions:\n                outcomes=self.nav_evidence[condition]\n                if len(outcomes)!=1:\n                    return None\n                delta=next(iter(outcomes))\n                return translate(scene,self.navigation_cohort,delta) if delta!=(0,0) else None\n        return super()._predict(scene,key,obs)\n\n    def choose(self,obs,objects,candidates):\n        self._ensure_level(obs.levels)\n        pair=self._current_pair(obs,objects) if obs.levels==self.intervention_level else None\n        self.nav_active=bool(pair and (pair[0].pattern,pair[0].ink)==(pair[1].pattern,pair[1].ink))\n        self.observed_scene=entities(objects,obs.pixels.shape)\n        self.nav_cache={}\n        try:\n            action,reason=super().choose(obs,objects,candidates)\n        finally:\n            self.nav_cache=None\n        if self.nav_active and self.pending and self.navigation_cohort:\n            view=self._view(self.observed_scene,obs,self.navigation_cohort)\n            condition=view,self.pending[\'key\']\n            if condition in self.nav_exceptions:\n                outcomes=self.nav_evidence[condition]\n                self.nav_pending={\'before\':obs.key(),\'action_index\':self.stats[\'observations\']+2,\n                    \'action\':action.id,\'condition\':hashlib.sha256(repr(condition).encode()).hexdigest(),\n                    \'predicted_delta\':next(iter(outcomes)) if len(outcomes)==1 else None,\n                    \'sources\':list(self.nav_sources[condition]),\'movers\':self.navigation_cohort}\n                reason+=\'; measured local motion exception while preserving matched goal reading\'\n        return action,reason\n\n    def observe(self,before,action,after,old_objects,new_objects):\n        old_scene,new_scene=entities(old_objects,before.pixels.shape),entities(new_objects,after.pixels.shape)\n        pending,self.nav_pending=self.nav_pending,None\n        usable=(before.levels==self.intervention_level and before.levels==after.levels\n                and action.id in (1,2,3,4) and after.state not in (\'GAME_OVER\',\'WIN\'))\n        budget=self._budget(before) if usable else None\n        usable=usable and budget is not None and budget[1]>0\n        cohort=self._cohort(old_scene,[(a,) for a in before.available]) if usable else None\n        record=None\n        if cohort is not None:\n            movers=cohort[2]\n            view=navigation_view(old_scene,before.pixels,movers)\n            delta=rigid_delta(old_scene,new_scene,movers)\n            vectors=self._measured_vectors((action.id,),old_scene)\n            nominal={vectors[s] for s in movers if s in vectors}\n            if view is not None and delta is not None and len(nominal)==1:\n                record=(view,(action.id,)),delta,next(iter(nominal))\n        if pending:\n            delta=rigid_delta(old_scene,new_scene,pending.pop(\'movers\')) if usable else None\n            pending.update(after=after.key(),actual_delta=delta,matched=delta==pending[\'predicted_delta\'])\n            self.nav_events.append(dict(kind=\'local_prediction_check\',**pending))\n        super().observe(before,action,after,old_objects,new_objects)\n        if record:\n            condition,delta,nominal=record\n            previous=self.nav_evidence[condition]\n            conflict=bool(previous) and delta not in previous\n            previous[delta]+=1\n            step=self.stats[\'observations\']+1\n            self.nav_sources[condition].append(step)\n            self.nav_stats[\'observed_local_transitions\']+=1\n            if delta!=nominal:\n                self.nav_exceptions.add(condition)\n                self.nav_stats[\'non_nominal_transitions\']+=1\n                self.nav_events.append({\'kind\':\'local_exception\',\'action_index\':step,\n                    \'before\':before.key(),\'after\':after.key(),\'action\':action.id,\n                    \'nominal_delta\':nominal,\'actual_delta\':delta,\'conflicting\':conflict})\n            self.nav_stats[\'conflicts\']+=conflict\n\n    def summary(self):\n        result=super().summary()\n        result[\'goal_navigation\']={\'stats\':dict(self.nav_stats),\'events\':self.nav_events,\n            \'local_conditions\':len(self.nav_evidence),\'exceptions\':len(self.nav_exceptions),\n            \'scope\':\'public goal-reading match only; measured nonlocal deltas are tentative local hypotheses; no reset-state restoration\'}\n        return result\n', 'experiment_refined.py': "from experiment_coarse import Explorer as C4Explorer\nfrom agent import Observation\nfrom efficient_controls import EfficientControls\n\nclass Explorer(C4Explorer):\n    def __init__(self,seed=0,policy='refined_controls'):\n        if policy!='refined_controls':raise ValueError(policy)\n        super().__init__(seed,'coarse_stages')\n        self.shared=EfficientControls(seed)\n        self.variant=policy\n", 'experiment_coarse.py': '"""Preserve the established prefix; use a small public-observation controller later."""\nfrom agent import Action, Observation, components\nfrom experiment_boundary_motion import Explorer as Baseline\nfrom association_dynamics import framed_patterns\nfrom shared_dynamics import entities\nfrom context_contact import body_box\nfrom coarse_controller import CoarseController\n\nclass Explorer(Baseline):\n    def __init__(self,seed=0,policy=\'coarse_stages\'):\n        if policy!=\'coarse_stages\':raise ValueError(policy)\n        super().__init__(seed,\'fiber_boundary_motion\')\n        self.variant=policy\n        self.coarse=None\n        self.reference_geometry=None\n\n    def choose(self,obs):\n        if obs.state in (\'GAME_OVER\',\'NOT_PLAYED\'):\n            self.reason=\'reset after terminal public response\'\n            return Action(0)\n        # Experimental late-level gate preserves the already verified prefix.\n        # Activation also requires an empirically learned actor and framed reference.\n        if obs.levels>=5 and self.coarse is None:\n            objects=components(obs.pixels)\n            scene=entities(objects,obs.pixels.shape)\n            cohort=self.shared._cohort(scene,[(a,) for a in obs.available])\n            body=body_box(scene,cohort[2]) if cohort else None\n            panels=framed_patterns(obs.pixels,objects)\n            refs=[p for p in panels if min(p.x,p.y,obs.pixels.shape[1]-p.x-p.width,obs.pixels.shape[0]-p.y-p.height)<=2]\n            if body and len(refs)==1 and self.shared.passable:\n                self.coarse=CoarseController(obs,body,refs[0],self.shared)\n        if self.coarse is not None and obs.levels>=5:\n            a,self.reason=self.coarse.choose(obs)\n            return Action(a)\n        return super().choose(obs)\n\n    def observe(self,before,action,after):\n        if self.coarse is not None and before.levels>=5:\n            self.coarse.observe(before,action,after)\n        else:\n            super().observe(before,action,after)\n\n    def diagnostics(self):\n        result=super().summary()\n        result[\'coarse_stages\']=self.coarse.summary() if self.coarse else {\'activated\':False}\n        return result\n', 'coarse_controller.py': '"""Coarse movement, one destination, prepare then transport.\n\nAll templates, colors, positions and contact effects are learned from public\nobservations in this run. No saved traces, game IDs, fixed action tails or game\nobjects are inputs. Projections are hypotheses, corrected after each action.\n"""\nfrom collections import Counter, defaultdict, deque\nfrom dataclasses import replace\nimport numpy as np\nfrom agent import components\nfrom association_dynamics import framed_patterns, fit_transforms\nfrom reconsideration import contact_landmarks\nfrom context_contact import canonical_sig\nfrom boundary_motion import strip\n\ndef box(p):return (p.x,p.y,p.width,p.height)\ndef center(b):return (b[0]+b[2]/2,b[1]+b[3]/2)\ndef overlap(a,b):return a[0]<b[0]+b[2] and b[0]<a[0]+a[2] and a[1]<b[1]+b[3] and b[1]<a[1]+a[3]\n\nclass CoarseController:\n    def __init__(self,obs,body,reference,prior):\n        x,y,w,h=body\n        self.template=obs.pixels[y:y+h,x:x+w].copy()\n        self.w,self.h=w,h\n        self.refbox=box(reference)\n        self.mask=reference.sig[0]\n        self.floor=prior.passable.most_common(1)[0][0]\n        budget=prior._budget(obs)\n        self.resource_color=budget[0] if budget else None\n        if self.resource_color is None:\n            # A level change can clear the previous rate before three samples\n            # exist. Read a long peripheral bar rather than inventing fuel.\n            excluded={self.mask,self.floor,*map(int,np.unique(self.template))}\n            runs=Counter()\n            for row in obs.pixels[-4:]:\n                last=None;length=0\n                for value in row:\n                    color=int(value)\n                    length=length+1 if color==last else 1\n                    last=color\n                    if color not in excluded:runs[color]=max(runs[color],length)\n            candidates=[(length,color) for color,length in runs.items() if length>=2*self.w]\n            if candidates:self.resource_color=max(candidates)[1]\n        self.effects={k:Counter(v) for k,v in prior.contact_table.items()}\n        self.types={}\n        for k,table in self.effects.items():\n            samples=list(table)\n            if k in prior.resource_keys:self.types[k]=\'fuel\'\n            elif k in prior.non_symmetric:self.types[k]=\'shape\'\n            elif any(a==c and b!=d for a,b,c,d in samples):self.types[k]=\'color\'\n            elif any(a!=c for a,b,c,d in samples):self.types[k]=\'rotate\'\n        self.boundaries={r[0] for r,n in prior.slide_rules.items() if n>=2 and r[1]==self.floor}\n        # The inherited controller clears local rules on a level transition,\n        # while retaining their public evidence. Reuse the observed boundary\n        # material, tentatively in either orientation; every edge is checked.\n        self.boundaries.update(e[\'rule\'][0] for e in prior.slide_events\n            if e.get(\'kind\')==\'boundary_fit\' and e.get(\'support\',0)>=2 and e[\'rule\'][1]==self.floor)\n        self.events=[];self.total=0;self.stats=Counter()\n        self.reset_local(obs.levels)\n\n    def reset_local(self,level):\n        self.level=level;self.tick=0;self.memory=None\n        self.walk=set();self.blocked=set();self.visits=Counter()\n        self.landmarks={};self.targets={};self.done=set();self.target=None;self.touched=None\n        self.motion={};self.motion_counts=defaultdict(Counter)\n        self.rate=1;self.capacity=0;self.rate_samples=Counter()\n        self.pending=None;self.last_update=None;self.fuel=0\n\n    def body(self,pixels):\n        v=np.lib.stride_tricks.sliding_window_view(pixels,(self.h,self.w))\n        yy,xx=np.where(np.all(v==self.template,axis=(-1,-2)))\n        return (int(xx[0]),int(yy[0])) if len(xx)==1 else None\n\n    def read(self,obs):\n        x,y,w,h=self.refbox\n        patch=obs.pixels[y:y+h,x:x+w]\n        ps=framed_patterns(patch,components(patch))\n        return (ps[0].pattern,ps[0].ink) if len(ps)==1 else None\n\n    def raw_fuel(self,pixels):\n        if self.resource_color is None:return 100\n        return max(int(np.count_nonzero(row==self.resource_color)) for row in pixels[-4:])\n\n    def tile(self,b):\n        cx,cy=center(b)\n        return (int(round((cx-self.w/2-self.origin[0])/self.w))*self.w+self.origin[0],\n                int(round((cy-self.h/2-self.origin[1])/self.h))*self.h+self.origin[1])\n\n    def update(self,obs):\n        if obs.levels!=self.level:self.reset_local(obs.levels)\n        if self.last_update==self.tick:return\n        self.last_update=self.tick\n        self.pos=self.body(obs.pixels)\n        self.reading=self.read(obs)\n        if self.pos is None:return\n        self.origin=(self.pos[0]%self.w,self.pos[1]%self.h)\n        if self.memory is None:self.memory=np.full_like(obs.pixels,self.mask)\n        visible=obs.pixels!=self.mask\n        self.memory[visible]=obs.pixels[visible]\n        self.visits[self.pos]+=1\n        objects=components(obs.pixels)\n        panels=framed_patterns(obs.pixels,objects)\n        for p in panels:\n            if box(p)!=self.refbox and p.sig[0]==self.mask and self.reading and p.pattern[0]==self.reading[0][0]:\n                b=box(p);self.targets[b]=(p.pattern,p.ink)\n                self.done.discard(b)  # A visibly returned goal invalidates collection memory.\n                self.memory[p.y:p.y+p.height,p.x:p.x+p.width]=obs.pixels[p.y:p.y+p.height,p.x:p.x+p.width]\n        for b in list(self.targets):\n            bx,by,bw,bh=b\n            patch=obs.pixels[by:by+bh,bx:bx+bw]\n            if b==self.touched and not overlap((*self.pos,self.w,self.h),b) and np.count_nonzero(patch==self.mask)==0:\n                self.done.add(b);self.target=None;self.touched=None\n                self.events.append({\'event\':\'goal_disappeared_after_matching_visit\',\'tick\':self.total,\'box\':b})\n        for p in contact_landmarks(obs.pixels,objects,panels,set()):\n            key=canonical_sig(p.sig)\n            if key[:2]!=(3,3):continue\n            # The inherited patch reader includes rectangular actor fragments;\n            # only known operator templates or a visible resource-colored ring qualify.\n            if key not in self.types:\n                arr=np.frombuffer(bytes.fromhex(key[2]),dtype=np.uint8)\n                if self.resource_color in arr and len(set(arr))==2:self.types[key]=\'fuel\'\n                else:continue\n            pos=self.tile((p.x,p.y,p.sig[1],p.sig[2]))\n            identity=(key,pos) if self.types[key]==\'fuel\' else key\n            item=self.landmarks.setdefault(identity,{\'key\':key,\'history\':[],\'transitions\':{},\'pos\':pos,\'seen\':self.tick})\n            history=item[\'history\']\n            if len(history)>=2 and history[-1][0]==self.tick-1 and history[-2][0]==self.tick-2:\n                item[\'transitions\'][history[-2][1],history[-1][1]]=pos\n            if not history or history[-1][0]!=self.tick:history.append((self.tick,pos))\n            item[\'pos\']=pos;item[\'seen\']=self.tick\n        for ident,item in list(self.landmarks.items()):\n            if self.types[item[\'key\']]!=\'fuel\' or item[\'seen\']==self.tick:continue\n            x,y=item[\'pos\'];patch=obs.pixels[y:y+self.h,x:x+self.w]\n            if self.resource_color not in patch and not overlap((*self.pos,self.w,self.h),(x,y,self.w,self.h)) and np.mean(patch==self.floor)>.7:\n                del self.landmarks[ident]\n        H,W=obs.pixels.shape\n        for y in range(self.origin[1],H-self.h+1,self.h):\n            for x in range(self.origin[0],W-self.w+1,self.w):\n                if overlap((x,y,self.w,self.h),self.refbox):continue\n                patch=self.memory[y:y+self.h,x:x+self.w]\n                if np.mean(patch==self.floor)>.5:self.walk.add((x,y))\n        for b in self.targets:self.walk.add(self.tile(b))\n        for item in self.landmarks.values():self.walk.add(item[\'pos\'])\n        self.walk.add(self.pos)\n        self.capacity=max(self.capacity,self.raw_fuel(obs.pixels))\n        self.fuel=self.raw_fuel(obs.pixels)/max(1,self.rate)\n\n    def forecast(self,item,horizon=64):\n        hist=item[\'history\'];current=item[\'pos\'];previous=hist[-2][1] if len(hist)>1 else current\n        result=[current]\n        # Second-order observed position transitions retain bounce direction.\n        for n in range(horizon):\n            nxt=item[\'transitions\'].get((previous,current))\n            if nxt is None:\n                nxt=current\n                if len(hist)>1:\n                    delta=(current[0]-previous[0],current[1]-previous[1])\n                    positions={p for _,p in hist}\n                    proposed=(current[0]+delta[0],current[1]+delta[1])\n                    if delta!=(0,0) and proposed in positions:nxt=proposed\n                    elif delta!=(0,0):\n                        reverse=(current[0]-delta[0],current[1]-delta[1])\n                        if reverse in positions:nxt=reverse\n            result.append(nxt);previous,current=current,nxt\n        return result\n\n    def predict(self,pos,action):\n        if (pos,action) in self.motion:return self.motion[pos,action]\n        dx,dy={1:(0,-self.h),2:(0,self.h),3:(-self.w,0),4:(self.w,0)}[action]\n        candidate=pos[0]+dx,pos[1]+dy\n        if candidate not in self.walk:return pos\n        nominal=(*candidate,self.w,self.h)\n        proposals=[]\n        for axis in (0,1):\n            for sign in (-1,1):\n                edge=strip(self.memory,nominal,axis,sign)\n                if edge.size and np.all(edge==edge[0]) and int(edge[0]) in self.boundaries:\n                    q=candidate\n                    while True:\n                        nxt=list(q);nxt[axis]+=sign*(self.w if axis==0 else self.h);nxt=tuple(nxt)\n                        if nxt not in self.walk:break\n                        q=nxt\n                    if q!=candidate:proposals.append(q)\n        return proposals[0] if len(set(proposals))==1 else candidate\n\n    def route(self,destination=None,operator=None,avoid=True,start=None,horizon=60):\n        forecasts=self.forecasts\n        start=self.pos if start is None else start\n        queue=deque([(start,0,())]);seen={(start,0)}\n        while queue:\n            pos,t,path=queue.popleft()\n            if t>=horizon:continue\n            for a in (1,2,3,4):\n                q=self.predict(pos,a)\n                if q==pos:continue\n                nt=t+1\n                hits=[ident for ident,fs in forecasts.items() if fs[min(nt,len(fs)-1)]==q]\n                if avoid and any(ident!=operator and self.types[self.landmarks[ident][\'key\']]!=\'fuel\' for ident in hits):continue\n                p=path+(a,)\n                if (operator is not None and operator in hits) or (destination is not None and q==destination):return p\n                state=(q,nt%24)\n                if state not in seen:seen.add(state);queue.append((q,nt,p))\n        return None\n\n    def select_target(self):\n        candidates=[b for b in self.targets if b not in self.done]\n        if self.target not in candidates:\n            if not candidates:self.target=None;return\n            def rank(b):\n                p,c=self.targets[b]\n                return (0 if self.reading[0]==p else 1 if fit_transforms(self.reading[0],p) else 2,self.reading[1]!=c,b)\n            self.target=min(candidates,key=rank)\n\n    def choose(self,obs):\n        self.update(obs)\n        if self.pos is None or self.reading is None:return 0,\'coarse: unreadable public actor/reference; reset\'\n        self.forecasts={ident:self.forecast(item) for ident,item in self.landmarks.items()}\n        self.select_target()\n        options=[];stage=\'explore\'\n        if self.target:\n            desired=self.targets[self.target]\n            if self.reading==desired:\n                stage=\'transport\'\n                destination=self.tile(self.target)\n                if self.pos==destination:self.touched=self.target\n                if self.touched==self.target:\n                    # Leave far enough to distinguish collection from actor occlusion.\n                    candidates=[q for q in self.walk if not overlap((*q,self.w,self.h),self.target)]\n                    for q in candidates:\n                        path=self.route(destination=q)\n                        if path:options.append((len(path),q,path,None))\n                    stage=\'confirm_collection\'\n                else:\n                    path=self.route(destination=destination)\n                    if path:options.append((len(path),destination,path,None))\n            else:\n                orbit=bool(fit_transforms(self.reading[0],desired[0]))\n                kinds={\'shape\'} if not orbit else set()\n                if orbit and self.reading[0]!=desired[0]:kinds.add(\'rotate\')\n                if orbit and self.reading[1]!=desired[1]:kinds.add(\'color\')\n                stage=\'prepare\'\n                for ident,item in self.landmarks.items():\n                    if self.types[item[\'key\']] not in kinds:continue\n                    path=self.route(operator=ident)\n                    if path:options.append((len(path),repr(ident),path,ident))\n        if not options:\n            stage=\'explore\'\n            unknown=[]\n            for q in self.walk:\n                gain=sum(1 for dy in range(-3,4) for dx in range(-3,4)\n                    if 0<=q[0]+dx*self.w<obs.pixels.shape[1] and 0<=q[1]+dy*self.h<obs.pixels.shape[0]\n                    and self.memory[q[1]+dy*self.h,q[0]+dx*self.w]==self.mask)\n                if gain:unknown.append((-gain/(1+self.visits[q]),q))\n            for score,q in sorted(unknown)[:12]:\n                path=self.route(destination=q)\n                if path:options.append((len(path)+score/4,q,path,None))\n        chosen=min(options,key=lambda x:(x[0],str(x[1]))) if options else None\n        supplies=[]\n        for ident,item in self.landmarks.items():\n            if self.types[item[\'key\']]!=\'fuel\':continue\n            path=self.route(destination=item[\'pos\'])\n            if path and len(path)<self.fuel:supplies.append((len(path),str(ident),path,ident))\n        reserve=3\n        if chosen and chosen[3] is not None and supplies:\n            endpoint=self.forecasts[chosen[3]][min(len(chosen[2]),60)]\n            reserves=[self.route(destination=self.landmarks[s[3]][\'pos\'],start=endpoint) for s in supplies]\n            lengths=[len(p) for p in reserves if p]\n            if lengths:reserve=min(lengths)+2\n        need=(chosen is None or len(chosen[2])+(0 if stage==\'transport\' else reserve)>=self.fuel)\n        if supplies and (need or min(s[0] for s in supplies)+3>=self.fuel):\n            chosen=min(supplies);stage=\'refuel\'\n        if chosen:\n            action=chosen[2][0]\n        else:\n            # No global plan: choose a real untried coarse edge, never repeat a known no-op.\n            choices=[(self.visits[self.predict(self.pos,a)],a) for a in (1,2,3,4) if self.predict(self.pos,a)!=self.pos]\n            action=min(choices)[1] if choices else 0\n            stage=\'local_probe\' if choices else \'reset_no_route\'\n        self.pending={\'pos\':self.pos,\'reading\':self.reading,\'fuel\':self.raw_fuel(obs.pixels),\n            \'predicted\':self.predict(self.pos,action) if action else self.pos,\n            \'hits\':[ident for ident,fs in self.forecasts.items() if fs[1]==(self.predict(self.pos,action) if action else self.pos)],\n            \'stage\':stage,\'action\':action}\n        self.events.append({\'tick\':self.total,\'level\':self.level,\'stage\':stage,\'body\':self.pos,\n            \'reading\':self.reading,\'target\':self.targets.get(self.target),\'fuel\':self.fuel,\n            \'planned\':len(chosen[2]) if chosen else None,\'action\':action})\n        return action,\'coarse: \'+stage\n\n    def observe(self,before,action,after):\n        self.total+=1;self.tick+=1\n        pending,self.pending=self.pending,None\n        if action.id==0 or after.levels!=before.levels:\n            self.reset_local(after.levels);return\n        actual=self.body(after.pixels)\n        if not pending or actual is None:return\n        pos=pending[\'pos\']\n        self.motion_counts[pos,action.id][actual]+=1\n        self.motion[pos,action.id]=actual\n        self.stats[\'motion_predictions\']+=1\n        self.stats[\'motion_mismatches\']+=actual!=pending[\'predicted\']\n        if actual!=pos:\n            old=before.pixels[pos[1]:pos[1]+self.h,pos[0]:pos[0]+self.w]\n            self.memory[pos[1]:pos[1]+self.h,pos[0]:pos[0]+self.w]=self.floor\n        raw=self.raw_fuel(after.pixels);delta=pending[\'fuel\']-raw\n        if 0<delta<=self.w*self.h:\n            self.rate_samples[delta]+=1;self.rate=self.rate_samples.most_common(1)[0][0]\n        reading=self.read(after)\n        hits=pending[\'hits\']\n        if reading!=pending[\'reading\'] and reading and len(hits)==1:\n            ident=hits[0];item=self.landmarks[ident];key=item[\'key\']\n            a,b=pending[\'reading\'];c,d=reading\n            self.effects.setdefault(key,Counter())[a,b,c,d]+=1\n            if a!=c and not fit_transforms(a,c):self.types[key]=\'shape\'\n            elif a==c and b!=d:self.types[key]=\'color\'\n            elif a!=c:self.types[key]=\'rotate\'\n            # An effect while the glyph is occluded supplies a public contact observation.\n            history=item[\'history\']\n            if history and history[-1][0]==self.tick-1:\n                if len(history)>1 and history[-2][0]==self.tick-2:item[\'transitions\'][history[-2][1],history[-1][1]]=actual\n                history.append((self.tick,actual));item[\'pos\']=actual;item[\'seen\']=self.tick\n        if raw>pending[\'fuel\']:\n            for ident,item in list(self.landmarks.items()):\n                if self.types[item[\'key\']]==\'fuel\' and item[\'pos\']==actual:del self.landmarks[ident]\n\n    def summary(self):\n        return {\'activated\':True,\'stats\':dict(self.stats),\'events\':self.events,\n            \'scope\':\'Observation-driven coarse planner; no game identifier, preset target, action tape or hidden state input\',\n            \'known_operator_types\':[{\'key\':k,\'type\':v} for k,v in self.types.items()]}\n', 'experiment_boundary_motion.py': "from agent import Action,Observation\nfrom experiment_unknown_contact import Explorer as PriorExplorer\nfrom boundary_motion import BoundaryMotionDynamics\nclass Explorer(PriorExplorer):\n    def __init__(self,seed=0,policy='fiber_boundary_motion'):\n        if policy!='fiber_boundary_motion':raise ValueError(policy)\n        super().__init__(seed,'fiber_unknown_contact');self.variant=policy\n        self.shared=BoundaryMotionDynamics(seed)\n", 'experiment_unknown_contact.py': '"""Swappable baseline / fifth-level contact-exploration candidate.\n\nIn a frozen release all imported baseline files live beside this module.\nThe evaluator supplies only public Observation instances to choose/observe.\n"""\nfrom agent import Action, Explorer as BaseExplorer, Observation\nfrom agent_minimal import TransitionLedger\nfrom fiber_probe import FiberProbeDynamics\nfrom unknown_contact import UnknownContactDynamics\n\nPOLICIES = (\'fiber_probe_coupled\', \'fiber_unknown_contact\')\n\n\nclass Explorer(BaseExplorer):\n    def __init__(self, seed=0, policy=\'fiber_unknown_contact\'):\n        if policy not in POLICIES:\n            raise ValueError(policy)\n        super().__init__(seed=seed, policy=\'review\')\n        self.variant = policy\n        self.shared = (FiberProbeDynamics(seed, \'coupled\') if policy == \'fiber_probe_coupled\'\n                       else UnknownContactDynamics(seed, \'coupled\', intervention_level=4))\n        self.public_evidence = TransitionLedger()\n\n    def observe(self, before, action, after):\n        self.public_evidence.add(before, action, after)\n        super().observe(before, action, after)\n\n    def summary(self):\n        return {\'variant\': self.variant, \'ledger\': self.ledger.summary(),\n                \'public_evidence\': self.public_evidence.summary(), \'shared\': self.shared.summary()}\n', 'agent_minimal.py': '"""Small, replaceable policies whose only input is the public observation.\n\nNo game ID, environment, source file, reference solution, or inferred color /\nshape semantics enter these policies. Exact public observations are evidence\nkeys, not a claim that identical screens have identical hidden game states.\n"""\nfrom __future__ import annotations\n\nimport hashlib\nimport random\nfrom collections import Counter, defaultdict, deque\nfrom dataclasses import dataclass\nfrom numbers import Integral\n\nimport numpy as np\n\nPOLICIES = ("uniform", "unknown_first", "frontier")\n\n\ndef _integer(value, name: str) -> int:\n    if isinstance(value, bool) or not isinstance(value, Integral):\n        raise ValueError(f"{name} must be an integer")\n    return int(value)\n\n\n@dataclass(frozen=True, eq=False)\nclass Observation:\n    pixels: np.ndarray\n    available: tuple[int, ...]\n    state: str\n    levels: int\n\n    def __post_init__(self) -> None:\n        pixels = np.asarray(self.pixels)\n        if pixels.ndim != 2 or not all(0 < d <= 64 for d in pixels.shape):\n            raise ValueError("Expected a nonempty public grid at most 64 by 64")\n        # Validate before conversion: negative / overflowing values must not wrap.\n        if pixels.dtype.kind not in "iu" or np.any(pixels < 0) or np.any(pixels > 15):\n            raise ValueError("Expected integer ARC palette indices 0..15")\n        pixels = pixels.astype(np.uint8, copy=True)\n        pixels.setflags(write=False)\n        available = tuple(sorted({_integer(a, "action ID") for a in self.available}))\n        if any(a < 0 or a > 7 for a in available):\n            raise ValueError("Unsupported public action ID; inspect the SDK contract")\n        levels = _integer(self.levels, "levels")\n        if levels < 0 or not isinstance(self.state, str) or not self.state:\n            raise ValueError("Invalid public state / completed-level count")\n        object.__setattr__(self, "pixels", pixels)\n        object.__setattr__(self, "available", available)\n        object.__setattr__(self, "levels", levels)\n\n    @classmethod\n    def from_frame(cls, frame) -> "Observation":\n        if len(frame.frame) == 0:\n            raise ValueError("No public image frame; the runner must handle an empty RESET")\n        state = frame.state.name if hasattr(frame.state, "name") else str(frame.state)\n        return cls(frame.frame[-1], tuple(frame.available_actions), state,\n                   frame.levels_completed)\n\n    @classmethod\n    def from_record(cls, record: dict) -> "Observation":\n        """Read the public-record schema used by baseline/run_probe.py."""\n        frames = record["frames"]\n        if not frames:\n            raise ValueError("No image in recorded public observation")\n        return cls(np.asarray(frames[-1]), tuple(record["available_actions"]),\n                   record["state"], record["levels_completed"])\n\n    def key(self) -> str:\n        metadata = repr((self.pixels.shape, self.available, self.state, self.levels)).encode()\n        return hashlib.sha256(metadata + self.pixels.tobytes()).hexdigest()\n\n\n@dataclass(frozen=True)\nclass Action:\n    id: int\n    x: int | None = None\n    y: int | None = None\n\n    def __post_init__(self) -> None:\n        action_id = _integer(self.id, "action ID")\n        if not 0 <= action_id <= 7:\n            raise ValueError("Unsupported public action ID")\n        object.__setattr__(self, "id", action_id)\n        if action_id == 6:\n            object.__setattr__(self, "x", _integer(self.x, "x"))\n            object.__setattr__(self, "y", _integer(self.y, "y"))\n        elif self.x is not None or self.y is not None:\n            raise ValueError("Coordinates belong only to public ACTION6")\n\n    def data(self) -> dict[str, int]:\n        return {"x": self.x, "y": self.y} if self.id == 6 else {}\n\n\ndef validate_action(obs: Observation, action: Action) -> None:\n    if obs.state == "WIN":\n        raise ValueError("No further action after WIN")\n    if action.id == 0:\n        if obs.state not in ("NOT_PLAYED", "GAME_OVER"):\n            raise ValueError("This policy does not reset an active game")\n        return\n    if obs.state in ("NOT_PLAYED", "GAME_OVER") or action.id not in obs.available:\n        raise ValueError("Unavailable action")\n    if action.id == 6:\n        h, w = obs.pixels.shape\n        if not (0 <= action.x < w and 0 <= action.y < h):\n            raise ValueError("Click is outside the public image")\n\n\nclass TransitionLedger:\n    """Measured outcomes only; observed_no_change does not mean no hidden effect."""\n\n    def __init__(self) -> None:\n        self.edges: dict[tuple[str, Action], Counter[str]] = defaultdict(Counter)\n        self.by_screen: dict[str, set[Action]] = defaultdict(set)\n        self.first_examples: dict[tuple[str, Action, str], int] = {}\n        self.conflicts: list[dict] = []\n        self.transitions = 0\n        self.pixel_unchanged = 0\n        self.observation_unchanged = 0\n\n    def add(self, before: Observation, action: Action, after: Observation) -> None:\n        before_key, after_key = before.key(), after.key()\n        key = (before_key, action)\n        outcomes = self.edges[key]\n        if outcomes and after_key not in outcomes:\n            previous = next(iter(outcomes))\n            self.conflicts.append({\n                "screen": before_key, "action": {"id": action.id, **action.data()},\n                "first_transition": self.first_examples[(before_key, action, previous)],\n                "new_transition": self.transitions,\n                "interpretation": "same public screen/action, different public result; cause unknown",\n            })\n        self.first_examples.setdefault((before_key, action, after_key), self.transitions)\n        outcomes[after_key] += 1\n        self.by_screen[before_key].add(action)\n        self.transitions += 1\n        self.pixel_unchanged += int(np.array_equal(before.pixels, after.pixels))\n        self.observation_unchanged += int(before_key == after_key)\n\n    def count(self, obs: Observation, action: Action) -> int:\n        return sum(self.edges.get((obs.key(), action), {}).values())\n\n    def status(self, obs: Observation, action: Action) -> str:\n        key = obs.key()\n        outcomes = self.edges.get((key, action))\n        if not outcomes:\n            return "unknown"\n        if len(outcomes) > 1:\n            return "conflicting"\n        return "observed_no_change" if key in outcomes else "observed_change"\n\n    def summary(self) -> dict:\n        status_counts = Counter()\n        for (before_key, _), outcomes in self.edges.items():\n            status = ("conflicting" if len(outcomes) > 1 else\n                      "observed_no_change" if before_key in outcomes else "observed_change")\n            status_counts[status] += 1\n        return {\n            "transitions": self.transitions,\n            "observed_screen_action_pairs": len(self.edges),\n            "observed_status_counts": dict(status_counts),\n            "screen_model_conflicts": len(self.conflicts),\n            "conflict_examples": self.conflicts[:8],\n            "pixel_unchanged_transitions": self.pixel_unchanged,\n            "public_observation_unchanged_transitions": self.observation_unchanged,\n            "certified_state_merges": 0,\n            "hidden_state_claims": 0,\n            "interpretation": "last public frame + available actions + state + levels; finite evidence only",\n        }\n\n\nclass Explorer:\n    """Same choose/observe entry points as the fixed baseline.\n\n    uniform: uniformly pick an available action ID, then a uniform pixel for 6.\n    unknown_first: try unobserved exact-screen/action pairs before revisiting.\n    frontier: same, then tentatively route over measured nonconflicting edges\n              to a previously seen public screen with untried actions.\n\n    All modes have the same action support, seed mechanism and evidence ledger.\n    No component, color, motion, direction or action-effect assumptions are used.\n    """\n\n    def __init__(self, seed: int = 0, policy: str = "unknown_first") -> None:\n        if policy not in POLICIES:\n            raise ValueError(f"Unknown policy: {policy}")\n        self.rng = random.Random(seed)\n        self.policy = policy\n        self.ledger = TransitionLedger()\n        self.steps = 0\n        self.reason = ""\n        self.screens: dict[str, Observation] = {}\n        self.route_decisions = 0\n\n    def _groups(self, obs: Observation, *, unknown_only: bool = False) -> dict[int, list[Action]]:\n        key = obs.key()\n        groups = {}\n        for action_id in obs.available:\n            if action_id == 0:\n                continue\n            if action_id == 6:\n                h, w = obs.pixels.shape\n                actions = [Action(6, x, y) for y in range(h) for x in range(w)]\n            else:\n                actions = [Action(action_id)]\n            if unknown_only:\n                actions = [a for a in actions if (key, a) not in self.ledger.edges]\n            if actions:\n                groups[action_id] = actions\n        return groups\n\n    def _sample(self, groups: dict[int, list[Action]]) -> Action:\n        action_id = self.rng.choice(sorted(groups))\n        return self.rng.choice(groups[action_id]) if action_id == 6 else groups[action_id][0]\n\n    def _route_to_unknown(self, obs: Observation) -> Action | None:\n        """Bounded tentative BFS; replan after every actual public response."""\n        start = obs.key()\n        queue = deque([(start, None, 0)])\n        seen = {start}\n        while queue and len(seen) <= 256:\n            key, first_action, depth = queue.popleft()\n            if depth >= 8:\n                continue\n            for action in sorted(self.ledger.by_screen.get(key, ()),\n                                 key=lambda a: (a.id, -1 if a.x is None else a.x,\n                                                -1 if a.y is None else a.y)):\n                if action.id == 0:\n                    continue\n                outcomes = self.ledger.edges[(key, action)]\n                if len(outcomes) != 1:\n                    continue  # A contradicted prediction is never a route edge.\n                destination = next(iter(outcomes))\n                if destination in seen or destination not in self.screens:\n                    continue\n                target = self.screens[destination]\n                if target.state in ("WIN", "GAME_OVER", "NOT_PLAYED") or target.levels != obs.levels:\n                    continue\n                seen.add(destination)\n                first = first_action if first_action is not None else action\n                if self._groups(target, unknown_only=True):\n                    return first\n                queue.append((destination, first, depth + 1))\n        return None\n\n    def choose(self, obs: Observation) -> Action:\n        if obs.state == "WIN":\n            raise RuntimeError("No further actions after WIN")\n        if obs.state in ("NOT_PLAYED", "GAME_OVER"):\n            self.reason = "RESET after public NOT_PLAYED / GAME_OVER"\n            return Action(0)\n        groups = self._groups(obs)\n        if not groups:\n            raise RuntimeError("No available action in unfinished game")\n        if self.policy == "uniform":\n            self.reason = "uniform available action ID; uniform pixel for ACTION6"\n            return self._sample(groups)\n        unknown = self._groups(obs, unknown_only=True)\n        if unknown:\n            self.reason = "unobserved exact public screen/action; effect unknown"\n            return self._sample(unknown)\n        if self.policy == "frontier":\n            route = self._route_to_unknown(obs)\n            if route is not None:\n                validate_action(obs, route)\n                self.reason = "tentative measured route to untried action; replan after response"\n                self.route_decisions += 1\n                return route\n        # Re-measure every action eventually, including no-change/conflicting ones.\n        # This never converts an observed no-change into a permanent invalidation.\n        key = obs.key()\n        counts = {a: sum(self.ledger.edges[(key, a)].values())\n                  for actions in groups.values() for a in actions}\n        minimum = min(counts.values())\n        least = {i: [a for a in actions if counts[a] == minimum]\n                 for i, actions in groups.items()}\n        action = self._sample({i: actions for i, actions in least.items() if actions})\n        self.reason = f"re-measure least sampled action; prior={self.ledger.status(obs, action)}"\n        return action\n\n    def observe(self, before: Observation, action: Action, after: Observation) -> None:\n        validate_action(before, action)\n        self.ledger.add(before, action, after)\n        self.screens[before.key()] = before\n        self.screens[after.key()] = after\n        self.steps += 1\n\n    def summary(self) -> dict:\n        return {"policy": self.policy, "steps": self.steps,\n                "route_decisions": self.route_decisions, "ledger": self.ledger.summary()}\n', 'submission_adapter.py': '"""Factory for the pinned official Agent interface; local check, not a submission."""\nfrom __future__ import annotations\n\nfrom types import SimpleNamespace\n\nfrom arcengine import GameAction, GameState\nfrom agents.agent import Agent\n\n\ndef make_agent_class(policy_factory, observation_type, *, policy_name, seed=0, total_budget=1200,\n                     initialization_resets=1):\n    """Explicit budget is local experiment configuration, never an official limit."""\n    if total_budget < initialization_resets or initialization_resets not in (0, 1):\n        raise ValueError("Invalid initialization RESET convention")\n\n    class MyAgent(Agent):\n        MAX_ACTIONS = total_budget - initialization_resets\n\n        def __init__(self, *args, **kwargs):\n            super().__init__(*args, **kwargs)\n            self.core = policy_factory(seed=seed, policy=policy_name)\n            self.action_counter = 0\n            self.pending = None\n            self.frames = [self._convert_raw_frame_data(self.arc_env.observation_space)]\n\n        def is_done(self, frames, latest_frame):\n            # Upstream Agent.main uses <=; this guard enforces the exact remaining cap.\n            return latest_frame.state is GameState.WIN or self.action_counter >= self.MAX_ACTIONS\n\n        def choose_action(self, frames, latest_frame):\n            if self.pending is not None:\n                raise RuntimeError("Previous action has no observed response")\n            # Only the allowed public fields enter the core policy.\n            public = SimpleNamespace(frame=latest_frame.frame, state=latest_frame.state,\n                                     levels_completed=latest_frame.levels_completed,\n                                     available_actions=latest_frame.available_actions)\n            before = observation_type.from_frame(public)\n            choice = self.core.choose(before)\n            data = dict(choice.data())\n            if choice.id != 0 and choice.id not in before.available:\n                raise ValueError("Unavailable action")\n            if choice.id == 6 and not (0 <= data["x"] < before.pixels.shape[1] and 0 <= data["y"] < before.pixels.shape[0]):\n                raise ValueError("Invalid click")\n            reasoning = {"policy": policy_name, "reason": str(getattr(self.core, "reason", ""))}\n            self.pending = (before, choice, data, reasoning)\n            action = GameAction.from_id(choice.id)\n            action.set_data(data)\n            action.reasoning = reasoning\n            return action\n\n        def do_action_request(self, action):\n            if self.pending is None:\n                raise RuntimeError("No pending action")\n            _, choice, data, reasoning = self.pending\n            if action.value != choice.id:\n                raise RuntimeError("Selected action differs from requested action")\n            # Enum values carry shared mutable data; use this instance\'s saved payload.\n            raw = self.arc_env.step(GameAction.from_id(choice.id), data=data, reasoning=reasoning)\n            return self._convert_raw_frame_data(raw)\n\n        def append_frame(self, frame):\n            if self.pending is None:\n                raise RuntimeError("Response without pending action")\n            before, choice, _, _ = self.pending\n            public = SimpleNamespace(frame=frame.frame, state=frame.state,\n                                     levels_completed=frame.levels_completed,\n                                     available_actions=frame.available_actions)\n            self.core.observe(before, choice, observation_type.from_frame(public))\n            self.pending = None\n            super().append_frame(frame)\n\n    return MyAgent\n'}
_root = Path(tempfile.gettempdir()) / "arc_coverage_c8b_54689f81d3ca89d0"
_root.mkdir(parents=True,exist_ok=True)
for _name,_text in _SOURCES.items():
    _path=_root/_name
    _bytes=_text.encode('utf-8')
    if not _path.exists() or _path.read_bytes()!=_bytes:_path.write_bytes(_bytes)
sys.path.insert(0,str(_root))
from submission_adapter import make_agent_class
from experiment_coverage_guarded import Explorer,Observation
MyAgent=make_agent_class(Explorer,Observation,policy_name='coverage_c8b',seed=0,
                        total_budget=1200,initialization_resets=1)


In [ ]:
import os,sys,shutil,subprocess
from pathlib import Path
rerun=bool(os.getenv('KAGGLE_IS_COMPETITION_RERUN'))
framework=Path('/kaggle/working/ARC-AGI-3-Agents' if rerun else '/tmp/arc_c8b_smoke_framework')
shutil.copytree('/kaggle/input/competitions/arc-prize-2026-arc-agi-3'+'/ARC-AGI-3-Agents',framework,dirs_exist_ok=True)
shutil.copy2('/tmp/my_agent.py',framework/'agents/templates/my_agent.py')
(framework/'agents/__init__.py').write_text("from .agent import Agent, Playback\nfrom .swarm import Swarm\nfrom .templates.random_agent import Random\nfrom .templates.my_agent import MyAgent\nAVAILABLE_AGENTS={'random':Random,'myagent':MyAgent}\n",encoding='utf-8')
sys.path.insert(0,str(framework))
from agents.agent import Agent
from agents.templates.my_agent import MyAgent
assert MyAgent.main is Agent.main
assert MyAgent.MAX_ACTIONS==1199
print('coverage c8b imported; official Agent.main preserved; payload 54689f81d3ca89d0')


In [ ]:
if rerun:
    subprocess.run(['curl','--fail','--retry','999','--retry-all-errors','--retry-delay','5','--retry-max-time','600','http://gateway:8001/api/games'],check=True)
    (framework/'.env').write_text('SCHEME=http\nHOST=gateway\nPORT=8001\nARC_API_KEY=test-key-123\nARC_BASE_URL=http://gateway:8001/\nOPERATION_MODE=online\nENVIRONMENTS_DIR=\nRECORDINGS_DIR=/kaggle/working/server_recording\n',encoding='utf-8')
    subprocess.run([sys.executable,'main.py','--agent','myagent'],cwd=framework,
                   env={**os.environ,'MPLBACKEND':'agg'},check=True)
else:
    # Official starter convention: commit validation output, not a game score.
    import pandas as pd
    pd.DataFrame([['1_0','1',True,1]],columns=['row_id','game_id','end_of_game','score']).to_parquet('/kaggle/working/submission.parquet',index=False)
